# Dungeon Master v2: Arquitectura Multi-Agente con Expansión Dinámica del Mundo

**Proyecto Final — NLP (MMIA 6021 4889)**  
**Pablo Encalada**  
**Octubre 2026**

---

## 1. Problema y motivación

En la **v1** demostramos que un agente LangGraph con herramientas supera a un LLM directo en consistencia narrativa (+20 puntos en LLM-as-judge). Sin embargo, el mundo era **estático**: 5 ubicaciones fijas, NPCs predefinidos, sin posibilidad de expansión.

En un juego de D&D real, el DM **inventa** nuevas locaciones, personajes y objetos sobre la marcha cuando el jugador explora. "Busco una puerta escondida" debería generar una nueva rama de la historia, no un "no hay nada aquí".

### Diseño experimental v2

| | v1 (Agente simple) | v2 (Multi-agente) |
|---|---|---|
| **Arquitectura** | 1 nodo agente + ToolNode | 3 nodos especializados: Narrator, WorldBuilder, Coherence |
| **Mundo** | Estático (5 ubicaciones fijas) | Dinámico (se expande con `create_location`, `create_npc`, `create_item`) |
| **Flujo** | agent → tools → agent | narrator → ¿expandir? → world_builder → coherence → narrator |
| **Evaluación** | Keywords + LLM-as-judge | Misma + métricas de expansión (nuevas zonas creadas, coherencia del mundo expandido) |

In [ ]:
import time, random, re, json, textwrap, copy, math, unicodedata, os  # noqa
from typing import Annotated, TypedDict, Literal
from collections import Counter
import networkx as nx

import pandas as pd
import matplotlib.pyplot as plt

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

from openai import OpenAI

print("Imports OK")

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Configuración del LLM
# ────────────────────────────────────────────────────────────────────────────

OLLAMA_MODEL = "qwen2.5:latest"  # 7B Q4_K_M
OLLAMA_BASE_URL = "http://localhost:11434"
EMBED_MODEL = "bge-m3"  # embeddings multilingües (ollama pull bge-m3)
EMBED_BASE_URL = OLLAMA_BASE_URL
JUDGE_MODEL = "qwen2.5:72b"
WATCHER_MODEL = os.environ.get("WATCHER_MODEL", JUDGE_MODEL)  # v3: vigilante de coherencia (en la Mac: el 7B)  # el juez debe ser más grande que el modelo evaluado (Semana 2 Clase 3)

llm = ChatOllama(model=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL, temperature=0.7)
llm_creative = ChatOllama(model=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL, temperature=0.9)  # más creativo para WorldBuilder
llm_strict = ChatOllama(model=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL, temperature=0.1)    # más determinista para Coherence

try:
    resp = llm.invoke("Di solo OK")
    print(f"Ollama conectado. Modelo: {OLLAMA_MODEL}")
    print(f"Tres perfiles de temperatura: narrator=0.7, world_builder=0.9, coherence=0.1")
except Exception as e:
    print(f"ERROR: {e}")

## 2. Mundo dinámico

El mundo inicial es el mismo que v1 (5 ubicaciones), pero ahora el `game_state` incluye un **mundo mutable** que puede crecer durante la partida. Las nuevas ubicaciones, NPCs y objetos creados por el WorldBuilder se registran en `game_state["world"]`.

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Definición del mundo (inicial — puede crecer dinámicamente)
# ────────────────────────────────────────────────────────────────────────────

# v3: el contenido de AUTOR (lugares, personajes, qué saben, a quién conocen) vive en data/mundo.json,
# separado del código. Lo que se crea durante la partida lo genera el WorldBuilder.
import glob, yaml

with open("data/mundo.json", encoding="utf-8") as f:
    _mundo = json.load(f)


def cargar_ficha(ruta):
    """Ficha .md de un personaje: encabezado YAML (datos exactos que usa el código: ubicación, si es hostil,
    qué vende, relaciones) + secciones en prosa (## Personalidad, ## Lo que sabe, ## Rumores, ## Opiniones…),
    que se dividen en fragmentos y se recuperan por RAG cuando el personaje habla."""
    texto = open(ruta, encoding="utf-8").read()
    _, cabecera, cuerpo = texto.split("---", 2)
    meta = yaml.safe_load(cabecera)
    cuerpo = re.sub(r"<!--.*?-->", "", cuerpo, flags=re.S)
    fragmentos, seccion = [], ""
    for linea in cuerpo.splitlines():
        if linea.startswith("## "):
            seccion = linea[3:].strip()
        elif linea.strip():
            fragmentos.append({"seccion": seccion, "texto": linea.strip().lstrip("- ").strip()})
    secc = lambda s: [f["texto"] for f in fragmentos if f["seccion"].lower() == s]
    return meta["nombre"], {
        "dialogue": meta.get("saludo", ""), "hostile": bool(meta.get("hostil")), "ubicacion": meta.get("ubicacion"),
        "personalidad": " ".join(secc("personalidad")),
        "conocimientos": [{"hecho": t, "fuente": "propio"} for t in secc("lo que sabe")] +
                         [{"hecho": t, "fuente": "rumor"} for t in secc("rumores")] +
                         [{"hecho": t, "fuente": "opinion"} for t in secc("opiniones")],
        "relaciones": meta.get("relaciones", []), "vende": meta.get("vende", {}), "temas_srd": meta.get("temas_srd", []),
        **{k: meta[k] for k in ("hp", "attack", "defense", "srd_ref", "stats_propios") if k in meta}, "ficha": ruta,
    }


WORLD_INITIAL = {"locations": _mundo["locations"],
                 "npcs": dict(cargar_ficha(r) for r in sorted(glob.glob("data/personajes/*.md")))}
INITIAL_STATE = _mundo["estado_inicial"]
REGLAS_COMBATE = _mundo["reglas_combate"]  # v3: CA del jugador, bonificador, daño de cada arma


def conocimientos_npc(datos):
    """Lo que sabe un NPC, en texto: de primera mano, rumores y relaciones (a quién conoce)."""
    out = []
    for c in datos.get("conocimientos", []):
        if isinstance(c, str):
            out.append(c)
        elif c.get("fuente") == "rumor":
            out.append(f"(rumor, dilo como 'dicen que…') {c['hecho']}")
        elif c.get("fuente") == "opinion":
            out.append(f"(tu opinión) {c['hecho']}")
        else:
            out.append(c["hecho"])
    out += [f"Conoces a {r['con']} ({r['relacion']}): {r['detalle']}." for r in datos.get("relaciones", [])]
    return out or [datos.get("description", datos.get("dialogue", ""))]


print("Mundo inicial (5 ubicaciones — puede crecer):")
for loc, data in WORLD_INITIAL["locations"].items():
    conns = ", ".join(data["connections"])
    print(f"  {loc:<12} -> {conns}")

## 3. Herramientas del Narrator + herramientas del WorldBuilder

Separamos las herramientas en dos grupos:
- **Narrator tools**: las mismas de v1 (`check_state`, `move_to`, `attack_enemy`, etc.)
- **WorldBuilder tools**: nuevas herramientas para expandir el mundo (`create_location`, `create_npc`, `create_item`)

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Estado mutable del juego + mundo dinámico
# ────────────────────────────────────────────────────────────────────────────

game_state = {}
world = {}  # mundo mutable (copia de WORLD_INITIAL + expansiones)


def reset_game():
    """Reinicia estado del jugador y mundo."""
    global game_state, world
    game_state = {
        "hp": INITIAL_STATE["hp"], "max_hp": INITIAL_STATE["max_hp"],
        "gold": INITIAL_STATE["gold"],
        "inventory": list(INITIAL_STATE["inventory"]),
        "location": INITIAL_STATE["location"],
        "events": [], "items_taken": [], "enemies_defeated": [],
        "locations_created": [],  # tracking de expansiones
        "npcs_created": [],
        "items_created": [],
        "turn": 0,          # contador de jugadas (para fechar lo creado)
        "momento": MOMENTOS[0],  # v3: momento del día (avanza con las jugadas)
        "chronicle": "",    # crónica: resumen de lo que salió de la ventana de contexto
    }
    world = copy.deepcopy(WORLD_INITIAL)
    # Registro del mundo: cada entidad guarda sus características, dónde está,
    # cuándo se creó y su estado. Vive en Python, no en el contexto del LLM.
    for loc, data in world["locations"].items():
        data.setdefault("created_turn", 0)
        for npc in data["npcs"]:
            world["npcs"][npc].update({"location": loc, "status": "presente",
                                       "dynamic": False, "created_turn": 0})
            world["npcs"][npc].setdefault("description", world["npcs"][npc]["dialogue"])
    world["items"] = {}
    for loc, data in world["locations"].items():
        for item in data["items"]:
            world["items"][item] = {"description": "", "location": loc,
                                    "status": "en el lugar", "dynamic": False, "created_turn": 0}


def _roll(notation: str) -> int:
    notation = notation.strip().lower()
    m = re.match(r"(\d+)d(\d+)([+-]\d+)?", notation)
    if not m:
        return random.randint(1, 20)
    n, sides, mod = int(m.group(1)), int(m.group(2)), int(m.group(3) or 0)
    return sum(random.randint(1, sides) for _ in range(n)) + mod


MOMENTOS = ["mañana", "mediodía", "tarde", "atardecer", "noche"]
JUGADAS_POR_MOMENTO = 6  # v3: el tiempo avanza cada 6 jugadas


def avanzar_tiempo():
    i = min(game_state["turn"] // JUGADAS_POR_MOMENTO, len(MOMENTOS) - 1)
    game_state["momento"] = MOMENTOS[i]


def _state_summary() -> str:
    inv = ", ".join(game_state["inventory"]) if game_state["inventory"] else "(vacio)"
    return (
        f"[HP: {game_state['hp']}/{game_state['max_hp']} | "
        f"Oro: {game_state['gold']} | "
        f"Ubicacion: {game_state['location']} | "
        f"Momento: {game_state.get('momento', 'mañana')} | "
        f"Inventario: {inv}]"
    )


def _world_summary() -> str:
    locs = list(world["locations"].keys())
    dynamic = [l for l, d in world["locations"].items() if d.get("dynamic")]
    return f"Ubicaciones: {len(locs)} ({len(dynamic)} creadas dinámicamente): {', '.join(locs)}"


def _world_memory() -> str:
    """Memoria compacta del mundo para los system prompts.
    Siempre incluye lo creado dinámicamente, los NPCs y la crónica,
    aunque los mensajes donde ocurrió ya no estén en la ventana de contexto."""
    lines = []
    dyn_locs = [(k, v) for k, v in world["locations"].items() if v.get("dynamic")]
    if dyn_locs:
        lines.append("Lugares descubiertos:")
        for k, v in dyn_locs:
            lines.append(f"- {k} (jugada {v['created_turn']}, conecta con {', '.join(v['connections'])}): "
                         f"{v['description'][:100]}")
    dyn_npcs = [(k, v) for k, v in world["npcs"].items() if v.get("dynamic")]
    if dyn_npcs:
        lines.append("Personajes creados:")
        for k, v in dyn_npcs:
            lines.append(f"- {k} en {v['location']} ({'hostil' if v['hostile'] else 'amigable'}, "
                         f"{v['status']}): {v['description'][:80]}")
    dyn_items = [(k, v) for k, v in world["items"].items() if v.get("dynamic")]
    if dyn_items:
        lines.append("Objetos creados: " + "; ".join(f"{k} ({v['location']}, {v['status']})"
                                                      for k, v in dyn_items))
    if game_state["chronicle"]:
        lines.append(f"Crónica:\n{game_state['chronicle']}")
    return "\n".join(lines) or "(nada creado todavía)"


def _similar_location(key: str):
    """Devuelve una ubicación existente con nombre casi igual (evita duplicados)."""
    words = set(key.split("_"))
    for existing in world["locations"]:
        ex_words = set(existing.split("_"))
        overlap = len(words & ex_words) / max(len(words | ex_words), 1)
        if overlap >= 0.5 and world["locations"][existing].get("dynamic"):
            return existing
    return None


# ── Narrator tools (mismas de v1) ──────────────────────────────────────────

def _norm_txt(t):
    return unicodedata.normalize("NFKD", t.lower()).encode("ascii", "ignore").decode().strip()


@tool
def roll_dice(notation: str) -> str:
    """Tira dados con notacion estandar (ej: '1d20', '2d6+3')."""
    return f"Tiraste {notation}: resultado = {_roll(notation)}"


@tool
def check_state() -> str:
    """Consulta el estado actual del jugador: HP, oro, inventario, ubicacion y conexiones."""
    loc = game_state["location"]
    loc_data = world["locations"][loc]
    available_items = [i for i in loc_data["items"] if i not in game_state["items_taken"]]
    npcs_here = [n for n in loc_data["npcs"] if n not in game_state["enemies_defeated"]]
    connections = ", ".join(loc_data["connections"])
    return (
        f"Estado del jugador: {_state_summary()}\n"
        f"Descripcion del lugar: {loc_data['description']}\n"
        f"Objetos aqui: {available_items or 'ninguno'}\n"
        f"NPCs aqui: {npcs_here or 'nadie'}\n"
        f"Puedes ir a: {connections}\n"
        f"Eventos previos: {game_state['events'][-5:] if game_state['events'] else 'ninguno'}\n"
        f"Mundo: {_world_summary()}"
    )


@tool
def talk_to(npc_name: str) -> str:
    """Habla con un NPC de la ubicacion actual."""
    loc = game_state["location"]
    npcs_here = [n for n in world["locations"][loc]["npcs"] if n not in game_state["enemies_defeated"]]
    npc = next((n for n in npcs_here if npc_name.strip().lower() in n.lower() or n.lower() in npc_name.lower()), None)
    if not npc:
        return f"No hay nadie llamado '{npc_name}' aqui. NPCs: {npcs_here or 'nadie'}"
    dialogo = world["npcs"][npc].get("dialogue", "...")
    game_state["events"].append(f"Hablo con {npc}: {dialogo}")
    return f'{npc} dice: "{dialogo}"'


@tool
def buy_item(item_name: str) -> str:
    """Compra un objeto o servicio a un comerciante de la ubicacion actual (precio fijo del catalogo)."""
    loc = game_state["location"]
    pedido = unicodedata.normalize("NFKD", item_name.lower()).encode("ascii", "ignore").decode()
    comerciantes = [n for n in world["locations"][loc]["npcs"]
                    if n not in game_state["enemies_defeated"] and world["npcs"][n].get("vende")]
    if not comerciantes:
        return f"NO SE COMPRO: aqui nadie vende nada. NPCs: {world['locations'][loc]['npcs'] or 'nadie'}"
    for npc in comerciantes:
        for articulo, datos in world["npcs"][npc]["vende"].items():
            palabras = [w for w in articulo.split() if len(w) >= 4]
            if articulo in pedido or any(w[:4] in pedido for w in palabras):  # "afile" → afilar
                precio = datos["precio"]
                if game_state["gold"] < precio:
                    return (f"NO SE COMPRO: {articulo} cuesta {precio} de oro y solo tienes {game_state['gold']}. "
                            f"{_state_summary()}")
                if "servicio" in datos:
                    antes, despues = datos["servicio"]
                    if antes not in game_state["inventory"]:
                        return f"NO SE COMPRO: {npc} necesita tu {antes} para eso. {_state_summary()}"
                    game_state["inventory"][game_state["inventory"].index(antes)] = despues
                else:
                    game_state["inventory"].append(articulo)
                game_state["gold"] -= precio
                game_state["events"].append(f"Compro {articulo} a {npc} por {precio} de oro")
                return f"COMPRA HECHA: {articulo} a {npc} por {precio} de oro. {_state_summary()}"
    catalogo = {n: list(world["npcs"][n]["vende"]) for n in comerciantes}
    return f"NO SE COMPRO: nadie aqui vende '{item_name}'. Catalogo: {catalogo}"


@tool
def consult_world(topic: str = "") -> str:
    """Consulta el registro permanente del mundo: lugares descubiertos, personajes, objetos,
    todos los eventos y la cronica de la aventura. Usalo cuando el jugador pregunte por el
    pasado, por lugares o personajes que ya conocio, o pida un resumen. 'topic' filtra
    (ej: 'goblin', 'sendero'); vacio devuelve todo."""
    # Memoria como grafo: con tema → subgrafo relevante; sin tema → resumen global + eventos
    if topic.strip():
        return graph_context(topic)
    eventos = "\n".join(f"{i}. {e}" for i, e in enumerate(game_state["events"], 1))
    return f"EVENTOS EN ORDEN (verdad del juego):\n{eventos}\n\n" + graph_summary()



@tool
def move_to(destination: str) -> str:
    """Mueve al jugador a una ubicacion conectada."""
    dest = destination.strip().lower()
    current = game_state["location"]
    connections = world["locations"][current]["connections"]
    if dest not in connections:
        return f"No puedes ir a '{dest}' desde {current}. Destinos validos: {', '.join(connections)}"
    game_state["location"] = dest
    game_state["events"].append(f"Se movio de {current} a {dest}")
    loc_data = world["locations"][dest]
    npcs = [n for n in loc_data["npcs"] if n not in game_state["enemies_defeated"]]
    items = [i for i in loc_data["items"] if i not in game_state["items_taken"]]
    return (
        f"Te mueves a: {dest}. {loc_data['description']}\n"
        f"NPCs aqui: {npcs or 'nadie'}. Objetos: {items or 'ninguno'}.\n"
        f"{_state_summary()}"
    )


@tool
def pick_up_item(item_name: str) -> str:
    """Recoge un objeto de la ubicacion actual."""
    item = item_name.strip().lower()
    loc = game_state["location"]
    available = [i for i in world["locations"][loc]["items"] if i not in game_state["items_taken"]]
    available_lower = [i.lower() for i in available]
    if item not in available_lower:  # v3: "tablero de ajedrez" → "viejo tablero de ajedrez" (si es uno solo)
        parecidos = [i for i in available_lower if item and (item in i or i in item)] or \
                    [i for i in available_lower if any(w in re.findall(r"[a-z]+", _norm_txt(item)) for w in _norm_txt(i).split()
                                                       if len(w) >= 5)]
        if len(parecidos) != 1:
            llevo = [i for i in game_state["inventory"] if any(w in re.findall(r"[a-z]+", _norm_txt(item))
                                                               for w in _norm_txt(i).split() if len(w) >= 5)]
            if llevo:
                return f"Ya llevas {llevo[0]}."
            return f"No hay eso aqui. Objetos disponibles: {available or 'ninguno'}"
        item = parecidos[0]
    idx = available_lower.index(item)
    real_name = available[idx]
    game_state["items_taken"].append(real_name)
    oro = re.search(r"(\d+) monedas de oro", real_name)
    if oro:  # las monedas suman al oro, no van al inventario como objeto
        game_state["gold"] += int(oro.group(1))
        game_state["events"].append(f"Recogio: {real_name} (+{oro.group(1)} de oro)")
        if real_name in world["items"]:
            world["items"][real_name]["status"] = "recogido"
        return f"Recogiste las monedas. Oro total (ya actualizado): {game_state['gold']}. {_state_summary()}"
    game_state["inventory"].append(real_name)
    game_state["events"].append(f"Recogio: {real_name}")
    if real_name in world["items"]:
        world["items"][real_name]["status"] = "en el inventario"
    return f"Recogiste: {real_name}. {_state_summary()}"


@tool
def drop_item(item_name: str) -> str:
    """Suelta, tira o lanza un objeto del inventario: queda en la ubicación actual."""
    pedido = _norm_txt(item_name)
    inv = game_state["inventory"]
    cand = [i for i in inv if pedido and (pedido in _norm_txt(i) or any(w in re.findall(r"[a-z]+", pedido) for w in _norm_txt(i).split()
                                                                       if len(w) >= 4))]
    if not cand:
        return f"No llevas eso. Inventario: {', '.join(inv) or 'nada'}"
    obj = cand[0]
    inv.remove(obj)
    loc = game_state["location"]
    if obj in game_state["items_taken"]:
        game_state["items_taken"].remove(obj)
    if obj not in world["locations"][loc]["items"]:
        world["locations"][loc]["items"].append(obj)
    world["items"].setdefault(obj, {"description": obj, "location": loc})
    world["items"][obj].update(status="en el suelo", location=loc)
    game_state["events"].append(f"Solto: {obj} (en {loc})")
    return f"Soltaste: {obj}. Queda en {loc}. {_state_summary()}"


@tool
def use_potion() -> str:
    """Usa una pocion de curacion del inventario."""
    if "pocion de curacion" not in game_state["inventory"]:
        return f"No tienes pociones. Inventario: {game_state['inventory']}"
    if game_state["hp"] >= game_state["max_hp"]:  # v3: no se desperdicia
        return f"Ya tienes la vida al maximo ({game_state['hp']}/{game_state['max_hp']}): guardas la pocion."
    heal = _roll("2d4+2")
    game_state["inventory"].remove("pocion de curacion")
    old_hp = game_state["hp"]
    game_state["hp"] = min(game_state["hp"] + heal, game_state["max_hp"])
    game_state["events"].append(f"Uso pocion: +{heal} HP ({old_hp} -> {game_state['hp']})")
    return f"Bebiste la pocion. Recuperas {heal} HP. {_state_summary()}"


@tool
def attack_enemy(enemy_name: str) -> str:
    """Resuelve un turno de combate contra un enemigo."""
    loc = game_state["location"]
    npcs_here = [n for n in world["locations"][loc].get("npcs", [])
                 if n not in game_state["enemies_defeated"]]
    enemy = None
    pedido = _norm_txt(enemy_name)
    for npc in npcs_here:  # "goblin", "Goblin centinela" o "ataco al goblin" → Goblin centinela
        if pedido in _norm_txt(npc) or any(w in re.findall(r"[a-z]+", pedido) for w in _norm_txt(npc).split() if len(w) >= 4):
            enemy = npc
            break
    if not enemy:
        return f"No hay ningun enemigo asi aqui. Presentes: {npcs_here or 'nadie'}"
    npc_data = world["npcs"][enemy]
    if not npc_data.get("hostile", False):
        return f"{enemy} no es hostil."
    # v3: combate D&D 5e simplificado con los stats del SRD: d20 + bonificador contra CA; los PG bajan por golpe
    rc = REGLAS_COMBATE
    pg_max = npc_data.get("hp", 7)
    npc_data.setdefault("hp_actual", pg_max)
    ca_enemigo = npc_data.get("defense", 10)
    arma = next((a for a in game_state["inventory"] if a in rc["dano_arma"]), None)
    dado_dano = rc["dano_arma"][arma] if arma else rc["dano_sin_arma"]
    d20 = random.randint(1, 20)
    total = d20 + rc["bonificador_ataque"]
    acierta = d20 == 20 or (d20 != 1 and total >= ca_enemigo)
    dano = (_roll(dado_dano) + (_roll(dado_dano) if d20 == 20 else 0)) if acierta else 0
    result = f"--- COMBATE vs {enemy} ---\n"
    result += (f"Tu ataque{f' con {arma}' if arma else ''}: d20={d20}+{rc['bonificador_ataque']}={total} vs CA {ca_enemigo} -> "
               f"{'CRÍTICO' if d20 == 20 else 'ACIERTO' if acierta else 'FALLO'}")
    if acierta:
        npc_data["hp_actual"] = max(0, npc_data["hp_actual"] - dano)
        result += f", {dano} de daño (PG de {enemy}: {npc_data['hp_actual']}/{pg_max})"
    if acierta and npc_data["hp_actual"] <= 0:
        game_state["enemies_defeated"].append(enemy)
        game_state["events"].append(f"Derroto a {enemy}")
        npc_data["status"] = "derrotado"
        result += f"\n¡{enemy} ha sido derrotado!"
    else:
        ca_jugador = rc["ca_jugador"] + (rc["bonus_escudo"] if "escudo de hierro" in game_state["inventory"] else 0)
        e20 = random.randint(1, 20)
        etotal = e20 + npc_data.get("to_hit", 3)
        ealcanza = e20 == 20 or (e20 != 1 and etotal >= ca_jugador)
        edano = _roll(npc_data.get("attack", "1d6")) if ealcanza else 0
        result += (f"\nAtaque de {enemy}: d20={e20}+{npc_data.get('to_hit', 3)}={etotal} vs tu CA {ca_jugador} -> "
                   f"{'ACIERTO' if ealcanza else 'FALLO'}")
        if ealcanza:
            game_state["hp"] = max(0, game_state["hp"] - edano)
            result += f", {edano} de daño a ti"
            if game_state["hp"] == 0:  # v3: antes llegar a 0 PG no tenía ninguna consecuencia
                game_state["events"].append(f"Cayo inconsciente en combate contra {enemy}")
                result += "\n¡Caes inconsciente! Despiertas más tarde, maltrecho, con 1 PG."
                game_state["hp"] = 1
        game_state["events"].append(f"Combate con {enemy}: {'le hiciste ' + str(dano) + ' de daño' if acierta else 'fallaste'}"
                                    f" (PG {npc_data['hp_actual']}/{pg_max})")
        result += f"\n{enemy} sigue en pie (PG {npc_data['hp_actual']}/{pg_max})."
    result += f"\n{_state_summary()}"
    return result


# ── WorldBuilder tools (NUEVAS en v2) ─────────────────────────────────────

@tool
def create_location(name: str, description: str, connected_to: str) -> str:
    """Crea una nueva ubicacion en el mundo y la conecta a una ubicacion existente.
    Usa esto cuando el jugador descubre un lugar nuevo (puerta escondida, pasadizo, etc.).
    El nombre debe ser en minusculas sin acentos (ej: 'pasadizo_secreto')."""
    key = name.strip().lower().replace(" ", "_")
    # Lo que se descubre está donde está el jugador (si no, sería inalcanzable)
    parent = game_state["location"]
    if key in world["locations"]:
        return f"La ubicacion '{key}' ya existe."
    similar = _similar_location(key)
    if similar:
        return (f"NO CREADA: ya existe '{similar}', muy parecida a '{key}'. "
                f"Reutiliza '{similar}' en vez de duplicarla.")
    if parent not in world["locations"]:
        return f"La ubicacion '{parent}' no existe. Ubicaciones: {list(world['locations'].keys())}"
    world["locations"][key] = {
        "description": description,
        "connections": [parent],
        "items": [],
        "npcs": [],
        "dynamic": True,
        "created_turn": game_state["turn"],
    }
    world["locations"][parent]["connections"].append(key)
    game_state["locations_created"].append(key)
    game_state["events"].append(f"Descubrio nueva ubicacion: {key} (conectada a {parent})")
    return (
        f"NUEVA UBICACION CREADA: '{key}'\n"
        f"Descripcion: {description}\n"
        f"Conectada a: {parent} (bidireccional)\n"
        f"Ahora puedes moverte ahi con move_to.\n"
        f"{_world_summary()}"
    )


@tool
def create_npc(name: str, description: str, location: str, hostile: bool) -> str:
    """Crea un nuevo NPC en una ubicacion. Usa esto para poblar zonas nuevas o añadir
    personajes cuando el jugador busca interaccion."""
    loc = location.strip().lower()
    if loc not in world["locations"]:
        return f"La ubicacion '{loc}' no existe."
    if name in world["npcs"]:
        return f"El NPC '{name}' ya existe."
    npc_data = {"dialogue": description, "description": description, "hostile": hostile,
                "location": loc, "status": "presente", "dynamic": True,
                "created_turn": game_state["turn"]}
    if hostile:
        npc_data.update({"hp": random.randint(8, 15), "attack": "1d6", "defense": 10})
    world["npcs"][name] = npc_data
    world["locations"][loc]["npcs"].append(name)
    game_state["npcs_created"].append(name)
    game_state["events"].append(f"Nuevo NPC: {name} en {loc}")
    return f"NPC CREADO: '{name}' en {loc} ({'hostil' if hostile else 'amigable'}). {description}"


@tool
def create_item(name: str, description: str, location: str) -> str:
    """Crea un nuevo objeto en una ubicacion. Usa esto para añadir tesoros, llaves,
    armas u otros objetos interesantes."""
    loc = location.strip().lower()
    if loc not in world["locations"]:
        return f"La ubicacion '{loc}' no existe."
    name = re.sub(r"^(un|una|unos|unas|el|la|los|las)\s+", "", name.strip().replace("_", " "), flags=re.I)  # "un vial" → "vial"
    world["locations"][loc]["items"].append(name)
    world["items"][name] = {"description": description, "location": loc, "status": "en el lugar",
                            "dynamic": True, "created_turn": game_state["turn"]}
    game_state["items_created"].append(name)
    game_state["events"].append(f"Nuevo objeto: {name} en {loc}")
    return f"OBJETO CREADO: '{name}' en {loc}. {description}"


# Dos conjuntos de herramientas
NARRATOR_TOOLS = [roll_dice, check_state, move_to, pick_up_item, use_potion, attack_enemy, talk_to, consult_world]
V1_TOOLS = [roll_dice, check_state, move_to, pick_up_item, use_potion, attack_enemy]  # v1 sin memoria
WORLD_TOOLS = [create_location, create_npc, create_item, check_state]
ALL_TOOLS = list({t.name: t for t in NARRATOR_TOOLS + WORLD_TOOLS}.values())

print(f"Narrator tools: {[t.name for t in NARRATOR_TOOLS]}")
print(f"WorldBuilder tools: {[t.name for t in WORLD_TOOLS]}")
print(f"Total herramientas únicas: {len(ALL_TOOLS)}")

### Memoria del mundo como grafo (GraphRAG local)

El mundo ya es un grafo: lugares conectados, NPCs y objetos que están en un lugar, un jugador que posee, derrota y habla con otros personajes. Guardarlo como grafo (`networkx`) conserva las **relaciones** entre entidades, no solo sus características, aunque salgan de la ventana de contexto.

- **Capa estructural** (sin LLM): se reconstruye del estado del juego en cada jugada → 100% confiable.
- **Capa narrativa** (cronista con salida estructurada): tripletas `(sujeto, relación, objeto)` de lo que solo existió en la narración.
- **Recuperación** (*local search*): entidades semilla (ubicación actual + entidades mencionadas, por nombre o por similitud con embeddings `bge-m3`) → vecindario a 1 salto → texto para el prompt. Si la acción nombra otro lugar, se agrega la **ruta más corta** del grafo (verificable, no inventada).


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Memoria del mundo como grafo de conocimiento (GraphRAG "local search")
# ────────────────────────────────────────────────────────────────────────────
# El mundo YA es un grafo: lugares conectados, NPCs y objetos que están en un lugar,
# el jugador que posee, derrota, habla con... Guardarlo como grafo conserva las RELACIONES
# entre entidades (no solo sus características) aunque salgan del contexto del LLM.
#
# Dos capas de aristas:
#   - estructural (fuente="herramienta"): se reconstruye del estado del juego en cada jugada.
#     Sin LLM → 100% confiable.
#   - narrativa (fuente="cronista"): tripletas (sujeto, relación, objeto) que el cronista extrae
#     de la narración con salida estructurada (lo que solo existió en el texto: advertencias,
#     pistas, promesas...).
#
# Recuperación (local search de GraphRAG): entidades semilla (ubicación actual + entidades
# mencionadas en la acción, por nombre o por similitud con embeddings bge-m3) → vecindario a
# 1 salto → se serializa como texto para el prompt. Si la acción nombra otro lugar, se agrega
# la ruta más corta (shortest_path), que es verificable.

import networkx as nx
import numpy as np
from langchain_ollama import OllamaEmbeddings

embedder = OllamaEmbeddings(model=EMBED_MODEL, base_url=EMBED_BASE_URL)

G = nx.MultiDiGraph()
narrative_triples = []   # [(sujeto, relación, objeto, jugada)] extraídas por el cronista
_emb_cache = {}          # texto del nodo → embedding (solo se calcula una vez por texto)


def _norm(text):
    """minúsculas, sin acentos, '_' → espacio."""
    t = unicodedata.normalize("NFKD", str(text).lower()).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", t.replace("_", " ")).strip()


def reset_graph():
    global G, narrative_triples, _emb_cache
    G = nx.MultiDiGraph()
    narrative_triples = []


def _add_edge(u, rel, v, fuente="herramienta", **attrs):
    if not G.has_node(u):
        G.add_node(u, tipo="concepto", descripcion="")
    if not G.has_node(v):
        G.add_node(v, tipo="concepto", descripcion="")
    G.add_edge(u, v, rel=rel, fuente=fuente, **attrs)


def sync_graph():
    """Reconstruye la capa estructural desde el estado del juego y agrega la narrativa."""
    G.clear()
    G.add_node("jugador", tipo="jugador",
               descripcion=f"HP {game_state['hp']}/{game_state['max_hp']}, oro {game_state['gold']}")
    for loc, d in world["locations"].items():
        G.add_node(loc, tipo="lugar", descripcion=d["description"], dynamic=d.get("dynamic", False),
                   created_turn=d.get("created_turn", 0))
    for name, d in world["npcs"].items():
        G.add_node(name, tipo="npc", descripcion=d.get("personalidad") or d.get("description", d.get("dialogue", "")),
                   hostil=d.get("hostile", False), estado=d.get("status", "presente"),
                   dynamic=d.get("dynamic", False), created_turn=d.get("created_turn", 0))
    for name, d in world["items"].items():
        G.add_node(name, tipo="objeto", descripcion=d.get("description", ""), estado=d.get("status", ""),
                   dynamic=d.get("dynamic", False), created_turn=d.get("created_turn", 0))
    for item in game_state["inventory"]:
        if not G.has_node(item):
            G.add_node(item, tipo="objeto", descripcion="", estado="en el inventario")

    for loc, d in world["locations"].items():
        for other in d["connections"]:
            _add_edge(loc, "conecta_con", other)
    for name, d in world["npcs"].items():
        _add_edge(name, "esta_en", d["location"])
    for name, d in world["items"].items():
        if d.get("status") == "en el lugar":
            _add_edge(name, "esta_en", d["location"])
    _add_edge("jugador", "esta_en", game_state["location"])
    for item in game_state["inventory"]:
        _add_edge("jugador", "posee", item)
    for e in game_state["events"]:
        if m := re.match(r"Derroto a (.+)", e):
            _add_edge("jugador", "derroto", m.group(1))
        elif m := re.match(r"Se movio de (\S+) a (\S+)", e):
            _add_edge("jugador", "visito", m.group(2))
        elif m := re.match(r"Hablo con (.+?): (.+)", e):
            _add_edge("jugador", "hablo_con", m.group(1), dijo=m.group(2))
        elif m := re.match(r"Descubrio nueva ubicacion: (\S+)", e):
            _add_edge("jugador", "descubrio", m.group(1))
        elif m := re.match(r"Compro (.+?) a (.+?) por (\d+) de oro", e):
            _add_edge("jugador", "compro", m.group(1), a=m.group(2), precio=int(m.group(3)))
        elif m := re.match(r"Uso pocion", e):
            _add_edge("jugador", "uso", "pocion de curacion")

    for name, d in world["npcs"].items():  # v3: relaciones entre personajes (contenido de autor)
        for r in d.get("relaciones", []):
            if r["con"] in world["npcs"]:
                _add_edge(name, "conoce", r["con"], relacion=r["relacion"], detalle=r["detalle"])
    for npc in {d["npc"] for d in game_state.get("dialogos", [])}:  # v3: a quién conoce el jugador
        _add_edge("jugador", "conoce", npc)
    for d in game_state.get("dialogos", []):  # lo que cada NPC le dijo realmente al jugador
        _add_edge(d["npc"], "dijo_a_jugador", "jugador", dijo=d["dijo"], jugada=d["jugada"])
    for clave, p in game_state.get("especulativos", {}).items():  # v3: pistas narrativas (aún no confirmadas)
        if not p.get("confirmada"):
            G.add_node(clave, tipo="pista", speculative=True,
                       descripcion=f"PISTA no confirmada: {p['mencionado_por']} {p['relacion'].replace('_', ' ')} {p['nombre']}")
    confirmadas = {k: p["confirmada"] for k, p in game_state.get("especulativos", {}).items() if p.get("confirmada")}
    for s, rel, o, t in narrative_triples:  # una pista confirmada apunta ya a la entidad real que creó el WorldBuilder
        s, o = confirmadas.get(s, s), confirmadas.get(o, o)
        _add_edge(s if s in G else (_resolve(s) or s), rel, o if o in G else (_resolve(o) or o), fuente="cronista", jugada=t)


def _resolve(name):
    """Nodo existente que corresponde a un nombre mencionado ('Goruk' → 'Goruk el tabernero')."""
    n = _norm(name)
    if not n:
        return None
    for node in G.nodes:
        nn = _norm(node)
        if n == nn or (len(n) >= 4 and (n in nn or nn in n)):
            return node
    return None


def _node_text(node):
    d = G.nodes[node]
    return f"{node} ({d.get('tipo', '')}): {d.get('descripcion', '')}"


def _semantic_seeds(text, k=2, umbral=0.55):
    """Nodos cuya descripción se parece a la acción (embeddings bge-m3)."""
    nodes = [n for n in G.nodes if G.nodes[n].get("tipo") in ("lugar", "npc", "objeto")]
    textos = [_node_text(n) for n in nodes]
    nuevos = [t for t in textos if t not in _emb_cache]
    if nuevos:
        for t, e in zip(nuevos, embedder.embed_documents(nuevos)):
            _emb_cache[t] = np.array(e) / np.linalg.norm(e)
    q = np.array(embedder.embed_query(text))
    q = q / np.linalg.norm(q)
    sims = sorted(((float(_emb_cache[t] @ q), n) for t, n in zip(textos, nodes)), reverse=True)
    return [n for s, n in sims[:k] if s >= umbral]


def _edge_txt(u, v, d):
    extra = f' (dijo: "{d["dijo"]}")' if d.get("dijo") else ""
    extra += f" ({d['relacion']}: {d['detalle']})" if d.get("relacion") else ""
    extra += f" (a {d['a']} por {d['precio']} de oro)" if d.get("precio") else ""
    extra += f" [narración, jugada {d['jugada']}]" if d.get("fuente") == "cronista" else ""
    return f"{u} —{d['rel']}→ {v}{extra}"


def _descubiertos_txt():
    """Lugares creados en la partida, numerados y delimitados (evita que el modelo los mezcle:
    'lost in the middle'). No cuenta los lugares ya conocidos que solo se visitaron."""
    desc = [(n, d) for n, d in G.nodes(data=True) if d.get("dynamic") and d.get("tipo") == "lugar"]
    if not desc:
        return "Lugares DESCUBIERTOS en la aventura: 0 (ninguno)"
    bloques = [f"  [{i}] {n} | se llega desde: {world['locations'][n]['connections'][0]} | "
               f"jugada {d.get('created_turn')} | {d.get('descripcion', '')[:90]}"
               for i, (n, d) in enumerate(sorted(desc, key=lambda x: x[1].get("created_turn", 0)), 1)]
    return f"Lugares DESCUBIERTOS en la aventura: {len(desc)}\n" + "\n".join(bloques)


def graph_context(accion, max_aristas=30):
    """Subgrafo relevante a la acción, serializado como texto (local search)."""
    sync_graph()
    loc = game_state["location"]
    semillas = {"jugador", loc}
    semillas |= {n for n in world["locations"][loc]["npcs"] if n not in game_state["enemies_defeated"]}  # v3: con su perfil
    for node in G.nodes:  # entidades mencionadas por nombre
        nn = _norm(node)
        if len(nn) >= 4 and (nn in _norm(accion) or any(w in _norm(accion).split() for w in nn.split() if len(w) >= 5)):
            semillas.add(node)
    try:
        semillas |= set(_semantic_seeds(accion))
    except Exception:
        pass  # sin embeddings, la recuperación por nombre sigue funcionando

    aristas, vistos = [], set()
    for s in semillas:
        for u, v, d in list(G.out_edges(s, data=True)) + list(G.in_edges(s, data=True)):
            key = (u, d["rel"], v)
            if key not in vistos:
                vistos.add(key)
                aristas.append(_edge_txt(u, v, d))
    lineas = [_descubiertos_txt()]
    lineas += ["Entidades relevantes:"] + [f"- {_node_text(n)}" for n in semillas if n != "jugador"]
    lineas += ["Relaciones:"] + [f"- {a}" for a in aristas[:max_aristas]]

    # Rutas: si la acción nombra otro lugar, la ruta más corta (solo por conecta_con)
    lugares = nx.Graph([(u, v) for u, v, d in G.edges(data=True) if d["rel"] == "conecta_con"])
    for destino in [n for n in semillas if G.nodes[n].get("tipo") == "lugar" and n != loc]:
        try:
            lineas.append(f"Ruta más corta de {loc} a {destino}: {' → '.join(nx.shortest_path(lugares, loc, destino))}")
        except (nx.NetworkXNoPath, nx.NodeNotFound):
            lineas.append(f"No hay ruta conocida de {loc} a {destino}.")
    return "\n".join(lineas)


def graph_summary():
    """Resumen global: lo creado dinámicamente + relaciones narrativas (para 'recordar' sin tema)."""
    sync_graph()
    lineas = [_descubiertos_txt()]
    lineas += [f"- {_node_text(n)}" for n, d in G.nodes(data=True) if d.get("dynamic")]
    pistas = [p for p in game_state.get("especulativos", {}).values()]
    lineas += [f"- PISTA {'CONFIRMADA → ' + p['confirmada'] if p.get('confirmada') else '(sin confirmar)'}: "
               f"{p['nombre']} ({p['mencionado_por']} {p['relacion'].replace('_', ' ')}, jugada {p['jugada']})" for p in pistas]
    lineas += [f"- {_edge_txt(u, v, d)}" for u, v, d in G.edges(data=True)
               if d.get("fuente") == "cronista" or d["rel"] in ("derroto", "hablo_con")]
    return "\n".join(lineas) or "(nada registrado todavía)"


def draw_graph(ax=None, titulo="Grafo del mundo"):
    """Dibuja el grafo: color por tipo de entidad, borde grueso = creado dinámicamente."""
    sync_graph()
    colores = {"lugar": "#64B5F6", "npc": "#FFB74D", "objeto": "#AED581", "jugador": "#E57373", "concepto": "#CE93D8",
               "pista": "#E0E0E0"}  # v3: pistas especulativas en gris
    H = nx.DiGraph()
    for u, v, d in G.edges(data=True):
        if not H.has_edge(u, v):
            H.add_edge(u, v, rel=d["rel"])
    H.add_nodes_from(G.nodes(data=True))
    if ax is None:
        _, ax = plt.subplots(figsize=(14, 10))
    pos = nx.spring_layout(H, seed=7, k=1.2)
    nx.draw_networkx_nodes(H, pos, ax=ax, node_size=900,
                           node_color=[colores.get(H.nodes[n].get("tipo"), "#BDBDBD") for n in H],
                           linewidths=[3 if H.nodes[n].get("dynamic") else 0.5 for n in H],
                           edgecolors=["#9E9E9E" if H.nodes[n].get("speculative") else "#333" for n in H])
    nx.draw_networkx_labels(H, pos, ax=ax, font_size=8,
                            labels={n: f"{n} ?" if H.nodes[n].get("speculative") else n for n in H})
    nx.draw_networkx_edges(H, pos, ax=ax, alpha=0.5, arrows=True, arrowsize=10)
    nx.draw_networkx_edge_labels(H, pos, ax=ax, font_size=6,
                                 edge_labels={(u, v): d["rel"] for u, v, d in H.edges(data=True)})
    ax.set_title(titulo)
    ax.axis("off")
    return ax


print("Grafo de conocimiento listo (networkx + embeddings bge-m3 para entidades semilla)")


### RAG vectorial sobre el SRD de D&D (conocimiento estático)

El mundo del juego es conocimiento **dinámico** → grafo. Las reglas de D&D son conocimiento **estático** → RAG vectorial clásico:
- Corpus: SRD 5.1 (CC-BY-4.0), 1285 documentos (monstruos, objetos mágicos, equipo, hechizos, reglas).
- Índice: ChromaDB + embeddings `bge-m3` (multilingüe: consultas en español sobre texto en inglés). Búsqueda híbrida: nombre exacto + similitud.
- **Narrator**: acción `reglas` → responde preguntas de reglas con el texto oficial en vez de inventar.
- **WorldBuilder**: recupera criaturas/objetos reales antes de crear; los NPCs creados heredan CA, PG y daño del SRD.


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# RAG vectorial sobre el SRD de D&D 5e (conocimiento ESTÁTICO: reglas, monstruos, objetos)
# ────────────────────────────────────────────────────────────────────────────
# Fuente: System Reference Document 5.1 (CC-BY-4.0), JSON de 5e-bits/5e-database.
# 1285 documentos (334 monstruos, 362 objetos mágicos, 237 equipo, 319 hechizos, 33 reglas),
# preparados por data/srd/build_docs_v3.py → data/srd/docs_v3.jsonl.
# Índice: ChromaDB persistente + embeddings bge-m3 (multilingüe: consultas en español sobre
# texto en inglés). Búsqueda híbrida: coincidencia exacta del nombre (ej. "goblin") primero,
# luego similitud semántica.
#
# Uso: (1) el Narrator consulta reglas con la acción 'reglas' en vez de inventarlas;
#      (2) el WorldBuilder recupera criaturas/objetos reales antes de crear contenido.
#
# This work includes material taken from the System Reference Document 5.1 ("SRD 5.1") by
# Wizards of the Coast LLC, licensed under CC-BY-4.0 (https://creativecommons.org/licenses/by/4.0/).

import chromadb
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage, HumanMessage

SRD_DIR = "data/srd"
# v3: docs_v3.jsonl agrega vulnerabilidades/resistencias/condiciones de monstruos y daño versátil/alcance de armas
srd_docs = {d["id"]: d for d in map(json.loads, open(f"{SRD_DIR}/docs_v3.jsonl"))}
# v3 (post hold-out 3): las reglas largas mezclan temas ("The Environment" = caídas + asfixia + visión + comida…) y su
# embedding no se parecía a "fall damage". Cada subsección "### …" se indexa además como documento propio (chunk).
for _d in [d for d in srd_docs.values() if d["type"] == "rule" and "### " in d["text"]]:
    for _sec in _d["text"].split("### ")[1:]:
        _w = _sec.split()  # título = palabras hasta que empieza la primera oración ("Falling A fall from…" → "Falling")
        _fin = next((i for i, x in enumerate(_w) if x[:1].islower() and x not in ("and", "of", "the", "or", "in")), len(_w))
        _titulo = " ".join(_w[:max(1, _fin - 1)])[:40]
        _id = f"{_d['id']}#{_norm(_titulo).replace(' ', '-')}"
        srd_docs[_id] = {"id": _id, "type": "rule", "name": _titulo, "parent": _d["id"], "seccion": True,
                         "text": f"{_d['name']} — {_titulo}. {_sec.strip()}", "metadata": _d.get("metadata", {})}
_chroma = chromadb.PersistentClient(path=f"{SRD_DIR}/chroma")
srd_index = _chroma.get_or_create_collection("srd_bge_m3_v3s", metadata={"hnsw:space": "cosine"})
if srd_index.count() == 0:  # se reutilizan los embeddings ya calculados del índice sin secciones
    try:
        _viejo = _chroma.get_collection("srd_bge_m3_v3").get(include=["embeddings", "metadatas"])
        for i in range(0, len(_viejo["ids"]), 500):
            srd_index.upsert(ids=_viejo["ids"][i:i + 500], embeddings=_viejo["embeddings"][i:i + 500],
                             metadatas=_viejo["metadatas"][i:i + 500])
    except Exception:
        pass

if srd_index.count() < len(srd_docs):  # se construye una sola vez (queda guardado en disco)
    ya = set(srd_index.get(include=[])["ids"])
    ids = [d for d in srd_docs if d not in ya]
    for i in range(0, len(ids), 16):
        lote = ids[i:i + 16]
        for intento in range(3):  # el runner de embeddings de Ollama a veces corta la conexión: reintentar
            try:
                emb = embedder.embed_documents([srd_docs[d]["text"] for d in lote])
                break
            except Exception:
                time.sleep(2)
        else:
            raise RuntimeError("No se pudo indexar el SRD (Ollama/bge-m3)")
        srd_index.upsert(ids=lote, embeddings=emb,
                         metadatas=[{"type": srd_docs[d]["type"], "name": srd_docs[d]["name"]} for d in lote])
print(f"Índice SRD: {srd_index.count()} documentos")


def buscar_srd(consulta, k=3, tipos=None):
    """Top-k documentos del SRD. Si algún documento coincide EXACTAMENTE por nombre, se devuelven solo
    esos (evita mezclar 'giant spider' con 'giant poisonous snake'); si no, similitud semántica."""
    q = _norm(consulta)
    exactos = [d for d in srd_docs.values() if (not tipos or d["type"] in tipos) and not d.get("seccion")
               and len(_norm(d["name"])) >= 4 and re.search(rf"\b{re.escape(_norm(d['name']))}\b", q)]
    exactos.sort(key=lambda d: -len(d["name"]))
    if exactos:
        return exactos[:1]
    res = srd_index.query(query_embeddings=[embedder.embed_query(consulta)], n_results=k + len(exactos),
                          where={"type": {"$in": tipos}} if tipos else None)
    semanticos = [srd_docs[i] for i in res["ids"][0]]
    vistos, out = set(), []
    for d in exactos + semanticos:
        if d["id"] not in vistos:
            vistos.add(d["id"])
            out.append(d)
    return out[:k]


SYSTEM_REESCRITOR = (
    "You translate a Spanish D&D question into the OFFICIAL ENGLISH name of the creature, item or rule in the "
    "D&D 5e SRD. Answer only the English name.\nExamples:\nveneno araña gigante -> giant spider\n"
    "stats de un oso pardo -> brown bear\nqué cura una poción de curación -> potion of healing\n"
    "cómo funciona la ventaja -> advantage and disadvantage\ndaño de una bola de fuego -> fireball")


def reescribir_consulta(consulta):
    """Query rewriting: el SRD está en inglés; 'araña gigante' no coincide por nombre con 'Giant Spider'."""
    try:
        r = llm_strict.invoke([SystemMessage(content=SYSTEM_REESCRITOR), HumanMessage(content=consulta)])
        return r.content.strip().splitlines()[0].strip(" .'\"") or consulta
    except Exception:
        return consulta


@tool
def consultar_reglas(consulta: str) -> str:
    """Consulta las reglas oficiales de D&D 5e (SRD): estadísticas de monstruos, objetos mágicos,
    hechizos y reglas de juego. Usalo cuando el jugador pregunte cómo funciona algo."""
    termino = reescribir_consulta(consulta)
    docs = buscar_srd(f"{termino} {consulta}", k=3)  # término oficial (coincidencia exacta) + consulta original
    game_state.setdefault("srd_contextos", []).append([d.get("parent", d["id"]) for d in docs])
    game_state["ultimo_srd"] = {"jugada": game_state["turn"], "docs": [d["id"] for d in docs]}
    # v3: 1500 caracteres (con 700 no llegaba el aliento del dragón rojo, en el carácter 1056)
    return "REGLAS OFICIALES (SRD 5.1):\n" + "\n".join(f"[{d['id']}] {d['text'][:1500]}" for d in docs)


ACCIONES_SRD_TIPOS = ["monster", "magic_item", "equipment"]  # lo que el WorldBuilder puede instanciar
UMBRAL_ANCLAJE = 0.45  # distancia coseno máxima para aceptar un documento como base de una creación


def stats_desde_srd(doc_id):
    """CA, PG y daño oficiales de un monstruo del SRD."""
    texto = srd_docs[doc_id]["text"]
    ca, pg = re.search(r"CA (\d+)", texto), re.search(r"PG (\d+)", texto)
    dano = re.search(r"Hit: \d+ \((\d+d\d+(?: ?[+-] ?\d+)?)\)", texto)
    to_hit = re.search(r"\+(\d+) to hit", texto)
    out = {"to_hit": int(to_hit.group(1))} if to_hit else {}
    if ca: out["defense"] = int(ca.group(1))
    if pg: out["hp"] = int(pg.group(1))
    if dano: out["attack"] = dano.group(1).replace(" ", "")
    return out


# v3: las fichas de autor con srd_ref usan los stats OFICIALES (antes el goblin tenía 12 PG / CA 10 / 1d6 escritos
# a mano, y el SRD dice 7 PG / CA 15 / 1d6+2: el juego contradecía las reglas que el narrador cita)
for _nombre, _d in WORLD_INITIAL["npcs"].items():
    if _d.get("srd_ref") in srd_docs and not _d.get("stats_propios"):
        _d.update(stats_desde_srd(_d["srd_ref"]))
        print(f"  {_nombre} ← {_d['srd_ref']}: CA {_d.get('defense')}, PG {_d.get('hp')}, daño {_d.get('attack')}")


def _tipo_criatura(doc_id):
    """'Goblin. Small humanoid (goblinoid), ...' → 'humanoid'."""
    m = re.match(r"[^.]+\.\s+\w+\s+(\w+)", srd_docs[doc_id]["text"])
    return m.group(1).lower() if m else ""


def resolver_referencia(ref, tipos):
    """Id válido a partir de lo que escriba el modelo: 'monster:goblin', 'Goblin', 'goblin'..."""
    for parte in re.split(r"[,;|]", ref or ""):
        parte = parte.strip().strip("'\"")
        if parte in srd_docs and srd_docs[parte]["type"] in tipos:
            return parte
        n = _norm(parte)
        for d in srd_docs.values():
            if d["type"] in tipos and len(n) >= 3 and _norm(d["name"]) == n:
                return d["id"]
    return None


def anclar_creacion(tipo, nombre, descripcion, hostil, ref):
    """Documento del SRD en que se basa una criatura u objeto creado (o None si nada encaja).
    1) la referencia que dio el WorldBuilder, si se puede resolver; 2) búsqueda con el nombre y la
    descripción de lo creado, filtrada por tipo; un NPC no hostil se ancla solo a humanoides."""
    if tipo == "npc" and not hostil:
        # v3: el SRD es casi todo bestiario; para el panadero de la esquina recuperaba ochre jelly o shrieker.
        # Un NPC amigable usa el perfil genérico de aldeano (convención de D&D) y su personalidad la crea el WorldBuilder.
        return "monster:commoner"
    tipos = ["monster"] if tipo == "npc" else ["magic_item", "equipment"]
    doc_id = resolver_referencia(ref, tipos)
    if doc_id and (tipo != "npc" or hostil or _tipo_criatura(doc_id) in ("humanoid", "beast")):
        return doc_id  # un NPC amigable no puede quedar con perfil de doppelganger/aberración/demonio
    termino = reescribir_consulta(f"{nombre}: {descripcion}")
    exacto = next((d["id"] for d in srd_docs.values() if d["type"] in tipos and len(_norm(d["name"])) >= 3
                   and re.search(rf"\b{re.escape(_norm(d['name']))}\b", _norm(termino))), None)
    if exacto:  # el término oficial nombra una criatura/objeto del SRD ("giant rat")
        return exacto
    consulta = f"{termino} {nombre} {descripcion}"
    res = srd_index.query(query_embeddings=[embedder.embed_query(consulta)], n_results=15,
                          where={"type": {"$in": tipos}})
    for i, dist in zip(res["ids"][0], res["distances"][0]):
        if tipo == "npc" and not hostil and _tipo_criatura(i) not in ("humanoid", "beast"):
            continue
        if dist <= UMBRAL_ANCLAJE:
            return i
        break
    # Convención de D&D: un aldeano/personaje no combatiente sin perfil propio usa el de "Commoner"
    return "monster:commoner" if tipo == "npc" and not hostil else None

for consulta in ["¿cuánta vida tiene un goblin?", "¿cómo funciona la ventaja?", "araña gigante venenosa"]:
    print(f"  '{consulta}' → {[d['id'] for d in buscar_srd(consulta)]}")


## 4. Grafo Multi-Agente LangGraph

El grafo tiene 3 nodos especializados:

```
START → narrator → should_use_tools? → narrator_tools → narrator
                 → should_expand?    → world_builder → wb_tools → coherence_check → narrator
                 → END
```

1. **Narrator**: narra la historia, usa herramientas de juego (check_state, move_to, etc.)
2. **WorldBuilder**: se activa cuando el jugador explora algo nuevo; crea ubicaciones/NPCs/items
3. **CoherenceChecker**: valida que la expansión sea consistente con el mundo existente

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# 4. Grafo Multi-Agente — ReAct estructurado (Pydantic)
# ────────────────────────────────────────────────────────────────────────────
# v1 usa ReAct con tool calling nativo: el modelo puede "escribir" la llamada como texto
# ("CallCheck_state: {...}") en vez de hacerla. Aquí cada paso del ciclo
# Pensamiento → Acción → Observación es una salida estructurada (Pydantic + JSON Schema):
# Ollama restringe la generación al schema, así que la acción SIEMPRE es una herramienta válida.

from pydantic import BaseModel, Field

MAX_PASOS = 4  # pasos ReAct por jugada

TRAZA = []  # (nodo, datos) de la jugada en curso — para ver qué decide cada agente (play(traza=True))


def _log(nodo, **datos):
    TRAZA.append((nodo, datos))


class Paso(BaseModel):
    """Un paso del ciclo ReAct."""
    intencion: Literal["accion", "pregunta"] = Field(
        description="'pregunta' si el jugador PIDE INFORMACIÓN (aunque no use '¿'); 'accion' si quiere HACER algo.")
    pensamiento: str = Field(description="Razonamiento breve: qué quiere el jugador y qué necesitas saber o hacer.")
    accion: Literal["ver_estado", "mover", "recoger", "soltar", "atacar", "curarse", "hablar", "comprar",
                    "tirar_dados", "recordar", "reglas", "responder"] = Field(
        description="Acción a usar, o 'responder' cuando ya tienes lo necesario para narrar.")
    argumento: str = Field(description=(
        "mover: destino. recoger: objeto. atacar: enemigo. hablar: personaje. comprar: objeto o servicio. "
        "tirar_dados: notación (1d20). "
        "recordar: tema (vacío = todo). reglas: qué consultar (criatura, objeto o regla). "
        "Vacío para ver_estado, curarse y responder."))


class MoveResult(BaseModel):
    """Resultado explícito de un intento de movimiento (el narrador no tiene que interpretarlo)."""
    success: bool
    message: str
    current_location: str  # SIEMPRE la ubicación real tras el intento


class NuevoContenido(BaseModel):
    """Decisión del WorldBuilder."""
    razon: str = Field(description="Por qué crear (o no) algo nuevo aquí.")
    tipo: Literal["ubicacion", "npc", "objeto", "nada"] = Field(
        description="Qué crear. 'nada' si lo que busca ya existe o no tiene sentido.")
    nombre: str = Field(description="Ubicación: minúsculas sin acentos con _ (ej: pasadizo_secreto). NPC/objeto: nombre propio.")
    descripcion: str = Field(description="1-2 oraciones, fantasía medieval, coherente con el mundo.")
    hostil: bool = Field(default=False, description="Solo NPCs: si es hostil.")
    referencia_srd: str = Field(description="id de la REFERENCIA DEL SRD en que te basas (ej. monster:goblin), "
                                            "o '' si ninguna encaja.")


class Veredicto(BaseModel):
    """Decisión del CoherenceChecker."""
    coherente: bool
    problema: str = Field(default="", description="Si no es coherente, qué falla.")


# ── System prompts especializados ─────────────────────────────────────────

SYSTEM_REACT = """\
Eres el Narrator, un Dungeon Master de D&D. Trabajas en pasos: piensas, eliges UNA acción,
ves su resultado (Observación) y repites hasta tener lo necesario; entonces eliges 'responder'.

ACCIONES:
- mover(destino): el jugador pide ir a un lugar ("voy a", "vuelvo a", "entro a"). La herramienta valida si es posible.
- recoger(objeto) · soltar(objeto): tira, deja o lanza algo que lleva · atacar(enemigo) · hablar(personaje)
- curarse() con una poción · tirar_dados(notación)
- comprar(objeto o servicio): pagar a un comerciante del lugar (precio fijo; la herramienta verifica el oro).
- ver_estado(): dónde está, HP, oro, inventario, qué hay en el lugar, a dónde puede ir.
- recordar(tema): el PASADO — lugares descubiertos, personajes conocidos, quién dijo qué, eventos,
  rutas entre lugares, resumen. Consulta el grafo de memoria del mundo.
- reglas(consulta): REGLAS OFICIALES de D&D (SRD) — stats de criaturas, qué hace un objeto o
  hechizo, cómo funciona una regla. Úsalo SIEMPRE que pregunten "según las reglas" o "cómo funciona".
- responder: ya tienes la Observación necesaria.

INTENCIÓN: 'pregunta' si el jugador pide información (¿dónde estoy?, ¿qué hace X?, dime...),
'accion' si quiere hacer algo. Una pregunta SOLO usa ver_estado, recordar o reglas.

REGLAS:
1. Una PREGUNTA, buscar o examinar NUNCA mueve al jugador.
2. Haz SOLO lo que pide la acción del jugador. Normalmente basta 1 paso y luego 'responder'.
3. No repitas una acción que ya hiciste en esta jugada.

EJEMPLOS (acción del jugador → primer paso):
- "Voy a la cueva." → {"accion": "mover", "argumento": "cueva"}
- "Vuelvo al bosque." → {"accion": "mover", "argumento": "bosque"}
- "Recojo la antorcha." → {"accion": "recoger", "argumento": "antorcha"}
- "Ataco al goblin." → {"accion": "atacar", "argumento": "goblin"}
- "Hablo con Goruk." → {"accion": "hablar", "argumento": "Goruk"}
- "Le pido a Bron que me cuente de su oficio." → {"intencion": "accion", "accion": "hablar", "argumento": "Bron"}
- "Le digo a Elena que hace buen día." → {"intencion": "accion", "accion": "hablar", "argumento": "Elena"}
- "Le compro una poción a Elena." → {"intencion": "accion", "accion": "comprar", "argumento": "pocion de curacion"}
- "Pago a Bron para que afile mi espada." → {"intencion": "accion", "accion": "comprar", "argumento": "afilar espada"}
- "Uso mi poción de curación." → {"accion": "curarse", "argumento": ""}
- "¿Cuántos HP tengo?" → {"accion": "ver_estado", "argumento": ""}
- "¿Dónde estoy exactamente?" → {"accion": "ver_estado", "argumento": ""}
- "Miro a mi alrededor." → {"accion": "ver_estado", "argumento": ""}
- "¿Qué lugares descubrí?" → {"accion": "recordar", "argumento": "lugares"}
- "Resume la aventura." → {"accion": "recordar", "argumento": ""}
- "¿Quién me habló del bosque?" → {"accion": "recordar", "argumento": "bosque"}
- "¿Cómo llego a la cueva desde aquí?" → {"accion": "recordar", "argumento": "cueva"}
- "Busco una puerta escondida." → {"accion": "ver_estado", "argumento": ""} (el DESCUBRIMIENTO, si lo hay, ya aparece en los pasos)
- "Según las reglas, ¿cuánta vida tiene un goblin?" → {"accion": "reglas", "argumento": "goblin"}
- "¿Cómo funciona la ventaja?" → {"accion": "reglas", "argumento": "ventaja advantage"}
- "¿Qué hace una poción de curación?" → {"accion": "reglas", "argumento": "potion of healing"}
Después de la Observación → {"accion": "responder", "argumento": ""}"""

SYSTEM_NARRADOR = """\
Eres un Dungeon Master de D&D que narra en español.
Narra en 2-4 oraciones lo que pasó, basándote SOLO en las Observaciones de las herramientas y en
la MEMORIA DEL MUNDO (grafo): son la verdad del juego. No infieras detalles adicionales.
REGLAS:
1. MOVIMIENTO RECHAZADO → NUNCA narres un desplazamiento: di que no puede ir y dónde sigue.
2. No narres acciones que no aparecen en las Observaciones (recoger, hablar, atacar...).
3. Usa EXACTAMENTE los números del estado (HP, oro). NUNCA los recalcules ni sumes.
4. Menciona solo personajes que la memoria ubica en el lugar actual.
5. Preguntas de REGLAS → responde con las REGLAS OFICIALES (SRD) de las Observaciones: cifras y
   efectos exactos (puedes traducir), sin agregar nada que no diga el texto.
6. Si hay un DESCUBRIMIENTO, nárralo. Si el jugador pregunta cómo llegar y hay "Ruta más corta",
   da EXACTAMENTE esa ruta.
7. PREGUNTA del jugador (¿...?) → respóndela; no narres que se mueve ni que hace algo.
8. Resumen de la aventura → sigue los EVENTOS en orden, sin agregar nada.
9. Lugares descubiertos → usa la lista numerada tal cual (nombre, dónde está), sin mezclarlos.
10. Si el jugador pidió hacer algo y NINGUNA Observación muestra que se hizo (o dice "NO SE EJECUTÓ"),
    la acción NO ocurrió: dilo. Nunca narres efectos (curarse, recoger, tirar dados) que no estén en las Observaciones.
11. Si hay un DIÁLOGO de un personaje, NO escribas lo que dice (el sistema agrega su frase textual):
    a lo sumo describe su gesto en una oración. Nunca inventes frases de personajes.
No inventes lugares, objetos, personajes, rutas ni números.
NO escribas la línea de estado [HP | Oro | ...]: la agrega el sistema.

EJEMPLOS:
- Observación: "pick_up_item → No hay 'escudo' aqui" → "Buscas un escudo, pero aquí no hay ninguno."
- Observación: "MOVIMIENTO RECHAZADO → el jugador está en: pantano" → "No hay camino directo desde aquí; sigues en el pantano."
- Observación: "MOVIMIENTO EXITOSO → el jugador está en: molino" → "Llegas al molino: ..." (describe el lugar nuevo)
- Sin Observación de use_potion y el jugador dijo "bebo la poción" → "No llegas a beber la poción: no ocurre nada."
- Observación: "check_state → Oro: 65" y pregunta "¿cuánto oro tengo?" → «Tienes 65 monedas de oro.»"""

SYSTEM_WORLD_BUILDER = """\
Eres el WorldBuilder, un diseñador de mundos de D&D. El jugador está buscando o explorando algo.
Decide si tiene sentido crear UNA cosa nueva en su ubicación actual: una ubicación (pasadizo,
sendero, sótano...), un NPC o un objeto.
- Si el jugador busca algo y NO existe todavía, CREA algo que encaje (es lo que hace un buen DM).
  Solo usa tipo = 'nada' si lo que busca ya existe en la MEMORIA DEL MUNDO o es absurdo para el lugar.
- No dupliques lugares existentes con otro nombre.
- Mantén el tono: fantasía medieval, coherente con lo que ya existe.
- Si creas una criatura u objeto y alguna REFERENCIA DEL SRD encaja, basate en ella (referencia_srd):
  sus estadísticas oficiales se copian automáticamente.

EJEMPLOS (acción del jugador → decisión):
- "Examino las paredes de la cueva buscando pasadizos." → ubicacion: pasadizo_secreto
- "Busco un sendero escondido entre los árboles." → ubicacion: sendero_escondido
- "Busco a alguien interesante que no haya visto antes." → npc: un personaje nuevo con nombre propio
- "Busco una puerta escondida detrás de la barra." → ubicacion: sotano_de_la_taberna
- "Busco algo de valor en el suelo." → objeto
- "Busco el pasadizo que ya encontré." → nada (ya existe)
- "Miro debajo de las mesas." → objeto: algo pequeño que alguien olvidó
- "Reviso los arbustos por si hay un rastro." → ubicacion: una vereda o claro escondido
- "¿Hay alguien más en la taberna?" → npc: un parroquiano con nombre propio y un rasgo memorable
- "Investigo el pozo de la plaza." → ubicacion u objeto, según el lugar"""

SYSTEM_COHERENCE = """\
Eres el verificador de coherencia de un mundo de D&D. Te muestran el mundo actual y una
creación nueva. Es INCOHERENTE si: duplica un lugar existente, contradice la memoria del mundo,
o no encaja con la ubicación donde se creó. En otro caso es coherente."""


# ── LLMs con salida estructurada ──────────────────────────────────────────

llm_narrador = ChatOllama(model=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL, temperature=0.4)  # menos adorno que 0.7
llm_reglas = ChatOllama(model=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL, temperature=0)  # v3: reglas sin variación
llm_planner = ChatOllama(model=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL, temperature=0.3)
planner = llm_planner.with_structured_output(Paso, method="json_schema", include_raw=True)
world_builder_llm = llm_creative.with_structured_output(NuevoContenido, method="json_schema")
coherence_llm = llm_strict.with_structured_output(Veredicto, method="json_schema")

# acción del planner → (herramienta, nombre del parámetro)
ACCIONES = {"ver_estado": (check_state, None), "mover": (move_to, "destination"),
            "recoger": (pick_up_item, "item_name"), "atacar": (attack_enemy, "enemy_name"),
            "curarse": (use_potion, None), "tirar_dados": (roll_dice, "notation"),
            "hablar": (talk_to, "npc_name"), "recordar": (consult_world, "topic"),
            "reglas": (consultar_reglas, "consulta"), "comprar": (buy_item, "item_name"),
            "soltar": (drop_item, "item_name")}

EXPLORACION = re.compile(r"busc|examin|explor|investig|escondid|secret|ocult|rastre", re.IGNORECASE)
PREGUNTA_RUTA = re.compile(r"c[oó]mo (llego|vuelvo|voy|regreso)|por d[oó]nde|qu[eé] lugares .*pasar|\bruta\b", re.IGNORECASE)
MOVIMIENTO = re.compile(r"^\s*(voy|vuelvo|regreso|entro|camino|me muevo|ir a)\b", re.IGNORECASE)


def _resolver_destino(destino):
    """'abertura' → 'pasadizo_secreto' si alguna conexión lo menciona en su nombre o descripción;
    si no, se deja igual (move_to lo rechazará con los destinos válidos)."""
    d = _norm(destino)
    conexiones = world["locations"][game_state["location"]]["connections"]
    if not d or d.replace(" ", "_") in conexiones:
        return destino
    for c in conexiones:
        if d in _norm(c) or any(w in _norm(world["locations"][c]["description"]).split() for w in d.split() if len(w) >= 5):
            return c
    return destino


REFERENCIA_DESCUBIERTO = re.compile(r"lo que (haya|he|hemos|acabo de)? ?(encontrado|descubierto|hallado|visto)|"
                                    r"\b(eso|ahi|alli|ese (sitio|lugar|pasaje|pasadizo)|lo (nuevo|descubierto|encontrado))\b")


def _destino_referido(text):
    """'Entro en lo que haya encontrado' no nombra el lugar: se resuelve al lugar DINÁMICO más reciente conectado
    a la ubicación actual (hold-out 3: el jugador se quedaba en la cueva y el narrador inventaba un cofre)."""
    if re.search(r"\b(atras|de donde vine|donde estaba antes|lugar anterior)\b", _norm(text)):
        anterior = game_state.get("ubicacion_anterior")  # "Vuelvo atrás"
        return anterior if anterior in world["locations"][game_state["location"]]["connections"] else None
    if not REFERENCIA_DESCUBIERTO.search(_norm(text)):
        return None
    conexiones = world["locations"][game_state["location"]]["connections"]
    dinamicos = [c for c in conexiones if world["locations"][c].get("dynamic")]
    return max(dinamicos, key=lambda c: world["locations"][c]["created_turn"]) if dinamicos else None


ALIAS_LUGARES = {"forja": "herreria", "herrero": "herreria", "bar": "taberna", "posada": "taberna", "meson": "taberna",
                 "mercado": "plaza", "fuente": "plaza", "arboleda": "bosque", "caverna": "cueva", "gruta": "cueva"}


def _location_in(text):
    """Ubicación conocida mencionada en el texto del jugador (sin acentos), o None. Acepta sinónimos ("la forja")."""
    t = unicodedata.normalize("NFKD", text.lower()).encode("ascii", "ignore").decode()
    hits = [k for k in world["locations"] if re.search(rf"\b{re.escape(k.replace('_', ' '))}\b", t.replace("_", " "))]
    hits += [v for a, v in ALIAS_LUGARES.items() if v in world["locations"] and re.search(rf"\b{a}\b", t)]
    return max(hits, key=len) if hits else None


class AgentState(TypedDict):
    messages: Annotated[list, add_messages]   # ventana: jugadas anteriores + acción actual
    pasos: list          # [(Paso, observación)] de esta jugada
    expansions: int      # expansiones en esta jugada (máximo 1)
    prompt_tokens: int   # máximo de tokens de prompt en esta jugada
    imagen: str          # ruta de la imagen generada en esta jugada (v3)
    narracion_libre: bool  # v3: la narración la escribió el LLM (el vigilante la revisa); si la armó el código, no
    verdad: str            # v3: la verdad del juego con la que se escribió (estado, observaciones, grafo)
    correccion: str        # v3: lo que el vigilante pide corregir
    revisiones: int        # v3: reintentos ya hechos (máximo 1)


def _discoveries_this_turn():
    """Lo creado en esta jugada, leído del registro del mundo."""
    t = game_state["turn"]
    out = [f"Descubres un lugar nuevo: {k} ({v['description']}). Se llega desde: {', '.join(v['connections'])}."
           for k, v in world["locations"].items() if v.get("dynamic") and v["created_turn"] == t]
    out += [f"Aparece un personaje nuevo en {v['location']}: {k} ({v['description']})."
            for k, v in world["npcs"].items() if v.get("dynamic") and v["created_turn"] == t]
    out += [f"Hay un objeto nuevo en {v['location']}: {k} ({v['description']})."
            for k, v in world["items"].items() if v.get("dynamic") and v["created_turn"] == t]
    return out


def _accion(state):
    return next((m.content for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), "")


def _consulta_grafo(state):
    """Consulta al grafo = acción actual + la anterior. Las preguntas de seguimiento
    ("¿Por qué lugares tengo que pasar?") no nombran entidades: heredan las de la jugada previa."""
    humanas = [m.content for m in state["messages"] if isinstance(m, HumanMessage)]
    return " ".join(humanas[-2:])


def _prompt_planner(state):
    """Un solo mensaje por paso: jugadas anteriores (breves), la acción actual y los pasos ya hechos.
    Repetir la acción del jugador en cada paso evita que el modelo tome la Observación como orden nueva."""
    previas = [(m.content, n.content[:150].replace("\n", " ")) for m, n in
               zip(state["messages"][:-1:2], state["messages"][1:-1:2])]
    lineas = []
    if previas:
        lineas.append("JUGADAS ANTERIORES:\n" + "\n".join(f"- Jugador: {a} → DM: {r}..." for a, r in previas))
    lineas.append(f"ACCIÓN ACTUAL DEL JUGADOR: {_accion(state)}")
    hechos = []
    found = _discoveries_this_turn()
    if found:
        hechos.append("(mundo) DESCUBRIMIENTO: " + " ".join(found))
    hechos += [f"{p.accion}({p.argumento}) → {o}" for p, o in state.get("pasos", [])]
    lineas.append("PASOS YA HECHOS EN ESTA JUGADA:\n" + ("\n".join(f"{i+1}. {h}" for i, h in enumerate(hechos))
                                                         or "(ninguno)"))
    lineas.append("¿Siguiente paso? Si ya tienes lo necesario para responder al jugador, accion = 'responder'.")
    return HumanMessage(content="\n\n".join(lineas))


# ── Nodos del grafo ──────────────────────────────────────────────────────

def pensar_node(state: AgentState):
    """Thought + Action: el planner elige el siguiente paso (salida estructurada)."""
    if len(state.get("pasos", [])) >= MAX_PASOS:
        return {"pasos": state["pasos"] + [(Paso(intencion="accion", pensamiento="límite de pasos", accion="responder", argumento=""), "")]}
    system = (SYSTEM_REACT + f"\n\nESTADO: {_state_summary()}\nMUNDO: {_world_summary()}"
              f"\n\nMEMORIA DEL MUNDO (grafo, lo relevante a esta acción):\n{graph_context(_consulta_grafo(state))}")
    out = planner.invoke([SystemMessage(content=system), _prompt_planner(state)])
    paso = out["parsed"] or Paso(intencion="accion", pensamiento="(salida inválida)", accion="responder", argumento="")
    propuesto = (paso.accion, paso.argumento)
    accion, previos = _accion(state), state.get("pasos", [])
    es_pregunta = INTENCION["tipo"].startswith("pregunta")
    if es_pregunta and paso.accion not in ("ver_estado", "recordar", "reglas", "responder", "hablar"):
        # Enrutamiento por intención (MRKL): una PREGUNTA solo consulta, nunca ejecuta acciones de juego
        paso = Paso(intencion="pregunta", pensamiento=paso.pensamiento,
                    accion="ver_estado" if not previos else "responder", argumento="")
    no_mueve = not _intent("mover")
    requiere = {"recoger": "recoger", "atacar": "atacar", "curarse": "curarse", "comprar": "comprar", "soltar": "soltar"}
    for tipo_ in ("soltar", "atacar", "recoger", "curarse", "comprar"):
        # "Ataco al lobo" → el planner miraba el estado y el narrador inventaba la pelea; ahora la herramienta
        # responde ("No hay ningún enemigo así aquí") y la narración no puede contradecirla
        if _intent(tipo_) and not previos and paso.accion != tipo_ and re.search(VERBOS_ACCION[tipo_], _norm(accion)):
            paso = Paso(intencion="accion", pensamiento=paso.pensamiento, accion=tipo_, argumento=accion)
    if paso.accion in requiere and not _intent(requiere[paso.accion]):
        # MRKL: una acción que cambia el estado solo si es la intención del jugador (explorar no es recoger)
        paso = Paso(intencion=paso.intencion, pensamiento=paso.pensamiento,
                    accion="ver_estado" if not previos else "responder", argumento="")
    se_dirige, compra = _intent("hablar"), _intent("comprar")
    npc_nombrado = _npc_aqui(accion) if se_dirige else None
    presentes = [n for n in world["locations"][game_state["location"]]["npcs"]
                 if n not in game_state["enemies_defeated"] and not world["npcs"][n].get("hostile")]
    if se_dirige and not npc_nombrado and len(presentes) == 1 and not _npc_ausente(accion):
        npc_nombrado = presentes[0]  # se dirige a alguien sin nombrarlo y hay un solo NPC: es a él
    ult = game_state.get("ultimo_srd")
    if paso.accion == "reglas" and ult and ult["jugada"] == game_state["turn"] - 1 and re.match(r"^\W*y\b", _norm(accion)):
        # "¿Y si la uso a dos manos?" no nombra el objeto: se consulta con la pregunta anterior + la actual
        paso = Paso(intencion="pregunta", pensamiento=paso.pensamiento, accion="reglas", argumento=_consulta_grafo(state))
    if _intent("pregunta_reglas") and not previos:
        # "Según las reglas…", "¿cómo funciona…?": siempre se consulta el SRD (no es opcional), con la pregunta
        # COMPLETA del jugador (el planner resumía en "caida" y la reescritura lo convertía en Feather Fall)
        paso = Paso(intencion="pregunta", pensamiento=paso.pensamiento, accion="reglas", argumento=accion)
    if paso.accion == "reglas" and ult and ult["jugada"] == game_state["turn"] - 1 and _seguimiento_corto(accion):
        # "¿Hay un máximo?", "¿Y su aliento?": se responde con los documentos de la pregunta anterior (buscar de nuevo con
        # la pregunta combinada traía otra regla y la respuesta se contradecía: "No hay un máximo")
        paso = Paso(intencion="pregunta", pensamiento=paso.pensamiento, accion="ver_estado", argumento="")
    if paso.accion == "comprar" and not compra:
        # El planner no compra por su cuenta (regatear o preguntar el precio no es comprar)
        paso = Paso(intencion=paso.intencion, pensamiento=paso.pensamiento, accion="responder", argumento="")
    if npc_nombrado and not previos and (paso.accion != "hablar" and not (paso.accion == "reglas" and
                                                                          PREGUNTA_REGLAS.search(_norm(accion)))):
        # "Le pregunto a Goruk..." → el primer paso es hablar con Goruk (no consultar reglas ni el estado)
        paso = Paso(intencion=paso.intencion, pensamiento=paso.pensamiento, accion="hablar", argumento=npc_nombrado)
    if paso.accion == "hablar" and any(p.accion == "hablar" and _npc_aqui(p.argumento) == _npc_aqui(paso.argumento)
                                       for p, _ in previos):
        paso = Paso(intencion=paso.intencion, pensamiento=paso.pensamiento, accion="responder", argumento="")
    if paso.accion == "hablar" and npc_nombrado and _npc_aqui(paso.argumento) not in (None, npc_nombrado) \
            and _npc_aqui(accion):
        # "Bron, ¿sabes algo de los goblins?" → también contestaba Ferrik, un NPC creado que estaba en la herrería
        paso = Paso(intencion=paso.intencion, pensamiento=paso.pensamiento, accion="responder", argumento="")
    if paso.accion == "hablar" and not se_dirige:
        # El planner no inicia conversaciones por su cuenta: solo si el jugador se dirige a alguien
        paso = Paso(intencion=paso.intencion, pensamiento=paso.pensamiento, accion="responder", argumento="")
    if paso.accion == "mover" and (no_mueve or any(p.accion == "mover" for p, _ in previos)):
        # Preguntar o buscar no mueve al jugador, y se mueve una vez por jugada (si el movimiento fue
        # rechazado, no busca otro camino por su cuenta). Antes se exigía un verbo de una lista fija
        # ("voy", "vuelvo"...): sobreajustado a la secuencia de desarrollo ("Salgo hacia" fallaba).
        paso = Paso(intencion=paso.intencion, pensamiento=paso.pensamiento, accion="ver_estado" if not previos else "responder", argumento="")
    destino = _location_in(accion) or (_destino_referido(accion) if _intent("mover") else None)
    if (paso.accion in ("responder", "ver_estado") and not previos and destino and not no_mueve
            and destino != game_state["location"]):
        # El jugador nombra un lugar y pide ir: si el planner responde sin moverlo, mover (la herramienta valida)
        paso = Paso(intencion=paso.intencion, pensamiento=paso.pensamiento, accion="mover", argumento=destino)
    if paso.accion == "responder" and not previos and not _discoveries_this_turn():
        # Nunca narrar con cero observaciones: primero mirar el estado real
        paso = Paso(intencion=paso.intencion, pensamiento=paso.pensamiento, accion="ver_estado", argumento="")
    if any(p.accion == paso.accion and (paso.accion in ("ver_estado", "curarse", "recordar", "reglas") or
                                        p.argumento.strip().lower() == paso.argumento.strip().lower())
           for p, _ in previos):
        paso = Paso(intencion=paso.intencion, pensamiento=paso.pensamiento, accion="responder", argumento="")  # repetir no aporta: responder
    tokens = (out["raw"].usage_metadata or {}).get("input_tokens", 0)
    _log("pensar", intencion=paso.intencion, pensamiento=paso.pensamiento, propuesto=propuesto,
         final=(paso.accion, paso.argumento), tokens=tokens)
    return {"pasos": state.get("pasos", []) + [(paso, None)],
            "prompt_tokens": max(state.get("prompt_tokens", 0), tokens)}


def actuar_node(state: AgentState):
    """Action → Observation: el código ejecuta la herramienta elegida."""
    paso, _ = state["pasos"][-1]
    tool_fn, arg_name = ACCIONES[paso.accion]
    if paso.accion == "mover":
        # Si el jugador nombró un destino, se intenta ESE destino (el planner a veces elegía un paso intermedio
        # de la ruta); move_to valida si es posible
        nombrado = _location_in(_accion(state)) or _destino_referido(_accion(state))
        destino = nombrado if nombrado and nombrado != game_state["location"] else _resolver_destino(paso.argumento)
        paso = Paso(intencion=paso.intencion, pensamiento=paso.pensamiento, accion="mover", argumento=destino)
    args = {arg_name: paso.argumento} if arg_name else {}
    antes = game_state["location"]
    estado_antes = (game_state["hp"], game_state["gold"], tuple(game_state["inventory"]), len(game_state["events"]))
    try:
        obs = tool_fn.invoke(args)
    except Exception as e:
        obs = f"Error al usar {paso.accion}: {e}"
    estado_despues = (game_state["hp"], game_state["gold"], tuple(game_state["inventory"]), len(game_state["events"]))
    if paso.accion in ("recoger", "curarse", "atacar", "comprar", "soltar") and estado_despues == estado_antes:
        obs = f"LA ACCIÓN NO SE EJECUTÓ (el estado no cambió). {obs}"
    # Disparadores de imagen (v3): primera visita a un lugar, combate
    visitados = game_state.setdefault("visitados", {INITIAL_STATE["location"]})
    if paso.accion == "mover" and game_state["location"] != antes:
        game_state["ubicacion_anterior"] = antes  # para "vuelvo atrás"
    if paso.accion == "mover" and game_state["location"] != antes:  # una imagen por ESCENA: cada llegada a un lugar
        primera = game_state["location"] not in visitados
        visitados.add(game_state["location"])
        game_state.setdefault("_disparos", []).append(f"{'primera visita' if primera else 'escena'}: {game_state['location']}")
    if paso.accion == "atacar" and "NO SE EJECUTÓ" not in obs and "No hay enemigo" not in obs:
        game_state.setdefault("_disparos", []).append("enemigo derrotado" if "derrotado" in obs else "combate")
    if paso.accion == "mover":
        r = MoveResult(success=game_state["location"] != antes, message=obs, current_location=game_state["location"])
        obs = (f"MOVIMIENTO {'EXITOSO' if r.success else 'RECHAZADO'} → el jugador está en: {r.current_location}. "
               f"{r.message}")
    _log("actuar", herramienta=tool_fn.name, args=args, observacion=obs)
    return {"pasos": state["pasos"][:-1] + [(paso, obs)]}


def _dialogo_pedido(accion):
    """Diálogo registrado al que se refiere la pregunta: el del NPC nombrado; si no nombra a nadie, el que comparte
    el tema ("¿quién me advirtió del bosque?"); "¿qué me dijo?" sin más → el último. Si ninguno corresponde,
    None (responde la memoria; antes se citaba siempre el último diálogo, fuera cual fuera la pregunta)."""
    dialogos = game_state.get("dialogos", [])
    a = _norm(accion)
    palabras = re.findall(r"[a-z]+", a)
    nombrados = [d for d in dialogos if any(w in palabras for w in _norm(d["npc"]).split() if len(w) >= 4)]
    if nombrados:
        return nombrados[-1]
    temas = {w for w in palabras if len(w) >= 5} - {"quien", "dijeron", "menciono", "advirtio", "conto", "ofrecio",
                                                         "respondio", "contesto", "explico", "recomendo", "aconsejo"}
    con_tema = [d for d in dialogos if temas & set(re.findall(r"[a-z]{5,}", _norm(d["dijo"])))]
    if con_tema:
        return con_tema[-1]
    if dialogos and not temas and not palabras[:1] == ["quien"]:
        return dialogos[-1]
    return None


def _narrar_combate(obs):
    """Ronda de combate contada desde la observación de attack_enemy (como las compras: sin margen para inventar)."""
    enemigo = re.search(r"COMBATE vs (.+?) ---", obs).group(1)
    m = re.search(r"Tu ataque(?: con (.+?))?: d20=\d+\+\d+=(\d+) vs CA (\d+) -> (CRÍTICO|ACIERTO|FALLO)"
                  r"(?:, (\d+) de daño \(PG de .+?: (\d+)/(\d+)\))?", obs)
    arma, total, ca, res, dano, pg, pg_max = m.groups()
    t = f"Atacas a {enemigo}{f' con tu {arma}' if arma else ''} (tirada {total} contra CA {ca}): "
    t += (f"{'¡golpe crítico! ' if res == 'CRÍTICO' else '¡aciertas! '}Le haces {dano} de daño"
          f"{'' if 'derrotado' in obs else f' y le quedan {pg}/{pg_max} PG'}." if res != "FALLO" else "fallas.")
    if "ha sido derrotado" in obs:
        return t + f" ¡{enemigo} cae derrotado!"
    e = re.search(r"Ataque de .+?: d20=\d+\+\d+=(\d+) vs tu CA (\d+) -> (ACIERTO|FALLO)(?:, (\d+) de daño a ti)?", obs)
    if e:
        et, eca, eres, edano = e.groups()
        t += f" {enemigo} te ataca (tirada {et} contra tu CA {eca}): "
        t += f"te hace {edano} de daño." if eres == "ACIERTO" else "falla."
    if "Caes inconsciente" in obs:
        t += " ¡Caes inconsciente! Despiertas más tarde, maltrecho, con 1 PG."
    return t + f" Te quedan {game_state['hp']}/{game_state['max_hp']} PG."


def _repite(texto, previo, umbral=0.5):
    """¿La narración nueva repite la anterior? (fracción de sus oraciones que ya estaban, palabra por palabra)"""
    norm = lambda t: {re.sub(r"\W+", " ", _norm(o)).strip() for o in re.split(r"(?<=[.!?])\s+", t.split("[HP:")[0])
                      if len(o) > 25}
    nuevas, viejas = norm(texto), norm(previo)
    return bool(nuevas) and len(nuevas & viejas) / len(nuevas) >= umbral


def _plano(t):
    """Texto sin signos ni markdown (tablas "| Half cover |", "**", "***"): para verificar citas literales."""
    return re.sub(r"[^a-z0-9]+", " ", _norm(t)).strip()


def _ruta(destino):
    lugares = nx.Graph([(a, b) for a, d in world["locations"].items() for b in d["connections"]])
    try:
        return nx.shortest_path(lugares, game_state["location"], destino)
    except Exception:
        return None


def respuesta_memoria(accion, seguimiento=False):
    """Preguntas de memoria con respuesta exacta (rutas, "¿puedo volver?", compras, lugares descubiertos, "¿encontré
    algo?", quién dijo qué, dónde estaba un objeto): las responde el código desde el registro del mundo. El narrador
    las respondía con contradicciones ("Goruk… Bron te afiló la espada") o inventaba ("un cofre oculto").
    Devuelve None si la pregunta no es de ese tipo (resúmenes, "¿qué hice primero?" → narrador)."""
    a, loc = _norm(accion), game_state["location"]
    nom = lambda k: k.replace("_", " ")
    if re.search(r"\bpuedo (volver|ir|regresar|llegar)\b", a):
        dest = _location_in(accion)
        if not dest and re.search(r"\b(ese|esa|ahi|alli|nuevo|descubiert|encontr)", a):
            din = [k for k, v in world["locations"].items() if v.get("dynamic")]
            dest = max(din, key=lambda k: world["locations"][k]["created_turn"]) if din else None
        if dest:
            if dest == loc:
                return f"Ya estás en {nom(dest)}."
            r = _ruta(dest)
            if r and len(r) == 2:
                return f"Sí: desde {nom(loc)} puedes ir directamente a {nom(dest)}."
            return f"Sí, pero no directamente: {' → '.join(map(nom, r))}." if r else f"No conoces un camino a {nom(dest)}."
    if PREGUNTA_RUTA.search(accion) or re.search(r"lugares (cruzo|atravieso|paso|recorro)|por donde paso", a):
        dest = _location_in(accion) or game_state.get("ultima_ruta")
        if dest and dest != loc and _ruta(dest):
            game_state["ultima_ruta"] = dest
            return f"La ruta más corta de {nom(loc)} a {nom(dest)} es: {' → '.join(map(nom, _ruta(dest)))}."
    compras = [m for m in (re.match(r"Compro (.+?) a (.+?) por (\d+) de oro", e) for e in game_state["events"]) if m]
    if re.search(r"\b(compr\w*|pague|pagaste|gaste|gastaste|afil\w*|vend\w*|cobr\w*)\b", a):
        tema = [m for m in compras if any(w[:4] in a for w in _norm(m.group(1)).split() if len(w) >= 4)]
        lista = tema or ([compras[i] for i in game_state.get("_ultimas_compras", []) if i < len(compras)]
                         if seguimiento else []) or compras
        game_state["_ultimas_compras"] = [compras.index(m) for m in lista]
        if not lista:
            return "No has comprado ni pagado nada todavía."
        return " ".join(f"Le pagaste {m.group(3)} de oro a {m.group(2)} por {m.group(1).replace('afilar espada', 'afilar tu espada')}."
                        for m in lista)
    din = sorted([k for k, v in world["locations"].items() if v.get("dynamic")], key=lambda k: world["locations"][k]["created_turn"])
    if re.search(r"(sitios|lugares) (\w+ ){0,2}(nuevos|descubiert|descubri|encontr|conoc)|cuantos (sitios|lugares)", a) or (
            seguimiento and re.search(r"cuales son|cuantos son|cuantos hay", a) and game_state.get("_ultima_memoria") == "lugares"):
        game_state["_ultima_memoria"] = "lugares"
        if not din:
            return "Todavía no descubriste lugares nuevos."
        return (f"Descubriste {len(din)} lugar{'es' if len(din) > 1 else ''} nuevo{'s' if len(din) > 1 else ''}: " +
                "; ".join(f"{nom(k)} (se llega desde {nom(world['locations'][k]['connections'][0])})" for k in din) + ".")
    if re.search(r"\b(encontre|descubri|halle)\b.*\balgo\b", a):
        t = game_state["turn"] - 1
        cosas = ([f"un lugar nuevo: {nom(k)}" for k, v in world["locations"].items() if v.get("dynamic") and v["created_turn"] == t] +
                 [k for k, v in world["items"].items() if v.get("dynamic") and v.get("created_turn") == t] +
                 [f"a {k}" for k, v in world["npcs"].items() if v.get("dynamic") and v.get("created_turn") == t])
        return ("Sí: encontraste " + ", ".join(cosas) + ".") if cosas else "No, no encontraste nada nuevo."
    if re.search(VERBOS_DECIR, a):
        d = _dialogo_pedido(accion)
        if d:
            return f'{d["npc"]} te dijo: "{d["dijo"]}"'
        if re.match(r"\W*quien", a):
            return "No recuerdo que nadie te dijera eso."
    m = re.search(r"donde (estaba|estaban|encontre|halle|recogi) (.+)", a)
    if m:
        obj = next((k for k in world["items"] if any(w in re.findall(r"[a-z]+", m.group(2))
                                                     for w in _norm(k).split() if len(w) >= 4)), None)
        if obj and world["items"][obj].get("location"):
            return f"{obj[0].upper() + obj[1:]} estaba{'n' if obj.endswith('s') else ''} en {nom(world['items'][obj]['location'])}."
    return None


def _corregir_erratas(t):
    for mal, bien in {"Recoljas": "Recoges", "Recolgas": "Recoges", "Recolijes": "Recoges", "Recoljes": "Recoges",
                      "Exames": "Examinas", "Entra en la": "Entras en la"}.items():
        t = t.replace(mal, bien)
    return t


def narrar_node(state: AgentState):
    """Respuesta final en texto libre, basada en las observaciones."""
    obs = [f"- {p.accion}({p.argumento}): {o}" for p, o in state.get("pasos", []) if o]
    found = _discoveries_this_turn()
    if found:
        obs.insert(0, "- DESCUBRIMIENTO: " + " ".join(found))
    memoria = graph_context(_consulta_grafo(state))
    # Memoria del SRD: si la jugada ANTERIOR consultó reglas y esta no, el seguimiento ("¿y con qué ataca?")
    # recibe los mismos documentos oficiales en vez de completar con lo que "sabe" el modelo
    ult = game_state.get("ultimo_srd")
    if ult and ult["jugada"] == game_state["turn"] - 1 and not any(p.accion == "reglas" for p, _ in state.get("pasos", [])):
        obs += ["- REGLAS OFICIALES consultadas en la jugada anterior (SRD 5.1):\n" +
                "\n".join(f"[{d}] {srd_docs[d]['text'][:700]}" for d in ult["docs"])]
    # La ruta del grafo (shortest_path) entra como observación SOLO si el jugador pregunta cómo llegar
    # (si no, el narrador la contaba como un viaje hecho aunque move_to hubiera rechazado el movimiento)
    if _intent("pregunta_memoria"):
        obs += [f"- grafo: {l}" for l in memoria.splitlines() if l.startswith(("Ruta más corta", "No hay ruta"))]
        obs += [] if not re.search(r"\bpuedo (volver|ir|regresar)\b", _norm(_accion(state))) else [
                f"- mapa: desde {game_state['location']} se puede ir a: "
                f"{', '.join(world['locations'][game_state['location']]['connections'])}"]
    contexto = ("Observaciones de esta jugada:\n" + ("\n".join(obs) or "(ninguna)") +
                f"\n\nMEMORIA DEL MUNDO (grafo, relevante a la acción):\n{memoria}")
    dialogos = [o[len("DIÁLOGO de "):].split(": ", 1) for p, o in state.get("pasos", [])
                if p.accion == "hablar" and o and o.startswith("DIÁLOGO de ")]
    solo_dialogo = dialogos and all(p.accion in ("hablar", "ver_estado", "responder") for p, _ in state.get("pasos", []))
    srd_jugada = [o for p, o in state.get("pasos", []) if p.accion == "reglas" and o]
    if not srd_jugada and ult and ult["jugada"] == game_state["turn"] - 1:   # seguimiento de una pregunta de reglas
        # "¿Hay un máximo?" se responde con el documento que respondió la pregunta anterior (con los 3 recuperados citaba
        # "Damage Rolls" y decía "No hay un máximo")
        srd_jugada = ["\n".join(f"[{d}] {srd_docs[d]['text'][:1500]}" for d in (ult.get("citados") or ult["docs"]))]
        if _intent("pregunta_reglas"):
            ult["jugada"] = game_state["turn"]  # una cadena de seguimientos ("¿Y su aliento?" → "¿Y su mordisco?") sigue
    if _intent("pregunta_reglas") or (srd_jugada and not dialogos and
                                                          not _intent("pregunta_estado", "pregunta_memoria",
                                                                      "pregunta_dialogo", "hablar") and
                                                          _accion(state).strip().startswith("¿") and
                                                          not any(p.accion not in ("reglas", "ver_estado", "responder")
                                                                  for p, _ in state.get("pasos", []))):
        # Pregunta de REGLAS: solo el texto oficial del SRD, sin escena ni memoria (no hay de dónde inventarla)
        fuente = chr(10).join(srd_jugada)[:5000]
        accion_r = _accion(state)
        if ult and ult["jugada"] == game_state["turn"] - 1 and _seguimiento_corto(accion_r):
            accion_r = _consulta_grafo(state)  # "¿Hay un máximo?" se entiende con la pregunta anterior
        # Anclaje por cita (como el DialogueAgent): 1) copiar LITERAL las oraciones del texto oficial que responden,
        # 2) el código verifica que estén en el texto, 3) traducir solo eso. Antes el 7B traducía directo y
        # parafraseaba mal ("restando 1d6", "un mínimo de 1", "LÍMITE de 4").
        mensajes_r = [SystemMessage(content=SYSTEM_SOLO_REGLAS), HumanMessage(
            content=f"TEXTO OFICIAL:\n{fuente or '(no se encontró)'}\n\nPregunta: {accion_r}")]
        fuente_n = re.sub(r"\s+", " ", _norm(fuente))
        try:
            r = respuesta_reglas.invoke(mensajes_r)
            citas = [c for c in r.citas if c.strip() and _plano(c) in _plano(fuente)]
            if r.citas and not citas:  # citó algo que no está en el texto: un reintento
                r = respuesta_reglas.invoke(mensajes_r + [HumanMessage(content=(
                    "Tus citas no aparecen en el TEXTO OFICIAL. Copia las oraciones EXACTAS del texto."))])
                citas = [c for c in r.citas if _plano(c) in _plano(fuente)]
            texto = r.respuesta
        except Exception:
            citas, texto = [], ""
        if not citas:  # sin cita verificable en el texto oficial: abstenerse (antes inventaba: beholder, "envenenado")
            texto = "No encontré eso en las reglas oficiales que tengo (SRD 5.1)."
        if citas:  # la respuesta final se traduce SOLO de las citas verificadas
            texto = llm_reglas.invoke([SystemMessage(content=SYSTEM_TRADUCIR_REGLAS), HumanMessage(
                content=f"Pregunta: {accion_r}\nTexto oficial:\n" + "\n".join(citas))]).content
        malos = [x for x in re.findall(r"\d+d\d+", _norm(texto)) if x not in fuente_n]
        if malos:  # dados que no están en el texto oficial
            texto = llm_reglas.invoke([SystemMessage(content=SYSTEM_TRADUCIR_REGLAS), HumanMessage(
                content=f"Pregunta: {accion_r}\nTexto oficial:\n" + ("\n".join(citas) or fuente[:1500]) +
                        f"\n(Ojo: {', '.join(malos)} NO está en el texto.)")]).content
        _log("reglas", citas=citas)
        if citas and game_state.get("ultimo_srd"):  # qué documento respondió de verdad (para el seguimiento)
            citados = [d for d in game_state["ultimo_srd"].get("docs", [])
                       if any(_plano(c) in _plano(srd_docs[d]["text"][:1500]) for c in citas)]
            if citados:
                game_state["ultimo_srd"]["citados"] = citados
        cjk = r"[\u3040-\u30ff\u4e00-\u9fff\uac00-\ud7af]"
        if re.search(cjk, texto):  # el 7B a veces mezcla chino: un reintento pidiendo solo español
            texto = llm_reglas.invoke([SystemMessage(content=SYSTEM_TRADUCIR_REGLAS), HumanMessage(
                content=f"Pregunta: {accion_r}\nTexto oficial:\n" + ("\n".join(citas) or fuente[:1500]) +
                        "\nResponde SOLO en español.")]).content
        texto = "\n".join(l for l in texto.splitlines() if not re.search(cjk, l)).strip()
        ingles = len(re.findall(r"\b(the|of|to|a|an|and|or|it|its|takes|creature|maximum|damage|each|feet)\b", texto.lower()))
        if texto and ingles >= 0.25 * max(1, len(texto.split())):  # copió la cita sin traducir ("to a maximum of 20d6.")
            texto = llm_reglas.invoke([SystemMessage(content=SYSTEM_TRADUCIR_REGLAS), HumanMessage(
                content=f"Pregunta: {accion_r}\nTexto oficial:\n" + ("\n".join(citas) or texto) +
                        "\nResponde en ESPAÑOL, en una oración completa.")]).content.strip()
        if not texto:  # nunca una respuesta vacía: la cita oficial tal cual
            texto = "Según el texto oficial (SRD 5.1): " + (" ".join(citas) or "no encontré esa regla.")
        _log("narrar", observaciones=obs, memoria_grafo=memoria)
        return {"messages": [AIMessage(content=f"{texto.strip()}\n\n{_state_summary()}")]}
    d = _dialogo_pedido(_accion(state)) if _intent("pregunta_dialogo") else None
    if d:
        # "¿Qué me respondió?": la frase literal que quedó registrada (no la reconstruye el LLM)
        texto = f'{d["npc"]} te dijo: "{d["dijo"]}"'
        _log("narrar", observaciones=obs + [f"- DIÁLOGO REGISTRADO: {texto}"], memoria_grafo=memoria)
        return {"messages": [AIMessage(content=f"{texto}\n\n{_state_summary()}")]}
    if _intent("pregunta_memoria", "pregunta_dialogo") and not dialogos:
        memoria_f = respuesta_memoria(_accion(state), INTENCION.get("seguimiento", False))
        if memoria_f:
            _log("narrar", observaciones=obs + [f"- RESPUESTA DE MEMORIA (código): {memoria_f}"], memoria_grafo=memoria)
            return {"messages": [AIMessage(content=f"{memoria_f}\n\n{_state_summary()}")]}
    if _intent("pregunta_estado") and not dialogos:
        # Pregunta de ESTADO: la respuesta factual la arma el código; el narrador solo pone una frase de ambiente
        factual = respuesta_factual(_accion(state))
        ambiente = llm_narrador.invoke([SystemMessage(content=SYSTEM_AMBIENTE), HumanMessage(
            content=f"Lugar: {game_state['location']} — {world['locations'][game_state['location']]['description']}\n"
                    f"Momento del día: {game_state.get('momento', 'mañana')} (la luz debe corresponder a este momento)")]).content
        ambiente = re.split(r"(?<=[.!?])\s+", ambiente.strip())[0]
        nombres = [_norm(n).split()[0] for n in world["npcs"]] + [_norm(i).split()[0] for i in world["items"]]
        if re.search(r"\d", ambiente) or any(n in _norm(ambiente) for n in nombres if len(n) >= 4):
            ambiente = ""  # la frase de ambiente no puede introducir hechos
        texto = f"{factual} {ambiente}".strip() if factual else ambiente
        _log("narrar", observaciones=obs + [f"- RESPUESTA FACTUAL: {factual}"], memoria_grafo=memoria)
        return {"messages": [AIMessage(content=f"{texto}\n\n{_state_summary()}")]}
    ausente = (_npc_ausente(_accion(state)) if not INTENCION["tipo"].startswith("pregunta") and not _intent("mover", "explorar")
               and not dialogos else None)  # "Le robo el oro a Goruk" en la plaza: le robaba a Elena
    if ausente and not _npc_aqui(_accion(state)):
        lugar_n = world["npcs"][ausente].get("location", "otro lugar").replace("_", " ")
        texto = f"{ausente} no está aquí (lo encuentras en {lugar_n})."
        _log("narrar", observaciones=obs + [f"- NPC AUSENTE: {ausente}"], memoria_grafo=memoria)
        return {"messages": [AIMessage(content=f"{texto}\n\n{_state_summary()}")]}
    if _intent("mover") and not any(p.accion == "mover" for p, _ in state.get("pasos", [])) \
            and not _location_in(_accion(state)) and not _destino_referido(_accion(state)):
        conexiones = ", ".join(c.replace("_", " ") for c in world["locations"][game_state["location"]]["connections"])
        texto = f"No conoces ese lugar. Desde {game_state['location'].replace('_', ' ')} puedes ir a: {conexiones}."
        _log("narrar", observaciones=obs, memoria_grafo=memoria)
        return {"messages": [AIMessage(content=f"{texto}\n\n{_state_summary()}")]}
    if _intent("mover") and not any(p.accion == "mover" for p, _ in state.get("pasos", [])) \
            and _location_in(_accion(state)) == game_state["location"]:
        texto = f"Ya estás en {game_state['location'].replace('_', ' ')}."  # antes: "Llegas a la plaza" estando ahí
        _log("narrar", observaciones=obs, memoria_grafo=memoria)
        return {"messages": [AIMessage(content=f"{texto}\n\n{_state_summary()}")]}
    if solo_dialogo:
        # Jugada que es solo conversación: no hay nada que narrar → no se llama al narrador (no puede inventar)
        texto = "\n\n".join(f"Hablas con {npc}.\n{npc}: {frase}" for npc, frase in dialogos)
    else:
        mensajes = [SystemMessage(content=SYSTEM_NARRADOR)] + state["messages"] + [HumanMessage(content=contexto)]
        if state.get("correccion"):  # v3: el vigilante encontró errores en la narración anterior
            mensajes.append(HumanMessage(content=f"Tu narración anterior tenía errores: {state['correccion']}\n"
                                                 "Reescríbela corrigiéndolos, sin agregar hechos nuevos."))
        response = llm_narrador.invoke(mensajes)
        previas_ai = [m.content for m in state["messages"][:-1] if isinstance(m, AIMessage)]
        if any(_repite(response.content, p_ai) for p_ai in previas_ai):
            # Eco: copió su narración anterior (pasaba en combate y con acciones sin herramienta). Se reescribe SIN
            # las jugadas previas en el contexto, solo con las observaciones de esta jugada.
            response = llm_narrador.invoke([SystemMessage(content=SYSTEM_NARRADOR), HumanMessage(
                content=f"Acción del jugador: {_accion(state)}\n\n{contexto}\n\nNarra SOLO lo que pasó en esta jugada.")])
        if re.search(r"[\u3040-\u30ff\u4e00-\u9fff\uac00-\ud7af]", response.content):
            # Qwen a veces cambia de idioma (chino) con mucho texto en inglés en el contexto: un reintento
            response = llm_narrador.invoke(mensajes + [response, HumanMessage(content="Responde SOLO en español.")])
        texto = "\n".join(l for l in response.content.splitlines()
                           if not re.search(r"[\u3040-\u30ff\u4e00-\u9fff\uac00-\ud7af]", l))
        # La línea de estado la pone el código (el LLM la copiaba mal: ubicación, inventario, HP)
        texto = re.sub(r"\*{0,2}\[HP:[^\]]*\]\*{0,2}", "", texto).strip()
        # Frases de personajes inventadas por el narrador: se quitan (solo valen las del DialogueAgent)
        texto = re.sub(r'(?m)^[^\n:]{3,40}:\s*[«"“][^\n]*$', "", texto)
        texto = re.sub(r'[«"“][^"»”]{12,}[»"”]', "", texto)
        oraciones = re.split(r"(?<=[.!?])\s+", texto)
        texto = " ".join(o for o in oraciones
                         if not re.search(r"\b(dice|dijo|responde|respondió|comenta|añade|grita|exclama|susurra|murmura)\b", o)
                     and not o.strip().startswith(",")).strip()
        # Movimiento: la narración no puede contradecir a MoveResult (el 7B copiaba "no hay camino… sigues en"
        # del ejemplo few-shot aunque el movimiento hubiera sido exitoso)
        for p_, o_ in state.get("pasos", []):
            if p_.accion != "mover" or not o_:
                continue
            lugar = game_state["location"].replace("_", " ")
            frases = re.split(r"(?<=[.!?])\s+", texto)
            if o_.startswith("MOVIMIENTO EXITOSO"):
                frases = [f for f in frases if not re.search(r"no (hay camino|puedes (ir|llegar|moverte))|sigues en", _norm(f))]
                texto = " ".join(frases).strip()
                if _norm(lugar) not in _norm(texto[:200]):
                    texto = f"Llegas a {lugar}. {texto}".strip()
            elif o_.startswith("MOVIMIENTO RECHAZADO"):
                frases = [f for f in frases if not re.search(r"\b(llegas|te mueves|entras)\b", _norm(f))]
                texto = " ".join(frases).strip()
                if not re.search(r"no (hay camino|puedes)", _norm(texto)):
                    texto = f"No hay camino directo hacia allí; sigues en {lugar}. {texto}".strip()
        if not any(p_.accion == "mover" and o_ and o_.startswith("MOVIMIENTO EXITOSO") for p_, o_ in state.get("pasos", [])):
            # sin movimiento no se "llega" a ningún lado (lo narraba al examinar o con texto sin sentido)
            texto = " ".join(f for f in re.split(r"(?<=[.!?])\s+", texto)
                             if not re.match(r"(llegas|entras|sales|regresas|vuelves|te diriges)\b", _norm(f))).strip()
        for linea_d in _discoveries_this_turn():  # lo recién creado no puede quedar fuera de la narración
            nombre_d = re.search(r": ([^ (]+)", linea_d)
            if nombre_d and _norm(nombre_d.group(1)).replace("_", " ").split()[0] not in _norm(texto):
                texto = f"{texto} {linea_d}".strip()
        texto = "\n\n".join([t for t in [texto] if t] + [f"{npc}: {frase}" for npc, frase in dialogos])
    texto = _corregir_erratas(texto)
    fallidas = [o.split("NO SE COMPRO:", 1)[1].split("[HP")[0].strip() for p, o in state.get("pasos", [])
                if p.accion == "comprar" and o and "NO SE COMPRO" in o]
    no_ejecutadas = [re.sub(r"LA ACCIÓN NO SE EJECUTÓ \(el estado no cambió\)\.\s*", "", o).split("[HP")[0].strip()
                     for p, o in state.get("pasos", [])
                     if p.accion in ("atacar", "recoger", "curarse", "soltar") and o and "NO SE EJECUTÓ" in o
                     and not any(p2.accion == p.accion and o2 and "NO SE EJECUTÓ" not in o2
                                 for p2, o2 in state.get("pasos", []))]
    if no_ejecutadas and not fallidas:  # el narrador narraba el ataque/recogida igual: lo dice la herramienta
        texto = " ".join(dict.fromkeys(no_ejecutadas))
    if fallidas:  # el narrador a veces narraba la compra igual: el resultado lo dice la herramienta
        texto = "No se pudo comprar: " + " ".join(fallidas)
    hechas = [re.search(r"COMPRA HECHA: (.+?) a (.+?) por (\d+) de oro", o) for p, o in state.get("pasos", [])
              if p.accion == "comprar" and o and "COMPRA HECHA" in o]
    if hechas and not fallidas:  # transacción: se narra tal cual ocurrió (el narrador agregaba escenas inventadas)
        texto = " ".join(f"Le pagas {m.group(3)} monedas de oro a {m.group(2)} y obtienes: {m.group(1)}." for m in hechas if m)
    combate = [o for p, o in state.get("pasos", []) if p.accion == "atacar" and o and "--- COMBATE vs" in o]
    if combate and not fallidas and not hechas:
        texto = " ".join(_narrar_combate(o) for o in combate)
    soltados = [re.search(r"Soltaste: (.+?)\. Queda en (.+?)\.", o) for p, o in state.get("pasos", [])
                if p.accion == "soltar" and o and o.startswith("Soltaste")]
    if soltados and not combate:  # el narrador "derretía" la espada en un fuego que no existía
        texto = " ".join(f"Sueltas {m.group(1)}: queda en el suelo ({m.group(2).replace('_', ' ')})." for m in soltados if m)
    combate = combate or soltados
    libre = not solo_dialogo and not fallidas and not hechas and not no_ejecutadas and not combate
    if not texto.strip():  # los filtros pueden dejarla vacía (p.ej. si era solo un diálogo inventado)
        loc = game_state["location"]
        texto = f"Estás en {loc.replace('_', ' ')}. {world['locations'][loc]['description']}"
    _log("narrar", observaciones=obs, memoria_grafo=memoria)
    return {"messages": [AIMessage(content=f"{texto}\n\n{_state_summary()}")], "narracion_libre": libre,
            "verdad": _verdad_del_juego(obs, memoria)}


def world_builder_node(state: AgentState):
    """WorldBuilder: decide si crear UNA cosa nueva (salida estructurada) y el código la crea."""
    loc = game_state["location"]
    system = (SYSTEM_WORLD_BUILDER + f"\n\nMUNDO ACTUAL: {_world_summary()}"
              f"\nLUGARES CREADOS: {_world_memory()}"
              f"\nMEMORIA DEL MUNDO (grafo):\n{graph_context(_consulta_grafo(state))}"
              f"\nUBICACION ACTUAL: {loc} - {world['locations'][loc]['description']}")
    # v3: sin referencias previas del SRD (para "exploro la taberna" devolvía clothes-common, saddle, mirror…);
    # lo creado se ancla al SRD DESPUÉS (enemigos → bestiario, objetos → equipo/objetos mágicos, umbral de similitud)
    refs = []
    pistas = {k: p for k, p in game_state.get("especulativos", {}).items() if not p.get("confirmada")}
    if pistas:  # v3: pistas narrativas que el jugador puede estar siguiendo
        system += ("\nPISTAS (mencionadas en la historia, todavía NO existen en el mundo):\n" +
                   "\n".join(f"- {p['nombre']} ({p['mencionado_por']} {p['relacion'].replace('_', ' ')})" for p in pistas.values()) +
                   "\nSi el jugador sigue una PISTA, crea ESA entidad (mismo nombre) conectada a su ubicación actual.")
    if refs:
        system += "\nREFERENCIAS DEL SRD:\n" + "\n".join(f"[{d['id']}] {d['text'][:250]}" for d in refs)
    try:
        c = world_builder_llm.invoke([SystemMessage(content=system), HumanMessage(content=_accion(state))])
    except Exception:
        c = None
    creado, ref = None, None
    if c and c.tipo != "nada" and c.nombre.strip():
        if c.tipo == "ubicacion":
            creado = create_location.invoke({"name": c.nombre, "description": c.descripcion, "connected_to": loc})
        elif c.tipo == "npc":
            creado = create_npc.invoke({"name": c.nombre, "description": c.descripcion, "location": loc, "hostile": c.hostil})
            ref = anclar_creacion("npc", c.nombre, c.descripcion, c.hostil, c.referencia_srd)
            _aplicar_srd_npc(c.nombre, ref)
        else:
            creado = create_item.invoke({"name": c.nombre, "description": c.descripcion, "location": loc})
            ref = anclar_creacion("objeto", c.nombre, c.descripcion, False, c.referencia_srd)
            if ref and c.nombre in world["items"]:
                world["items"][c.nombre].update({"srd_ref": ref})
                world["items"][c.nombre]["description"] += f" [SRD: {srd_docs[ref]['name']}]"
    if _discoveries_this_turn():
        game_state.setdefault("_disparos", []).append("creación del WorldBuilder")
    if creado and c:  # v3: si lo creado corresponde a una pista, la pista queda CONFIRMADA
        for clave, p in game_state.get("especulativos", {}).items():
            # se confirma si lo creado contiene el SUSTANTIVO PRINCIPAL de la pista ("bruja" en "bruja del pantano"):
            # crear el pantano no confirma a la bruja
            nucleo = next((w for w in _norm(p["nombre"]).split() if len(w) >= 4), "")
            creado_txt = _norm(f"{c.nombre} {c.descripcion}").replace("_", " ").split()
            if not p.get("confirmada") and nucleo and nucleo in creado_txt:  # "Gorlina la Sabia, una bruja…" confirma
                p["confirmada"] = c.nombre
                game_state["events"].append(f"Pista confirmada: {p['nombre']} → {c.nombre}")
    _log("world_builder", refs_srd=[d["id"] for d in refs], decision=c.model_dump() if c else None,
         resultado=creado, anclado_a=ref)
    return {"expansions": 1}


PERSONAJES_PARTIDA = os.environ.get("PERSONAJES_PARTIDA", "")  # v3: carpeta donde los NPCs creados escriben su ficha


def escribir_ficha(nombre):
    """El NPC creado por el WorldBuilder queda como ficha .md legible (mismo formato que las de autor)."""
    if not PERSONAJES_PARTIDA or nombre not in world["npcs"]:
        return None
    d = world["npcs"][nombre]
    meta = {"nombre": nombre, "ubicacion": d.get("location"), "hostil": bool(d.get("hostile")),
            "creado_en_jugada": d.get("created_turn"), "srd_ref": d.get("srd_ref"),
            **{k: d[k] for k in ("hp", "attack", "defense") if k in d}}
    os.makedirs(PERSONAJES_PARTIDA, exist_ok=True)
    ruta = f"{PERSONAJES_PARTIDA}/{_norm(nombre).replace(' ', '_')}.md"
    with open(ruta, "w", encoding="utf-8") as f:
        f.write("---\n" + yaml.safe_dump(meta, allow_unicode=True, sort_keys=False) + "---\n"
                "<!-- Generado por el WorldBuilder durante la partida -->\n\n"
                f"## Personalidad\n{d.get('description', '')}\n")
    return ruta


def _aplicar_srd_npc(nombre, ref):
    """Copia CA, PG y daño oficiales del SRD a un NPC creado (en vez de inventarlos)."""
    doc = srd_docs.get(ref)
    if not doc or doc["type"] != "monster" or nombre not in world["npcs"]:
        return
    npc = world["npcs"][nombre]
    npc.update({"srd_ref": ref, **stats_desde_srd(ref)})
    npc["description"] += f" [SRD: {doc['name']} — CA {npc['defense']}, PG {npc['hp']}, daño {npc.get('attack', '?')}]"


def _undo_creations():
    """Deshace lo creado en esta jugada (si Coherence lo rechaza)."""
    t = game_state["turn"]
    for k in [k for k, v in world["locations"].items() if v.get("dynamic") and v["created_turn"] == t]:
        for other in world["locations"][k]["connections"]:
            world["locations"][other]["connections"].remove(k)
        del world["locations"][k]
        game_state["locations_created"].remove(k)
    for k in [k for k, v in world["npcs"].items() if v.get("dynamic") and v["created_turn"] == t]:
        world["locations"][world["npcs"][k]["location"]]["npcs"].remove(k)
        del world["npcs"][k]
        game_state["npcs_created"].remove(k)
    for k in [k for k, v in world["items"].items() if v.get("dynamic") and v["created_turn"] == t]:
        world["locations"][world["items"][k]["location"]]["items"].remove(k)
        del world["items"][k]
        game_state["items_created"].remove(k)


def coherence_node(state: AgentState):
    """CoherenceChecker: valida lo creado; si es incoherente, se deshace."""
    found = _discoveries_this_turn()
    if not found:
        return {}
    # v3: el mundo SIN lo recién creado (si no, el checker ve la creación en el mundo y la marca como duplicado)
    t = game_state["turn"]
    nuevos = {k for k, v in list(world["locations"].items()) + list(world["npcs"].items()) + list(world["items"].items())
              if v.get("dynamic") and v.get("created_turn") == t}
    mundo = json.dumps({k: {"desc": v["description"][:60], "conn": [x for x in v["connections"] if x not in nuevos]}
                        for k, v in world["locations"].items() if k not in nuevos}, ensure_ascii=False)
    memoria = "\n".join(l for l in graph_context(game_state["location"]).splitlines()
                         if not any(_norm(n) in _norm(l) for n in nuevos))
    try:
        v = coherence_llm.invoke([SystemMessage(content=SYSTEM_COHERENCE), HumanMessage(
            content=f"Mundo existente: {mundo}\nMemoria (grafo):\n{memoria}"
                    f"\n\nCreación nueva (todavía NO está en el mundo): {' '.join(found)}")])
    except Exception:
        v = None
    _log("coherence", creacion=found, veredicto=v.model_dump() if v else None,
         deshecho=bool(v and not v.coherente))
    if v and not v.coherente:
        _undo_creations()
        game_state["events"].append(f"Coherence rechazó una creación: {v.problema[:80]}")
        game_state["_disparos"] = [d for d in game_state.get("_disparos", []) if d != "creación del WorldBuilder"]
        return {}
    # v3: la ficha .md del NPC creado se escribe solo si el CoherenceChecker aprobó la creación
    for nombre, d in world["npcs"].items():
        if d.get("dynamic") and d.get("created_turn") == game_state["turn"]:
            escribir_ficha(nombre)
    return {}


# ── Respuestas factuales (v3): lo que es estado del juego no pasa por el LLM ──
PREGUNTA_ESTADO = re.compile(
    r"inventario|llevo|tengo|queda|quedan|equipo|objetos|\boro\b|monedas|dinero|vida|\bhp\b|salud|heridas|"
    r"donde (estoy|me encuentro)|en que lugar|ubicacion|sigo en|que hay aqui|quien (esta|hay) aqui|que veo", re.I)
PREGUNTA_REGLAS = re.compile(
    r"segun las reglas|como funciona|cuanto dano|que dano|estadisticas|puntos de golpe|clase de armadura|"
    r"es venenos|es inmune|que hace (exactamente )?(un|una|el|la)\b", re.I)


def _es_pregunta_estado(accion):
    a = _norm(accion)
    return (accion.strip().startswith("¿") or a.startswith(("reviso", "miro mi", "cuanto", "que tengo", "donde")) or
            "?" in accion) and bool(PREGUNTA_ESTADO.search(a)) and not PREGUNTA_REGLAS.search(a)


def respuesta_factual(accion):
    """Respuesta armada por el CÓDIGO desde el estado real (el narrador no puede cambiarla)."""
    a, loc = _norm(accion), game_state["location"]
    inv = game_state["inventory"]
    partes = []
    # ¿Tengo X? → sí/no explícito para cada objeto mencionado
    for objeto in set(world["items"]) | set(inv) | {"pocion de curacion"}:
        clave = _norm(objeto).split()[0]
        if len(clave) >= 4 and clave[:5] in a:
            partes.append(f"Sí, llevas {objeto}." if objeto in inv else f"No, no llevas {objeto}.")
    if re.search(r"inventario|llevo|equipo|objetos|que tengo|mochila", a) and not partes:
        partes.append(f"Llevas: {', '.join(inv) if inv else 'nada'}.")
    if re.search(r"\boro\b|monedas|dinero", a):
        partes.append(f"Tienes {game_state['gold']} monedas de oro.")
    if re.search(r"\bhora\b|de noche|de dia|momento|anochece|amanece|\bes tarde\b|que tiempo", a):
        luz = {"mañana": "la luz de la mañana entra clara", "mediodía": "el sol está en lo alto",
               "tarde": "la luz de la tarde se alarga", "atardecer": "el sol se está poniendo",
               "noche": "ya es de noche y todo está oscuro"}[game_state.get("momento", "mañana")]
        partes.append(f"Es {game_state.get('momento', 'mañana')}: {luz}.")
    if re.search(r"vida|\bhp\b|salud|heridas", a):
        partes.append(f"Tienes {game_state['hp']}/{game_state['max_hp']} puntos de vida.")
    if re.search(r"donde (estoy|me encuentro)|en que lugar|ubicacion|sigo en", a):
        partes.append(f"Estás en {loc.replace('_', ' ')}. Desde aquí puedes ir a: "
                      f"{', '.join(c.replace('_', ' ') for c in world['locations'][loc]['connections'])}.")
    if re.search(r"que hay aqui|quien (esta|hay) aqui|que veo|objetos", a):
        objs = [i for i in world["locations"][loc]["items"] if i not in game_state["items_taken"]]
        npcs = [n for n in world["locations"][loc]["npcs"] if n not in game_state["enemies_defeated"]]
        partes.append(f"Aquí hay: {', '.join(objs) or 'ningún objeto'}. Personajes: {', '.join(npcs) or 'nadie'}.")
    if not partes:  # la intención es pregunta de estado pero ninguna pieza calzó: resumen completo
        objs = [i for i in world["locations"][loc]["items"] if i not in game_state["items_taken"]]
        npcs = [n for n in world["locations"][loc]["npcs"] if n not in game_state["enemies_defeated"]]
        partes.append(f"Estás en {loc.replace('_', ' ')}. Aquí hay: {', '.join(objs) or 'ningún objeto'}. "
                      f"Personajes: {', '.join(npcs) or 'nadie'}. Llevas: {', '.join(inv) or 'nada'}.")
    return " ".join(partes)


SYSTEM_AMBIENTE = """\
Eres un Dungeon Master. Escribe UNA sola oración breve de ambiente (sonidos, luz, olor) sobre el lugar
indicado. NO menciones objetos, personajes, números ni acciones del jugador. En español."""

SYSTEM_SOLO_REGLAS = """\
Respondes preguntas de reglas de D&D 5e usando SOLO el TEXTO OFICIAL que te dan (en inglés).
citas: copia LITERALMENTE, en inglés, la(s) oración(es) del texto que responden la pregunta (para una criatura:
la línea con CA y PG y la de sus ataques). Si el texto trae varias reglas o criaturas, cita SOLO la que pregunta.
Un hechizo u objeto mágico solo si la pregunta lo nombra (para "¿cuánto daño hace una caída?" se cita la regla de
caídas, no Feather Fall).
respuesta: la respuesta en español (1-3 oraciones) con las cifras exactas. No describas ningún lugar ni escena."""

SYSTEM_TRADUCIR_REGLAS = """\
Responde la pregunta en español (1-3 oraciones) traduciendo FIELMENTE el texto oficial de D&D 5e que te dan.
Copia las cifras y dados tal cual (1d6, 20d6, CA 11, PG 34). No agregues nada que no esté en el texto.
Términos: "PG 34 (4d10)" = 34 puntos de golpe (4d10), "CA 11" = clase de armadura 11, bludgeoning damage = daño
contundente, slashing = cortante, piercing = perforante, saving throw = tirada de salvación, VD (challenge) = valor de
desafío, Hit Dice = dados de golpe, short/long rest = descanso corto/largo."""


class RespuestaReglas(BaseModel):
    citas: list[str] = Field(description="Oraciones copiadas literalmente del TEXTO OFICIAL que responden la pregunta")
    respuesta: str = Field(description="Respuesta en español con las cifras exactas")


respuesta_reglas = llm_reglas.with_structured_output(RespuestaReglas, method="json_schema")


# ── Clasificador de intención (v3, patrón MRKL) ───────────────────────────
# Antes el ruteo dependía de ~40 regex en español (verbos de movimiento, compra, diálogo, preguntas de estado o
# reglas): frágiles ante frases nuevas (el hold-out 1 lo mostró con "Salgo hacia"). Ahora un clasificador con
# salida estructurada decide UNA intención por jugada y todo el ruteo la usa.
# Medición (45 frases coloquiales etiquetadas, ninguna del hold-out 2): regex 80% · clasificador 7B 93%, 0.12 s.

class IntencionJugador(BaseModel):
    tipo: Literal["mover", "explorar", "hablar", "comprar", "atacar", "recoger", "soltar", "curarse",
                  "pregunta_estado", "pregunta_reglas", "pregunta_memoria", "pregunta_dialogo", "otra"]


SYSTEM_INTENCION = """\
Clasifica lo que escribe el jugador de un juego de rol en UNA intención:
- mover: quiere ir a otro lugar ("me voy a…", "vuelvo al…", "entro en…")
- explorar: busca/examina/inspecciona algo en el lugar (algo escondido, un rastro, un objeto)
- hablar: se dirige a un personaje: le pregunta, le cuenta, lo saluda, le pide su opinión ("¿conoces…?", "¿qué opinas…?")
- comprar: quiere comprar o PAGAR por algo que se vende (pociones, afilar la espada). PEDIR algo a alguien
  (agua, un favor, información) NO es comprar: es hablar
- atacar: pelea, golpea, ataca a alguien · recoger: toma/agarra/guarda un objeto · curarse: bebe/usa una poción, se cura
- soltar: tira, deja, lanza, arroja o se deshace de un objeto que LLEVA ("lanzo la espada al fuego", "dejo la antorcha")
- pregunta_estado: pregunta por SU estado actual: vida, oro, inventario, dónde está, quién/qué hay aquí, qué hora es
- pregunta_reglas: pregunta por las reglas de D&D: estadísticas de criaturas, daño de armas, cómo funciona una regla,
  qué efecto tiene una condición o un descanso ("¿qué pasa si estoy envenenado?", "¿cuánto cura un descanso corto?")
- pregunta_memoria: pregunta por el PASADO o el mapa: qué lugares descubrió, resumen o "cuéntame la aventura",
  quién le HIZO algo antes ("¿quién me afiló la espada?"), cómo llegar a un lugar, si puede volver a un sitio
- pregunta_dialogo: pregunta qué le DIJO un personaje ANTES (pasado: "¿qué me dijo…?", "¿qué me contó…?").
  Si le pregunta AHORA a un personaje ("¿qué me cuenta Goruk…?", "¿qué aventuras me menciona…?") es hablar
- otra: nada de lo anterior
Ejemplos: "me voy pa la herrería" → mover · "miro debajo de la mesa" → explorar · "oye tabernero, ¿hay trabajo?" → hablar ·
"¿cuánto oro llevo?" → pregunta_estado · "¿qué daño hace una daga?" → pregunta_reglas · "¿cómo llego a la plaza?" → pregunta_memoria ·
"le arreo al lobo" → atacar · "¿qué me contó Bron?" → pregunta_dialogo ·
"busco a alguien que no conozca" → explorar (buscar gente nueva es explorar, no hablar con alguien presente) ·
"le pido un vaso de agua al tabernero" → hablar · "¿qué historias me cuenta Bron?" → hablar ·
"¿quién hay por aquí?" → pregunta_estado · "le pregunto al tabernero qué sabe" → hablar ·
"¿quién me afiló la espada?" → pregunta_memoria · "cuéntame la aventura completa" → pregunta_memoria ·
"¿puedo volver a ese sitio?" → pregunta_memoria (preguntar si puede ir no es ir) ·
"¿cuánto cuesta la poción?" → hablar (preguntar el precio no es comprar) · "tiro la antorcha" → soltar"""

clasificador_intencion = ChatOllama(model=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL, temperature=0).with_structured_output(
    IntencionJugador, method="json_schema")
INTENCION = {"tipo": "otra", "seguimiento": False}


# Detector de PREGUNTAS DE SEGUIMIENTO ("¿Hay un máximo?" continúa la anterior; "¿Qué pasa si estoy envenenado?" no).
# Contar palabras fallaba y el 7B acertaba 8-10 de 20 casos; el modelo grande (el del vigilante) 19/20, ~0.5 s.
# Solo se llama cuando la jugada anterior fue una pregunta y esta es una pregunta corta.
class Seguimiento(BaseModel):
    tema_propio: str = Field(description="lo que la pregunta nueva NOMBRA ella misma, o 'ninguno'")
    continua: Literal["si", "no"]


SYSTEM_SEGUIMIENTO = """\
Decides si la PREGUNTA NUEVA de un jugador continúa la PREGUNTA ANTERIOR (no se entiende sola) o trae un tema propio.
tema_propio: la criatura, objeto, regla o cosa que la pregunta nueva NOMBRA ella misma; "ninguno" si no nombra nada
(pronombres como "su", "sus", "lo", "eso" o palabras sueltas como "máximo", "peso", "daño" NO son un tema propio).
continua: "si" cuando tema_propio es "ninguno"; "no" cuando la pregunta nueva nombra su propio tema.
Ejemplos:
- anterior "¿cuánto daño hace una caída?" · nueva "¿Hay un máximo?" → tema_propio "ninguno", continua "si"
- anterior "¿qué es un dragón rojo?" · nueva "¿Y su aliento?" → "ninguno", "si"
- anterior "¿qué hace la bolsa de contención?" · nueva "¿Cuánto peso aguanta?" → "ninguno", "si"
- anterior "¿qué daño hace una espada larga?" · nueva "¿Qué pasa si estoy envenenado?" → "envenenado", "no"
- anterior "¿qué lugares descubrí?" · nueva "¿Cuántos son?" → "ninguno", "si"
- anterior "¿qué es un beholder?" · nueva "¿Qué hace la ventaja?" → "ventaja", "no"
- anterior "¿quién me afiló la espada?" · nueva "¿Cuánto le pagué?" → "ninguno", "si"
- anterior "¿qué es un orco?" · nueva "¿Dónde estoy?" → "ubicación del jugador", "no"."""

detector_seguimiento = ChatOllama(model=WATCHER_MODEL, base_url=OLLAMA_BASE_URL, temperature=0).with_structured_output(
    Seguimiento, method="json_schema")  # intención de la jugada en curso (la fija el nodo clasificar)


def _intent(*tipos):
    return INTENCION["tipo"] in tipos


REGATEO = (r"cuanto (cuesta|vale|cobras|sale|pides)|\bprecio\b|me (la|lo) (dejas|das) (en|por)|rebaja|descuento|"
           r"mas barat|regate")
MIRAR_ALREDEDOR = re.compile(r"^(miro|echo (un vistazo|una mirada|una ojeada)|observo|contemplo)( (a mi )?alrededor| el lugar|"
                             r" por (la|el) \w+| (la|el) \w+| a mi alrededor en (la|el) \w+)?$")


VERBOS_ACCION = {"atacar": r"\b(atac|ataco|golpe|luch|pele|embist|apunal|arreo|pego|me lanzo contra)",
                 "recoger": r"\b(recoj|recog|cojo|coger|tomo|agarr|guardo|levant|me llevo)",
                 "curarse": r"\b(bebo|tomo|uso|me curo|curarme)", "soltar": r"\b(tiro|lanzo|dejo|suelto|arrojo|me deshago)",
                 "comprar": r"\b(compr|pago|pagar|le pago|vendeme|me llevo)"}
REVISAR_INVENTARIO = re.compile(r"^(revis\w*|miro|chequeo) (todo )?(mi|mis|lo que llevo|el equipo|el inventario)\b|^(que|cuanto) llevo")


def _comerciante_aqui():
    loc = game_state["location"]
    return next((n for n in world["locations"][loc]["npcs"] if world["npcs"].get(n, {}).get("vende")
                 and n not in game_state["enemies_defeated"]), None)


def _seguimiento_corto(accion):
    """Pregunta de seguimiento que no se entiende sola: "¿Hay un máximo?", "¿Cuáles son?", "¿Y si…?"."""
    a = accion.strip()
    return a.startswith("¿") and INTENCION.get("seguimiento", False)


VERBOS_DECIR = r"\b(dijo|dijeron|conto|contaron|respondio|contesto|menciono|explico|advirtio|ofrecio|recomendo|aconsejo|pidio|hablo)\b"


def clasificar_node(state: AgentState):
    try:
        humanas = [m.content for m in state["messages"] if isinstance(m, HumanMessage)]
        INTENCION["tipo"] = clasificador_intencion.invoke([SystemMessage(content=SYSTEM_INTENCION), HumanMessage(
            content=f"Jugador (está en {game_state['location']}): {_accion(state)}")]).tipo
        INTENCION["seguimiento"] = False
        previa_txt = humanas[-2] if len(humanas) > 1 else ""
        if (previa_txt and game_state.get("_intencion_previa", "").startswith("pregunta")
                and _accion(state).strip().startswith("¿") and len(re.findall(r"\w+", _accion(state))) <= 8):
            INTENCION["seguimiento"] = detector_seguimiento.invoke([SystemMessage(content=SYSTEM_SEGUIMIENTO), HumanMessage(
                content=f"Pregunta anterior: {previa_txt}\nPregunta nueva: {_accion(state)}")]).continua == "si"
    except Exception:
        INTENCION["tipo"], INTENCION["seguimiento"] = "otra", False
    accion = _accion(state)
    if _intent("pregunta_dialogo") and not re.search(VERBOS_DECIR, _norm(accion)):
        # "¿qué aventuras me menciona Goruk?" (presente, Goruk está aquí) → conversar;
        # "¿quién me afiló la espada?" / "cuéntame la aventura" no preguntan qué DIJO alguien → memoria
        INTENCION["tipo"] = "hablar" if _npc_aqui(accion) else "pregunta_memoria"
    if _intent("mover") and accion.strip().startswith("¿"):
        INTENCION["tipo"] = "pregunta_memoria"  # "¿Puedo volver a ese sitio?" pregunta, no mueve (hold-out 3)
    if MOVIMIENTO.search(_norm(accion)) and _destino_referido(accion):
        INTENCION["tipo"] = "mover"  # "Entro en lo que haya encontrado" (el clasificador dudaba con explorar)
    if re.match(r"^(le )?(pregunto|digo|cuento|pido|saludo)\b", _norm(accion)) and _npc_aqui(accion):
        INTENCION["tipo"] = "hablar"  # "le pregunto al tabernero qué sabe del bosque" se dirige a alguien presente
    if _intent("comprar") and re.search(REGATEO, _norm(accion)):
        # "¿Cuánto cuesta?", "¿Me la dejas en 5?": se conversa con el comerciante; comprar solo si lo pide
        INTENCION["tipo"] = "hablar" if _npc_aqui(accion) or _comerciante_aqui() else "pregunta_estado"
    if _intent("mover") and re.match(r"(busco|buscar|examino|investigo|exploro)\b", _norm(accion).strip(" ¿?.")) \
            and not _location_in(accion):
        INTENCION["tipo"] = "explorar"  # "Busco una puerta trasera" → puede descubrir algo (antes: "No conoces ese lugar")
    if REVISAR_INVENTARIO.search(_norm(accion).strip(" ¿?.")):
        INTENCION["tipo"] = "pregunta_estado"  # "Reviso lo que llevo / todo mi equipo" (salía recoger o explorar)
    if _intent("explorar") and accion.strip().startswith("¿"):
        INTENCION["tipo"] = "pregunta_memoria"  # "¿Encontré algo?" pregunta por lo que pasó: no crea contenido
    if _intent("explorar") and MIRAR_ALREDEDOR.match(_norm(accion).strip(" .!")):
        INTENCION["tipo"] = "otra"  # "Miro a mi alrededor" describe el lugar; buscar/examinar sí puede descubrir
    previa = game_state.get("_intencion_previa")
    if _seguimiento_corto(accion) and previa in ("pregunta_memoria", "pregunta_reglas"):
        # "¿Cuánto le pagué?", "¿Cuáles son?" siguen a la pregunta de memoria; "¿Hay un máximo?", "¿Y si es de tres
        # cuartos?" siguen a la de reglas (antes respondían con el estado o buscaban "venenos" en el SRD)
        INTENCION["tipo"] = previa
    game_state["_intencion_previa"] = INTENCION["tipo"]
    _log("clasificar", tipo=INTENCION["tipo"])
    return {}


# ── DialogueAgent (v3) ───────────────────────────────────────────────────
# Antes, talk_to devolvía SIEMPRE la misma frase fija del NPC, ignorando lo que decía el jugador, y el
# narrador la parafraseaba o la omitía. Ahora un agente habla COMO el NPC, con su perfil, lo que el
# grafo dice de él (local search), su historial con el jugador y, si aplica, el SRD; el narrador cita
# su respuesta textual y queda en el grafo como lo que realmente dijo.

SYSTEM_DIALOGO = """\
Eres {nombre}. {personalidad}
Estás en: {lugar}.

LO QUE SABES (lo relevante de tu ficha para esta conversación):
{conocimientos}
{srd}
EL VIAJERO CONOCE A: {conoce}
EL VIAJERO NO CONOCE A: {no_conoce} — si mencionas a alguien de esta lista, PRESÉNTALO: quién es, dónde
está y de qué lo conoces tú (ej. "hay una comerciante en la plaza, una tal …").

LO QUE YA LE DIJISTE A ESTE VIAJERO:
{historial}

REGLAS:
- Respondes en primera persona, en español, como tu personaje, a lo que el viajero te acaba de decir.
- Responde a TODO lo que el viajero te dice AHORA (si pide algo que sirves o vendes, atiéndelo).
- Comparte como MÁXIMO 1 o 2 cosas, las más relevantes para lo que te pregunta. No recites todo lo que sabes.
- Solo hablas de personas que conoces (aparecen en LO QUE SABES como "Conoces a…").
- Los rumores dilos como rumores ("dicen que…"), no como hechos seguros.
  No repitas frases que ya le dijiste.
- No inventes cómo están las cosas (agotado, de buen humor, cerrado, peligroso hoy…): solo LO QUE SABES.
- Usa SOLO lo que sabes. Si te preguntan algo que no está en lo que sabes, admite con tus propias
  palabras y en personaje que no lo sabes — NO inventes lugares, criaturas, precios ni consejos.
- No te contradigas con lo que ya dijiste.
- 1 a 3 oraciones. Solo tu diálogo: no narres acciones del viajero ni escribas acotaciones."""

class RespuestaNPC(BaseModel):
    """Anclaje por cita: el NPC debe copiar el hecho de LO QUE SABES en que se basa."""
    hecho_usado: str = Field(description="Copia LITERAL la línea de LO QUE SABES en que basas tu respuesta, "
                                         "o '' si ninguna responde a lo que te dicen.")
    respuesta: str = Field(description="Tu diálogo, 1-3 oraciones, en español.")


llm_dialogo = ChatOllama(model=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL, temperature=0.2).with_structured_output(
    RespuestaNPC, method="json_schema")


def _npc_ausente(texto):
    """NPC conocido que el jugador nombra pero que NO está en este lugar ("le pregunto a Goruk…" en la plaza)."""
    loc, t = game_state["location"], set(re.findall(r"[a-z]+", _norm(texto)))
    return next((n for n, d in world["npcs"].items() if d.get("location") != loc and n not in game_state["enemies_defeated"]
                 and any(w in t for w in _norm(n).split()[:1] if len(w) >= 4)), None)


def _npc_aqui(nombre):
    """NPC presente en la ubicación actual que corresponde al nombre que dio el planner."""
    loc = game_state["location"]
    presentes = [n for n in world["locations"][loc]["npcs"] if n not in game_state["enemies_defeated"]]
    n = " ".join(re.findall(r"[a-z]+", _norm(nombre)))  # sin puntuación ("goruk?" → "goruk")
    return next((p for p in presentes if n and (n in _norm(p) or _norm(p) in n
                 or any(w in _norm(p).split() for w in n.split() if len(w) >= 4))), None)


VACIAS = set("""pregunto preguntar vuelvo digo pido hablo charlo trato cuento quiero saber sabes sabe puedes
puede donde como cuando quien cual cuanto sobre tiene tienes hay algo alguna alguno favor viajero tambien
porque entonces ahora aqui alli esto esta ese esos estas estos sobre desde hasta entre para
conoce conoces algun tomo dame dime solo pero bien mucho poco muy""".split())
GENERICAS = {"informacion", "importante", "consejo", "consejos", "ayuda", "noticias", "rumores", "rumor", "novedades",
             "hola", "buenas", "buenos", "saludos", "gracias", "adios", "estas", "tal",
             "lugar", "sitio", "zona", "pueblo", "relevante", "interesante", "sabes", "cuentame", "contarme",
             "oido", "historias", "chismes", "algo", "aventuras", "aventura", "misiones", "trabajo", "peligros"}


def _sabe_del_tema(mensaje, npc, conocimientos):
    """Chequeo léxico (como un BM25 mínimo): ¿alguna palabra clave del mensaje aparece en lo que sabe el NPC?
    Si el mensaje no tiene palabras clave concretas (saludo, pedido genérico) se deja hablar."""
    texto = _norm(" ".join(conocimientos))
    nombres = set(_norm(npc).split())
    claves = [w for w in re.findall(r"[a-z]{4,}", _norm(mensaje)) if w not in VACIAS and w not in nombres]
    concretas = [w for w in claves if w not in GENERICAS]
    if not concretas or any(w in GENERICAS for w in claves):
        return True  # pedido abierto ("¿rumores raros?", "¿alguna noticia?"): comparte lo que sabe
    return any(w[:5] in texto for w in concretas)


_emb_fichas = {}  # texto de fragmento → embedding normalizado (se calcula una vez)


def recuperar_de_ficha(npc, mensaje, conocimientos, k=4):
    """RAG sobre la ficha del NPC: los k fragmentos más relevantes para lo que dice el jugador (no la ficha
    entera: así no recita todo lo que sabe). Las relaciones con personas nombradas en el mensaje siempre entran."""
    if len(conocimientos) <= k:
        return conocimientos
    nuevos = [t for t in conocimientos if t not in _emb_fichas]
    if nuevos:
        for t, e in zip(nuevos, embedder.embed_documents(nuevos)):
            _emb_fichas[t] = np.array(e) / np.linalg.norm(e)
    consulta = re.sub(rf"\b({'|'.join(map(re.escape, npc.lower().split()))})\b", "", mensaje.lower())
    q = np.array(embedder.embed_query(consulta or mensaje))
    q = q / np.linalg.norm(q)
    ranking = sorted(conocimientos, key=lambda t: -float(_emb_fichas[t] @ q))
    nombrados = [t for t in conocimientos if t.startswith("Conoces a ") and
                 any(w in _norm(mensaje) for w in _norm(t[10:].split("(")[0]).split() if len(w) >= 4)]
    return list(dict.fromkeys(nombrados + ranking[:k]))


def dialogo_node(state: AgentState):
    """Acción 'hablar': el NPC responde a lo que dijo el jugador (o fallback a talk_to si es hostil/falla)."""
    paso, _ = state["pasos"][-1]
    npc = _npc_aqui(paso.argumento)
    if not npc:
        obs = talk_to.invoke({"npc_name": paso.argumento})  # "No hay nadie llamado X aquí..."
        _log("dialogo", npc=paso.argumento, respuesta=None, observacion=obs)
        return {"pasos": state["pasos"][:-1] + [(paso, obs)]}
    datos = world["npcs"][npc]
    if datos.get("hostile"):  # el goblin no conversa
        obs = talk_to.invoke({"npc_name": npc}) + " (Es hostil: no conversa.)"
        _log("dialogo", npc=npc, respuesta=None, observacion=obs)
        return {"pasos": state["pasos"][:-1] + [(paso, obs)]}

    sync_graph()
    vecindario = [(u, d["rel"], v) for u, v, d in list(G.out_edges(npc, data=True)) + list(G.in_edges(npc, data=True))
                  if d["rel"] not in ("hablo_con", "dijo_a_jugador")]
    conocimientos = conocimientos_npc(datos)
    conocimientos = conocimientos + [f"{u} {r.replace('_', ' ')} {v}" for u, r, v in vecindario if r != "conoce"]
    conocidos_jugador = sorted({d["npc"] for d in game_state.get("dialogos", [])} | {npc})
    no_conocidos = [r["con"] for r in datos.get("relaciones", []) if r["con"] not in conocidos_jugador]
    srd = ""
    if datos.get("temas_srd"):
        docs = buscar_srd(reescribir_consulta(_accion(state)), k=2, tipos=datos["temas_srd"])
        srd = "Sobre tu oficio (reglas oficiales):\n" + "\n".join(f"- {d['text'][:300]}" for d in docs) + "\n"
    # Historial: solo lo que dijo sabiendo (las abstenciones no son información) y lo más reciente;
    # el 7B tiende a COPIAR frases del historial, así que se limita
    historial = [d["dijo"] for d in game_state.get("dialogos", []) if d["npc"] == npc and d.get("sabe", True)][-2:]
    system = SYSTEM_DIALOGO.format(
        nombre=npc, personalidad=datos.get("personalidad", datos.get("description", "")),
        lugar=game_state["location"], srd=srd,
        conocimientos="\n".join(f"- {c}" for c in recuperar_de_ficha(npc, _accion(state), conocimientos)),
        conoce=", ".join(conocidos_jugador), no_conoce=", ".join(no_conocidos) or "(nadie relevante)",
        historial="\n".join(f'- "{h}"' for h in historial) or "(nada todavía)")
    try:
        mensajes = [SystemMessage(content=system), HumanMessage(content=f"El viajero te dice: {_accion(state)}")]
        r = llm_dialogo.invoke(mensajes)
        # Sabe si (1) cita un hecho de lo que sabe y (2) el tema de la pregunta aparece en lo que sabe
        sabe = bool(r.hecho_usado.strip()) and _sabe_del_tema(_accion(state), npc, conocimientos)
        if not sabe:
            # No tiene la información → abstención (patrón "missing" de CRAG: abstenerse es correcto). Se genera
            # con un prompt MÍNIMO, sin sus conocimientos ni nombres: así no puede desviar ("pregúntale a Bron").
            r.respuesta = llm_narrador.invoke([
                SystemMessage(content=f"Eres {npc}. {datos.get('personalidad', '')} Hablas en español, en primera persona."),
                HumanMessage(content=f"El viajero te dice: {_accion(state)}\nNo sabes nada de eso. Dilo en una sola "
                                     "oración, en personaje, sin dar consejos ni mencionar a nadie.")]).content
        respuesta = r.respuesta.strip().strip('"').replace("\n", " ")
        if not respuesta:  # a veces el 7B devuelve el campo vacío
            r = llm_dialogo.invoke(mensajes + [HumanMessage(content="Responde con al menos una oración, como tu personaje.")])
            respuesta = r.respuesta.strip().strip('"').replace("\n", " ") or "..."
        dichas = [d["dijo"] for d in game_state.get("dialogos", []) if d["npc"] == npc]
        parecida = lambda a, b: len(set(_norm(a).split()) & set(_norm(b).split())) >= 0.7 * max(1, len(set(_norm(a).split())))
        if any(parecida(respuesta, d) for d in dichas):
            # El 7B copiaba su respuesta anterior: se regenera pidiendo responder a lo NUEVO
            r = llm_dialogo.invoke(mensajes + [HumanMessage(
                content=f"Eso ya lo dijiste. Responde a lo que te pregunta AHORA ({_accion(state)}) con algo distinto.")])
            respuesta = r.respuesta.strip().strip('"').replace("\n", " ")
        if re.search(rf"\b{re.escape(npc.split()[0])}\b", respuesta) or re.match(
                r"^\W*(asiente|sonr[ií]e|te mira|se (acerca|r[ií]e|encoge)|mira)\b", _norm(respuesta)):
            # Escribió narración (se nombra a sí mismo / describe gestos) en vez de hablar: un reintento
            r = llm_dialogo.invoke(mensajes + [HumanMessage(
                content="Escribe SOLO tus palabras, en primera persona, sin describir gestos ni nombrarte.")])
            respuesta = r.respuesta.strip().strip('"').replace("\n", " ")
            propio = npc.split()[0]
            frases = [f for f in re.split(r"(?<=[.!?])\s+", respuesta) if not re.search(rf"\b{re.escape(propio)}\b", f)]
            respuesta = " ".join(frases).strip() or respuesta
        respuesta = re.sub(rf"^\s*{re.escape(npc)}\s*:\s*", "", respuesta).strip().strip('"')
    except Exception:
        return {"pasos": state["pasos"][:-1] + [(paso, talk_to.invoke({"npc_name": npc}))]}  # fallback

    # Persistencia: lo que el jugador realmente oyó → evento (capa estructural del grafo) + historial del NPC
    if not any(d["npc"] == npc for d in game_state.get("dialogos", [])):
        game_state.setdefault("_disparos", []).append(f"primera conversación: {npc}")
    game_state.setdefault("dialogos", []).append({"npc": npc, "dijo": respuesta, "jugada": game_state["turn"], "sabe": sabe})
    game_state["events"].append(f"Hablo con {npc}: {respuesta}")
    try:  # v3: lo que menciona el NPC entra al grafo YA (pistas), no 3 jugadas después con el cronista
        registrar_tripletas(cronista.invoke([SystemMessage(content=SYSTEM_CRONISTA),
                                             HumanMessage(content=f"DM: {npc} dice: \"{respuesta}\"")]).hechos[:5])
    except Exception:
        pass
    obs = f'DIÁLOGO de {npc}: "{respuesta}"'
    _log("dialogo", npc=npc, respuesta=respuesta, sabe=sabe, conocimientos=conocimientos, srd=bool(srd), observacion=obs)
    return {"pasos": state["pasos"][:-1] + [(paso, obs)]}


# ── Vigilante de coherencia (v3) ─────────────────────────────────────────
# El CoherenceChecker original solo validaba creaciones (y nunca rechazó ninguna); la coherencia real la
# mantenían parches deterministas sueltos. El vigilante revisa CADA narración libre: extrae sus
# afirmaciones (como RAGAS) y las verifica contra la verdad del juego; si alguna contradice o inventa un
# hecho de juego, vuelve al narrador con la corrección (1 vez); si persiste, quita esas oraciones.
# Patrón de verificación/autocorrección (CRITIC, Reflexion) + faithfulness en línea.

class AfirmacionesJuego(BaseModel):
    afirmaciones: list[str] = Field(description="Hechos de JUEGO afirmados (máx 6): quién está, qué hay, qué pasó, "
                                                "rasgos de personajes, resultados. NO incluyas ambiente (luz, olores, sonidos).")


class Chequeo(BaseModel):
    """Verificación de UNA afirmación: primero la evidencia, después el veredicto (si el veredicto va primero,
    el modelo decide antes de leer la evidencia y marca afirmaciones correctas como contradicciones)."""
    evidencia: str = Field(description="Cita textual de la VERDAD relacionada con la afirmación, o 'ninguna'.")
    estado: Literal["compatible", "inventa", "contradice"] = Field(description=(
        "compatible: la verdad lo confirma, o es una acción/resultado que muestran las observaciones, o es un detalle "
        "menor que no cambia el juego · inventa: agrega un objeto, criatura, personaje, lugar, rasgo de personaje o "
        "resultado que NO aparece en la verdad · contradice: dice algo opuesto a la verdad"))


_extraer_af = ChatOllama(model=WATCHER_MODEL, base_url=OLLAMA_BASE_URL, temperature=0).with_structured_output(
    AfirmacionesJuego, method="json_schema")
_chequear_af = ChatOllama(model=WATCHER_MODEL, base_url=OLLAMA_BASE_URL, temperature=0).with_structured_output(
    Chequeo, method="json_schema")
NOTA_TEMPORAL = ("La VERDAD describe el estado DESPUÉS de esta jugada y las OBSERVACIONES lo que pasó en ella: es "
                 "normal que la narración cuente la transición (\"llegas a…\", \"recoges…\") si las observaciones la muestran.")
VIGILANTE_ACTIVO = os.environ.get("VIGILANTE", "1") == "1"


def _verdad_del_juego(obs, memoria):
    loc = game_state["location"]
    npcs = [n for n in world["locations"][loc]["npcs"] if n not in game_state["enemies_defeated"]]
    perfiles = [f"- {n}: {world['npcs'][n].get('personalidad') or world['npcs'][n].get('description', '')}" for n in npcs]
    return (f"ESTADO: {_state_summary()}\nLUGAR {loc}: {world['locations'][loc]['description']}\n"
            f"PERSONAJES PRESENTES:\n" + ("\n".join(perfiles) or "- nadie") +
            f"\nOBSERVACIONES DE ESTA JUGADA:\n" + ("\n".join(obs) or "(ninguna)") + f"\nMEMORIA (grafo):\n{memoria}")


def vigilante_node(state: AgentState):
    """Revisa la narración libre contra la verdad del juego (no las partes armadas por código)."""
    if not VIGILANTE_ACTIVO or not state.get("narracion_libre"):
        return {"correccion": ""}
    narracion = state["messages"][-1].content.split("[HP:")[0]
    narracion = "\n".join(l for l in narracion.splitlines() if not re.match(r"^[^:\n]{3,40}:\s*\"", l))  # sin diálogos citados
    try:  # dos pasos, como RAGAS: extraer afirmaciones de juego y verificar cada una por separado
        afs = _extraer_af.invoke(f"Extrae los hechos de juego afirmados en esta narración:\n{narracion[:1500]}").afirmaciones[:6]
        problemas = []
        for af in afs:
            ch = _chequear_af.invoke(f"{NOTA_TEMPORAL}\n\nVERDAD:\n{state.get('verdad', '')[:3500]}\n\n"
                                     f"AFIRMACIÓN: {af}\n\n¿Es compatible con la verdad?")
            if ch.estado != "compatible":
                problemas.append((ch.estado, af, ch.evidencia))
    except Exception as e:
        _log("vigilante", problemas=[], error=type(e).__name__, revisiones=state.get("revisiones", 0))
        return {"correccion": ""}
    _log("vigilante", problemas=problemas, total=len(afs), revisiones=state.get("revisiones", 0))
    if not problemas:
        return {"correccion": ""}
    if state.get("revisiones", 0) == 0:  # 1.ª vez: el narrador reescribe con la corrección
        return {"correccion": " · ".join(f"\"{af}\" {('no está en el juego' if v == 'inventa' else 'es falso')}"
                                          f" (la verdad: {ev[:120]})" for v, af, ev in problemas), "revisiones": 1}
    # 2.ª vez: se quitan las oraciones que contienen lo señalado
    malas = [set(re.findall(r"[a-z]{4,}", _norm(af))) for _, af, _ in problemas]
    texto, estado = state["messages"][-1].content.split("\n\n[HP:")[0], _state_summary()
    lineas = []
    for linea in texto.split("\n"):
        if re.match(r"^[^:\n]{3,40}:\s*\"", linea):  # los diálogos citados se conservan
            lineas.append(linea)
            continue
        frases = [f for f in re.split(r"(?<=[.!?])\s+", linea)
                  if not any(m and len(m & set(re.findall(r"[a-z]{4,}", _norm(f)))) >= max(2, len(m) // 2) for m in malas)]
        lineas.append(" ".join(frases))
    limpio = "\n".join(l for l in lineas if l.strip()).strip()
    if len(limpio) < 0.5 * len(texto):  # castigo conservador: si borraría más de la mitad, queda la versión reescrita
        return {"correccion": ""}
    return {"messages": [AIMessage(content=f"{limpio}\n\n{estado}")], "correccion": ""}


def vigilante_router(state: AgentState):
    return "narrar" if state.get("correccion") else "imagen"


# ── ImageGenerator (v3) ──────────────────────────────────────────────────
# Imagen con SDXL-Turbo: UNA POR ESCENA (cada llegada a un lugar) y UNA POR ACCIÓN (cada jugada en que el
# jugador hace algo: atacar —cada ronda—, recoger, comprar, hablar, explorar, curarse); las preguntas no generan
# imagen. Lo decide el CÓDIGO (intención + disparadores de las herramientas), no un tag del LLM. Corre después de narrar: no bloquea ni cambia la narración, y si
# falla el turno sigue sin imagen. Apagado por defecto (evaluación); IMAGENES=1 para jugar/demo.

GENERAR_IMAGENES = os.environ.get("IMAGENES", "0") == "1"
IMAGE_DEVICE = os.environ.get("IMAGE_DEVICE", "cuda:1")  # en este servidor la GPU 1; en el Spark, la que esté libre
IMAGES_DIR = "images"
_pipe_sd = None

SYSTEM_PROMPT_IMAGEN = """\
Convert this D&D scene into a prompt for an image generator.
Answer ONLY with English keywords separated by commas: place, lighting, characters, objects, mood.
Only VISIBLE things: no numbers, no game terms (damage, HP, gold amounts), no dialogue.
Name each character once with its exact count (e.g. "a single goblin", "one adventurer").
Always end with: digital painting, concept art, D&D fantasy, detailed"""

llm_imagen = ChatOllama(model=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL, temperature=0.3)


def _pipe():
    global _pipe_sd
    if _pipe_sd is None:
        import torch
        from diffusers import AutoPipelineForText2Image
        _pipe_sd = AutoPipelineForText2Image.from_pretrained(
            "stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16").to(IMAGE_DEVICE)
        _pipe_sd.set_progress_bar_config(disable=True)
    return _pipe_sd


def imagen_node(state: AgentState):
    """Genera la imagen si hubo un momento visual en esta jugada (game_state['_disparos'])."""
    motivos = list(game_state.get("_disparos", []))
    if not motivos and INTENCION["tipo"] in ("explorar", "hablar", "comprar", "atacar", "recoger", "curarse", "mover"):
        motivos = [f"acción: {INTENCION['tipo']}"]  # una imagen por ACCIÓN; las preguntas no generan imagen
    if not GENERAR_IMAGENES or not motivos:
        return {}
    t0 = time.time()
    try:
        narracion = state["messages"][-1].content.split("[HP:")[0]
        loc = game_state["location"]
        npcs = [n for n in world["locations"][loc]["npcs"] if n not in game_state["enemies_defeated"]]
        prompt = llm_imagen.invoke([SystemMessage(content=SYSTEM_PROMPT_IMAGEN), HumanMessage(
            content=f"Lugar: {loc} — {world['locations'][loc]['description']}\nPersonajes: {', '.join(npcs) or 'ninguno'}"
                    f"\nEscena: {narracion[:800]}")]).content.strip().splitlines()[0]
        # Validación: se quitan las keywords que nombran personajes o criaturas que NO están en este lugar
        # (el traductor agregaba "a single goblin" en la taberna)
        ausentes = {w for n in world["npcs"] if n not in npcs for w in _norm(n).split()[:1] if len(w) >= 4} | \
                   {"goblin", "goblins", "dragon", "orc", "troll", "witch", "wizard"} - {w for n in npcs for w in _norm(n).split()}
        prompt = ", ".join(k for k in prompt.split(",") if not any(a in _norm(k) for a in ausentes))
        imagen = _pipe()(prompt=prompt, negative_prompt="text, watermark, blurry, modern, sci-fi, anime, low quality, deformed",
                         num_inference_steps=4, guidance_scale=0.0, width=512, height=512).images[0]
        os.makedirs(IMAGES_DIR, exist_ok=True)
        ruta = f"{IMAGES_DIR}/turno_{game_state['turn']}_{loc}.png"
        imagen.save(ruta)
        _log("imagen", motivos=motivos, prompt=prompt, ruta=ruta, segundos=round(time.time() - t0, 1))
        return {"imagen": ruta}
    except Exception as e:  # la imagen nunca rompe la partida
        _log("imagen", motivos=motivos, prompt=None, ruta=None, error=f"{type(e).__name__}: {e}"[:200])
        return {}


# ── Routing ──────────────────────────────────────────────────────────────

def start_router(state: AgentState):
    """Si el jugador busca/explora, el WorldBuilder actúa primero; el narrator ve el resultado como Observación."""
    if _intent("explorar"):
        return "world_builder"
    if _intent("mover", "hablar"):  # v3: seguir una pista ("voy al pantano a buscar a la bruja") la vuelve real
        accion = _norm(_accion(state))
        for p in game_state.get("especulativos", {}).values():
            if not p.get("confirmada") and any(w in accion for w in _norm(p["nombre"]).split() if len(w) >= 4):
                return "world_builder"
    return "pensar"


def pensar_router(state: AgentState):
    paso, _ = state["pasos"][-1]
    if paso.accion == "responder":
        return "narrar"
    return "dialogo" if paso.accion == "hablar" else "actuar"  # MRKL: 'hablar' va al DialogueAgent


# ── Construir el grafo ────────────────────────────────────────────────────

graph = StateGraph(AgentState)
graph.add_node("pensar", pensar_node)
graph.add_node("actuar", actuar_node)
graph.add_node("narrar", narrar_node)
graph.add_node("world_builder", world_builder_node)
graph.add_node("coherence", coherence_node)
graph.add_node("dialogo", dialogo_node)
graph.add_node("imagen", imagen_node)
graph.add_node("vigilante", vigilante_node)

graph.add_node("clasificar", clasificar_node)
graph.add_edge(START, "clasificar")  # v3: primero se clasifica la intención (MRKL)
graph.add_conditional_edges("clasificar", start_router, {"world_builder": "world_builder", "pensar": "pensar"})
graph.add_edge("world_builder", "coherence")
graph.add_edge("coherence", "pensar")
graph.add_conditional_edges("pensar", pensar_router, {"actuar": "actuar", "dialogo": "dialogo", "narrar": "narrar"})
graph.add_edge("dialogo", "pensar")
graph.add_edge("actuar", "pensar")   # ciclo ReAct: observación → nuevo pensamiento
graph.add_edge("narrar", "vigilante")  # v3: toda narración libre pasa por el vigilante de coherencia
graph.add_conditional_edges("vigilante", vigilante_router, {"narrar": "narrar", "imagen": "imagen"})
graph.add_edge("imagen", END)

app_v2 = graph.compile()

print("Grafo Multi-Agente v2 (ReAct estructurado) compilado:")
print("  START → clasificar [intención] → (explorar) world_builder → coherence → pensar")
print("  START → pensar → actuar → pensar (ciclo ReAct) → narrar → END")
print("  pensar → (hablar) dialogo [DialogueAgent] → pensar")
print("  narrar → vigilante [coherencia] → (incoherente: narrar de nuevo, 1 vez) → imagen → END")


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# 4.1 Wrapper del agente multi-agente
# ────────────────────────────────────────────────────────────────────────────

WINDOW_PLAYS = 3  # jugadas recientes que el LLM ve completas; lo anterior pasa a la crónica

SYSTEM_CRONISTA = """\
Eres el cronista de una partida de D&D. Te muestran jugadas que van a salir de la memoria
de corto plazo. Extrae los HECHOS CLAVE de la narración que valga la pena recordar:
lugares y cómo son, personajes y lo que dijeron de importante, pistas, promesas, misterios.
- Solo hechos EXPLÍCITOS en las jugadas. No inventes.
- NO incluyas HP, oro ni inventario (eso lo registra el sistema).
- SOLO relaciones donde participa un personaje, lugar u objeto
  (nada de ambiente: olores, luz, fuego, "es agradable"...). Si no hay ninguna, lista vacía.
- Incluye las MENCIONES de lugares o personajes que todavía no se conocen, con un nombre corto:
  Goruk dice "una bruja vive en el pantano" → (Goruk, menciono, bruja del pantano) y (bruja del pantano, vive_en, pantano).
- Máximo 5 hechos, como tripletas (sujeto, relación, objeto), en español.
Ejemplo: Goruk dice "Ten cuidado en el bosque" → (Goruk, advirtio_sobre, bosque).
- "X viene por aquí cada noche" → (X, frecuenta, lugar), NO (X, vive_en, lugar).
- NO incluyas descripciones del ambiente (luz, olores, decoración) ni a dónde se puede ir (eso ya se sabe)."""


class Tripleta(BaseModel):
    sujeto: str = Field(description="Entidad (personaje, lugar, objeto). Usa el nombre tal como aparece.")
    relacion: str = Field(description="Verbo en snake_case: advirtio_sobre, vive_en, busca, teme, conoce, prometio...")
    objeto: str = Field(description="Entidad o concepto relacionado.")


class Cronica(BaseModel):
    hechos: list[Tripleta] = Field(description="Hechos clave nuevos como tripletas (máximo 5).")


cronista = llm_strict.with_structured_output(Cronica, method="json_schema")
MAX_HECHOS = 15


GENERICOS = {"algo", "alguien", "lugar", "cosa", "cosas", "viajero", "jugador", "gente", "aventura", "nada",
             "historia", "historias", "interes", "ayuda", "favor"}


def registrar_tripletas(nuevos):
    """Capa narrativa del grafo (v3): se acepta una tripleta si AL MENOS UN extremo es una entidad conocida
    (lugar, NPC u objeto); si el otro no existe, se registra como PISTA especulativa: existe en el grafo como
    referencia narrativa ("Goruk mencionó una bruja en el pantano"), no como entidad del mundo (no se puede ir
    ni hablar con ella) hasta que el WorldBuilder la cree de verdad al seguir la pista."""
    sync_graph()
    pistas = game_state.setdefault("especulativos", {})
    for h in nuevos:
        rel = h.relacion.strip().lower().replace(" ", "_")
        if any(t_ in rel for t_ in ("asien", "sonri", "mira", "observ", "salud", "sient", "suspir", "ilumin", "huele",
                                    "pregunt", "ofrec", "interes", "invit", "banad", "brill", "decor", "suena",
                                    "puede_ir", "conecta", "ubicacion", "tiene_luz", "calid")):  # gestos, estética, mapa
            continue
        s_, o_ = _resolve(h.sujeto), _resolve(h.objeto)
        conocido = lambda n: n and G.nodes[n].get("tipo") in ("lugar", "objeto", "npc")
        if conocido(s_) and G.nodes[s_].get("tipo") == "npc" and rel in ("vive_en", "esta_en", "vive", "habita_en") \
                and world["npcs"].get(s_, {}).get("location") and o_ != world["npcs"][s_]["location"]:
            continue  # contradice la capa estructural (Bron vive en la herrería, no en la taberna)
        if conocido(s_) and conocido(o_):
            t = (s_, rel, o_, game_state["turn"])
        elif conocido(s_) or conocido(o_):
            nuevo = (h.objeto if conocido(s_) else h.sujeto).strip()
            clave = _norm(nuevo).replace(" ", "_")
            if len(clave) < 4 or _norm(nuevo) in GENERICOS or G.has_node(nuevo) and not G.nodes[nuevo].get("speculative"):
                continue
            if clave not in pistas:
                pistas[clave] = {"nombre": nuevo, "mencionado_por": s_ if conocido(s_) else o_, "relacion": rel,
                                 "jugada": game_state["turn"], "confirmada": None}
            t = (s_, rel, clave, game_state["turn"]) if conocido(s_) else (clave, rel, o_, game_state["turn"])
        else:
            continue  # dos entidades inventadas de la nada: no
        if t[:3] not in [x[:3] for x in narrative_triples]:
            narrative_triples.append(t)


def update_chronicle(dropped):
    """Incorpora a la crónica los hechos de las jugadas que salen de la ventana de contexto."""
    texto = "\n".join(f"Jugador: {a}\nDM: {r[:500]}" for a, r in dropped)
    try:
        nuevos = cronista.invoke([SystemMessage(content=SYSTEM_CRONISTA), HumanMessage(content=texto)]).hechos[:5]
    except Exception:
        nuevos = []
    antes = len(narrative_triples)
    registrar_tripletas(nuevos)
    _log("cronista", jugadas_que_salen=[a for a, _ in dropped],
         propuestas=[(h.sujeto, h.relacion, h.objeto) for h in nuevos],
         aceptadas=[t[:3] for t in narrative_triples[antes:]])
    hechos = [f"{s} —{r}→ {o}" for s, r, o, _ in narrative_triples]
    game_state["chronicle"] = "\n".join(f"- {h}" for h in hechos[-MAX_HECHOS:])


def imprimir_traza(desde=0, ancho=300):
    """Imprime las entradas nuevas de TRAZA (qué decidió cada agente). Devuelve cuántas van impresas."""
    corto = lambda x, n=ancho: (str(x)[:n] + "…") if len(str(x)) > n else str(x)
    for nodo, d in TRAZA[desde:]:
        if nodo == "clasificar":
            print(f"  🧭 INTENCIÓN: {d['tipo']}")
        elif nodo == "world_builder":
            dec = d["decision"] or {}
            print(f"  🌍 WORLD_BUILDER  SRD recuperado: {d['refs_srd']}")
            print(f"       decide: {dec.get('tipo')} '{dec.get('nombre')}' — {corto(dec.get('razon', ''))}")
            if d["resultado"]:
                print(f"       → {corto(d['resultado'])}")
            if d["anclado_a"]:
                print(f"       anclado al SRD: {d['anclado_a']}")
        elif nodo == "coherence":
            print(f"  🛡️  COHERENCE  {d['veredicto']}  {'→ CREACIÓN DESHECHA' if d['deshecho'] else ''}")
        elif nodo == "pensar":
            cambio = f"  (el planner propuso {d['propuesto'][0]}({d['propuesto'][1]}); una regla lo cambió)" \
                if tuple(d["propuesto"]) != tuple(d["final"]) else ""
            print(f"  🧠 PENSAR [{d['intencion']}] {corto(d['pensamiento'], 160)}")
            print(f"       → {d['final'][0]}({d['final'][1]}){cambio}   [{d['tokens']} tokens]")
        elif nodo == "dialogo":
            if d.get("respuesta"):
                print(f"  💬 DIALOGO {d['npc']} (perfil {len(d['conocimientos'])} hechos{', +SRD' if d['srd'] else ''}"
                      f"{'' if d.get('sabe', True) else ', NO SABE → abstención'})")
                print(f"       \"{corto(d['respuesta'])}\"")
            else:
                print(f"  💬 DIALOGO {d['npc']} → {corto(d['observacion'])}")
        elif nodo == "vigilante":
            if d.get("error"):
                print(f"  🛡️  VIGILANTE no pudo revisar ({d['error']})")
            elif not d["problemas"]:
                print(f"  🛡️  VIGILANTE ✓ {d['total']} afirmaciones, ninguna inventa ni contradice")
            else:
                accion_v = "el narrador reescribe" if d["revisiones"] == 0 else "se quitan esas oraciones"
                print(f"  🛡️  VIGILANTE ✗ {len(d['problemas'])} problema(s) → {accion_v}")
                for v, t, ev in d["problemas"]:
                    print(f"       [{v}] \"{corto(t, 120)}\" (evidencia: {corto(ev, 100)})")
        elif nodo == "imagen":
            if d.get("ruta"):
                print(f"  🎨 IMAGEN ({', '.join(d['motivos'])}) {d['segundos']}s → {d['ruta']}")
                print(f"       prompt: {corto(d['prompt'], 200)}")
            else:
                print(f"  🎨 IMAGEN falló: {d.get('error')}")
        elif nodo == "actuar":
            print(f"  🔧 ACTUAR {d['herramienta']}({d['args']}) → {corto(d['observacion'])}")
        elif nodo == "narrar":
            print(f"  🕸️  GRAFO (contexto recuperado para esta jugada):")
            for linea in d["memoria_grafo"].splitlines()[:25]:
                print(f"       {linea[:ancho]}")
        elif nodo == "cronista":
            print(f"  📜 CRONISTA  salen de la ventana: {d['jugadas_que_salen']}")
            print(f"       propuso: {d['propuestas']}")
            print(f"       al grafo: {d['aceptadas']}")
    return len(TRAZA)


class MultiAgentDM:
    """Dungeon Master multi-agente (ReAct estructurado) con mundo dinámico y memoria persistente.

    Memoria en dos niveles:
      - Corto plazo: el LLM ve solo las últimas WINDOW_PLAYS jugadas (acción + narración),
        así el contexto no crece sin límite.
      - Largo plazo: el registro del mundo (world) y la crónica viven en Python y se inyectan
        en los system prompts / se consultan con consult_world.
    """

    def __init__(self, app):
        self.app = app
        self.history = []         # [(acción, narración)] dentro de la ventana
        self.turn_times = []
        self.context_tokens = []  # tokens de prompt máximos por jugada (para graficar)
        self.react_traces = []    # [(pensamiento, acción, argumento, observación)] por jugada
        self.expansion_log = []   # registro de expansiones
        self.ultima_imagen = None
        reset_game()
        reset_graph()

    def play(self, action: str, traza: bool = False) -> str:
        """Juega una acción. traza=True imprime lo que hace cada agente a medida que se ejecuta."""
        TRAZA.clear()
        game_state["turn"] += 1
        game_state["_disparos"] = []
        avanzar_tiempo()
        window = [m for a, r in self.history for m in (HumanMessage(content=a), AIMessage(content=r))]
        state = {"messages": window + [HumanMessage(content=action)],
                 "pasos": [], "expansions": 0, "prompt_tokens": 0, "revisiones": 0, "correccion": ""}
        t0 = time.time()
        try:
            # stream en vez de invoke: mismo resultado, pero nodo a nodo (para la traza)
            result, impresos = state, 0
            for modo, chunk in self.app.stream(state, {"recursion_limit": 40}, stream_mode=["updates", "values"]):
                if modo == "values":
                    result = chunk
                elif traza:
                    impresos = imprimir_traza(impresos)
        except Exception as e:  # no romper la partida por un fallo puntual
            result = {**state, "messages": state["messages"] + [AIMessage(content=f"(error del DM: {type(e).__name__})")]}
        self.turn_times.append(time.time() - t0)
        self.context_tokens.append(result.get("prompt_tokens", 0))
        self.ultima_imagen = result.get("imagen")  # v3: ruta de la imagen de esta jugada (o None)
        self.react_traces.append([(p.pensamiento, p.accion, p.argumento, o) for p, o in result.get("pasos", [])])

        # Registrar si hubo expansión
        new_locs = [l for l in game_state.get("locations_created", [])
                    if l not in [e.get("location") for e in self.expansion_log]]
        for loc in new_locs:
            self.expansion_log.append({
                "location": loc,
                "turn": len(self.turn_times),
                "description": world["locations"].get(loc, {}).get("description", ""),
            })

        last = result["messages"][-1]
        final = last.content if isinstance(last, AIMessage) and last.content else "(sin respuesta)"

        # Ventana: las últimas jugadas; lo que sale va a la crónica
        self.history.append((action, final))
        if len(self.history) > WINDOW_PLAYS:
            dropped, self.history = self.history[:-WINDOW_PLAYS], self.history[-WINDOW_PLAYS:]
            update_chronicle(dropped)
        if traza:
            imprimir_traza(impresos)
        return final


# ── Baseline v1 (para comparar) ──────────────────────────────────────────

SYSTEM_PROMPT_BASELINE = """\
Eres un Dungeon Master de D&D. Narras una aventura interactiva en espanol.

MUNDO:
- Ubicaciones: taberna, plaza, herreria, bosque, cueva.
- Taberna conecta con plaza y bosque. Plaza conecta con taberna, herreria y bosque.
  Herreria conecta con plaza. Bosque conecta con plaza, taberna y cueva. Cueva conecta con bosque.
- NPCs: Goruk (taberna), Elena (plaza), Bron (herreria), Goblin (cueva, hostil).

JUGADOR: HP 20/20, Oro 15, Inventario: espada oxidada, pocion de curacion. Ubicacion: taberna.

REGLAS:
- Solo movimiento a ubicaciones conectadas.
- Combate: el jugador ataca con d20+5 contra la CA del enemigo (goblin: CA 15, 7 PG, ataca con +4 y hace 1d6+2);
  el jugador tiene CA 14. Varios golpes hasta bajar los PG a 0.
- Elena vende pociones de curación a 10 de oro. Bron afila la espada por 5 de oro.
- Si el jugador busca algo que no existe, puedes inventarlo de forma coherente.
- Narra brevemente (3-4 oraciones). Muestra [HP: X/20 | Oro: X | Ubicacion: X | Inventario: ...]"""


class BaselineDM:
    def __init__(self, llm):
        self.llm = llm
        self.messages = [SystemMessage(content=SYSTEM_PROMPT_BASELINE)]
        self.turn_times = []

    def play(self, action: str) -> str:
        self.messages.append(HumanMessage(content=action))
        t0 = time.time()
        response = self.llm.invoke(self.messages)
        self.turn_times.append(time.time() - t0)
        self.messages.append(response)
        return response.content


# ── Agente v1 simple (para comparar) ────────────────────────────────────

SYSTEM_PROMPT_V1 = """\
Eres un Dungeon Master de D&D. Narras una aventura interactiva en español.
INSTRUCCIONES OBLIGATORIAS:
1. SIEMPRE llama a check_state PRIMERO.
2. Si el jugador quiere MOVERSE, usa move_to.
3. Si el jugador quiere RECOGER, usa pick_up_item.
4. Si el jugador quiere ATACAR, usa attack_enemy.
5. Si el jugador quiere CURARSE, usa use_potion.
6. Para dados, usa roll_dice.
NUNCA respondas sin usar check_state primero."""

llm_v1_tools = llm.bind_tools(V1_TOOLS)

class V1State(TypedDict):
    messages: Annotated[list, add_messages]


def v1_agent_node(state: V1State):
    state_info = _state_summary()
    system = SYSTEM_PROMPT_V1 + f"\n\nESTADO: {state_info}"
    messages = [SystemMessage(content=system)] + state["messages"]
    response = llm_v1_tools.invoke(messages)
    return {"messages": [response]}

def v1_should_continue(state: V1State):
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"
    return END

v1_tool_node = ToolNode(V1_TOOLS)
graph_v1 = StateGraph(V1State)
graph_v1.add_node("agent", v1_agent_node)
graph_v1.add_node("tools", v1_tool_node)
graph_v1.add_edge(START, "agent")
graph_v1.add_conditional_edges("agent", v1_should_continue, {"tools": "tools", END: END})
graph_v1.add_edge("tools", "agent")
app_v1 = graph_v1.compile()


class AgentDMv1:
    def __init__(self, app):
        self.app = app
        self.state = {"messages": []}
        self.turn_times = []
        reset_game()

    def play(self, action: str) -> str:
        self.state["messages"].append(HumanMessage(content=action))
        t0 = time.time()
        result = self.app.invoke(self.state)
        self.turn_times.append(time.time() - t0)
        self.state = result
        for msg in reversed(result["messages"]):
            if isinstance(msg, AIMessage) and msg.content and not getattr(msg, 'tool_calls', None):
                return msg.content
        return result["messages"][-1].content if result["messages"] else "(sin respuesta)"


# Prueba rápida del multi-agente
print("Probando multi-agente v2...")
dm_v2 = MultiAgentDM(app_v2)
resp = dm_v2.play("Miro a mi alrededor en la taberna.")
print(f"v2 — Turno 1: {resp[:200]}...")
print(f"Tiempo: {dm_v2.turn_times[-1]:.2f}s")
print(f"Mundo: {_world_summary()}")
print(f"Tokens de prompt: {dm_v2.context_tokens[-1]}")

## 5. Evaluación: secuencia con exploración dinámica

La secuencia v2 incluye los mismos 25 turnos de v1 **más 10 turnos adicionales** diseñados para provocar expansión del mundo:
- Buscar puertas escondidas
- Explorar rincones no mapeados
- Preguntar por NPCs desconocidos
- Navegar a ubicaciones recién creadas

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Secuencia de prueba v2: 35 turnos (25 de v1 + 10 de expansión)
# ────────────────────────────────────────────────────────────────────────────

TEST_SEQUENCE_DEV = [
    # === BLOQUE 1-5: mismos de v1 (exploración, reglas, combate, navegación, memoria) ===
    ("Miro a mi alrededor en la taberna.",
     "¿Qué objetos hay aquí que pueda recoger?",
     "item_mention", "antorcha"),
    ("Recojo la antorcha.",
     "¿Qué tengo en mi inventario ahora?",
     "inventory", "antorcha"),
    ("Hablo con Goruk el tabernero.",
     "¿Dónde estoy?",
     "location", "taberna"),
    ("Voy al bosque.",
     "¿Dónde estoy ahora?",
     "location", "bosque"),
    ("Recojo las hierbas medicinales.",
     "¿Qué tengo en mi inventario?",
     "inventory", "hierbas"),
    ("Voy a la herrería.",
     "¿Pude ir a la herrería desde el bosque?",
     "illegal_move", "no"),
    ("¿Dónde estoy exactamente?",
     "¿Sigo en el bosque?",
     "location", "bosque"),
    ("Voy a la cueva.",
     "¿Dónde estoy?",
     "location", "cueva"),
    ("Reviso mi equipo antes de avanzar.",
     "¿Qué objetos tengo exactamente?",
     "inventory_full", "antorcha,hierbas medicinales,espada oxidada,pocion de curacion"),
    ("Ataco al goblin con mi espada.",
     "¿Cuántos HP tengo ahora?",
     "hp_mentioned", "hp"),
    ("Abro el cofre y tomo las monedas de oro.",
     "¿Cuánto oro tengo ahora?",
     "gold_change", "gold"),

    # === BLOQUE 6: EXPANSIÓN DINÁMICA (turnos 12-21) — solo v2 lo maneja bien ===

    # Turno 12: buscar algo escondido en la cueva
    ("Examino las paredes de la cueva buscando pasadizos secretos.",
     "¿Encontré algo nuevo?",
     "expansion", "nuevo"),

    # Turno 13: entrar a la nueva ubicación (si fue creada)
    ("Entro al nuevo lugar que descubrí.",
     "¿Dónde estoy ahora?",
     "new_location", "dynamic"),

    # Turno 14: interactuar en la nueva zona
    ("Miro a mi alrededor en este nuevo lugar.",
     "¿Hay algo interesante aquí?",
     "exploration", "desc"),

    # Turno 15: volver y verificar que la ruta existe
    ("Vuelvo a la cueva.",
     "¿Puedo volver al lugar que descubrí?",
     "connection_check", "si"),

    # Turno 16: volver al bosque
    ("Vuelvo al bosque.",
     "¿Dónde estoy?",
     "location", "bosque"),

    # Turno 17: buscar un sendero nuevo en el bosque
    ("Busco un sendero escondido entre los árboles.",
     "¿Encontré un camino nuevo?",
     "expansion", "nuevo"),

    # Turno 18: ir a la plaza
    ("Voy a la plaza.",
     "¿Dónde estoy ahora?",
     "location", "plaza"),

    # Turno 19: buscar NPCs nuevos
    ("Busco a alguien interesante en la plaza que no haya visto antes.",
     "¿Hay algún personaje nuevo?",
     "npc_creation", "nuevo"),

    # Turno 20: volver a la taberna
    ("Voy a la taberna.",
     "¿Dónde estoy?",
     "location", "taberna"),

    # Turno 21: buscar una bodega escondida en la taberna
    ("Busco una puerta escondida detrás de la barra de la taberna.",
     "¿Encontré algo?",
     "expansion", "nuevo"),

    # === BLOQUE 7: MEMORIA DE EXPANSIONES (turnos 22-25) ===

    # Turno 22: ¿recuerda las expansiones?
    ("¿Cuántos lugares nuevos descubrí durante la aventura?",
     "¿Qué lugares nuevos existen ahora?",
     "expansion_memory", "dynamic"),

    # Turno 23: usar poción
    ("Uso mi poción de curación.",
     "¿Se eliminó la poción de mi inventario?",
     "potion_used", "pocion"),

    # Turno 24: inventario final
    ("Reviso todo mi inventario.",
     "¿Qué tengo exactamente?",
     "inventory_full", "espada oxidada,antorcha,hierbas medicinales"),

    # Turno 25: resumen de la aventura
    ("Resume todo lo que me pasó en esta aventura.",
     "¿Qué eventos importantes ocurrieron?",
     "event_memory", "goblin"),

    # === BLOQUE 8: MEMORIA RELACIONAL (turnos 26-27) — requieren relaciones entre entidades ===

    # Turno 26: ruta (el jugador está en la taberna) → taberna → bosque → cueva
    ("¿Cómo vuelvo a la cueva desde aquí?",
     "¿Por qué lugares tengo que pasar?",
     "route", "bosque"),

    # Turno 27: quién dijo qué (turno 3: Goruk advirtió sobre el bosque)
    ("¿Quién me advirtió sobre el bosque al principio?",
     "¿Qué me dijo exactamente?",
     "relation_memory", "goruk"),
]

EVENTOS_CANONICOS_DEV = ("tomó la antorcha, habló con Goruk, recogió hierbas, intentó ir a la herrería sin "
                         "conexión, fue a la cueva, combatió al goblin, tomó el oro del cofre, buscó lugares ocultos.")

# ────────────────────────────────────────────────────────────────────────────
# Secuencia HOLD-OUT: evaluación final
# ────────────────────────────────────────────────────────────────────────────
# Toda la iteración de v2 (memoria, ReAct estructurado, grafo, arreglos de debilidades) se hizo
# mirando los fallos de TEST_SEQUENCE_DEV → evaluar con ella sobreestima a v2 (sobreajuste al
# conjunto de prueba). Esta secuencia NO se usó para ajustar nada: mismos tipos de turno, pero
# otras acciones, otro recorrido (plaza → herrería primero), otros NPCs y otras rutas.
# Además agrega 4 turnos de REGLAS (SRD) que miden precisión factual.

TEST_SEQUENCE_HOLDOUT = [
    ("Observo la taberna con atención.", "¿Hay algo que pueda llevarme?", "item_mention", "antorcha"),
    ("Salgo hacia la plaza.", "¿En qué lugar estoy?", "location", "plaza"),
    ("Hablo con Elena la comerciante.", "¿Qué me ofreció Elena?", "relation_memory", "elena"),
    ("Voy a la herrería.", "¿Dónde me encuentro?", "location", "herreria"),
    ("Tomo el escudo de hierro.", "¿Qué llevo en el inventario?", "inventory", "escudo"),
    ("Hablo con Bron el herrero.", "¿Qué me dijo Bron?", "relation_memory", "bron"),
    ("Voy directamente a la cueva.", "¿Llegué a la cueva?", "illegal_move", "cueva"),
    ("¿Dónde estoy ahora mismo?", "¿Sigo en la herrería?", "location", "herreria"),
    ("Regreso a la plaza.", "¿Dónde estoy?", "location", "plaza"),
    ("Voy al bosque.", "¿Dónde estoy ahora?", "location", "bosque"),
    ("Avanzo hacia la cueva.", "¿Dónde estoy?", "location", "cueva"),
    ("Golpeo al goblin con mi espada.", "¿Cuánta vida me queda?", "hp_mentioned", "hp"),
    ("Saco las monedas del cofre.", "¿Cuánto oro llevo?", "gold_change", "gold"),
    # SRD 1: stats de una criatura (SRD: Goblin CA 15, PG 7, cimitarra 1d6+2)
    ("Según las reglas de D&D, ¿cuántos puntos de golpe y qué clase de armadura tiene un goblin?",
     "¿Y con qué ataca normalmente?", "srd", "7,15"),
    ("Investigo el fondo de la cueva en busca de una grieta oculta.", "¿Hallé algo?", "expansion", "nuevo"),
    ("Entro en lo que encontré.", "¿Dónde estoy ahora?", "new_location", "dynamic"),
    # SRD 2: ¿es venenosa? (SRD: Giant Spider, mordida 1d8+3 + salvación CON CD 11 o 2d8 de veneno)
    ("Si aquí hubiera una araña gigante, ¿sería venenosa según las reglas?",
     "¿Qué daño hace su mordedura?", "srd", "venen,2d8"),
    ("Vuelvo a la cueva.", "¿Puedo regresar al lugar nuevo?", "connection_check", "si"),
    ("Busco algún objeto de valor escondido entre los huesos.", "¿Encontré algún objeto?", "expansion", "nuevo"),
    ("Vuelvo al bosque.", "¿Dónde estoy?", "location", "bosque"),
    ("Uso mi poción de curación.", "¿Me queda alguna poción?", "potion_used", "pocion"),
    # SRD 3: objeto (SRD: Potion of Healing, común, 2d4 + 2 PG)
    ("¿Qué hace exactamente una poción de curación según las reglas?", "¿Cuántos puntos cura?", "srd", "2d4"),
    # SRD 4: regla (SRD: ventaja → se tira un segundo d20 y se usa el mayor)
    ("¿Cómo funciona la ventaja en una tirada según las reglas?", "¿Qué dado tiro?", "srd", "d20"),
    ("¿Cómo llego a la herrería desde aquí?", "¿Qué lugares cruzo?", "route", "bosque→plaza→herreria"),
    ("¿Quién me ofreció afilar la espada?", "¿Por cuánto oro?", "relation_memory", "bron"),
    ("¿Qué lugares nuevos descubrí?", "¿Cuántos son?", "expansion_memory", "dynamic"),
    ("Cuéntame todo lo que hice hoy.", "¿Qué fue lo más importante?", "event_memory", "goblin"),
]

EVENTOS_CANONICOS_HOLDOUT = (
    "miró la taberna, fue a la plaza, habló con Elena (vende pociones), fue a la herrería, tomó el escudo de "
    "hierro, habló con Bron (afilar la espada por 5 de oro), intentó ir a la cueva sin conexión, volvió a la "
    "plaza, fue al bosque y a la cueva, combatió al goblin, tomó 50 monedas del cofre, buscó una grieta "
    "oculta y objetos entre los huesos, usó la poción de curación.")

# Documentos del SRD que responden cada pregunta de reglas (ground truth del juez y contexto de faithfulness)
SRD_GOLD = {
    TEST_SEQUENCE_HOLDOUT[13][0]: ["monster:goblin"],
    TEST_SEQUENCE_HOLDOUT[16][0]: ["monster:giant-spider"],
    TEST_SEQUENCE_HOLDOUT[21][0]: ["magic_item:potion-of-healing"],
    TEST_SEQUENCE_HOLDOUT[22][0]: ["rule:advantage-and-disadvantage"],
}

# ────────────────────────────────────────────────────────────────────────────
# Secuencia HOLD-OUT 2 — escrita ANTES de aplicar los arreglos post-hold-out-1
# ────────────────────────────────────────────────────────────────────────────
# La secuencia hold-out 1 se "gastó": sus fallos guiaron los arreglos de v2. Esta segunda secuencia
# no se usó para ajustar nada y da el número final de generalización. Otro vocabulario de acciones
# ("Agarro", "Me dirijo", "Camino hasta", "Sigo hacia", "Paso a", "Me curo"), otro recorrido y otras
# preguntas de reglas (lobo, esqueleto, bolsa de contención, cobertura).

TEST_SEQUENCE_HOLDOUT2 = [
    ("Echo un vistazo a mi alrededor.", "¿Qué objetos veo por aquí?", "item_mention", "antorcha"),
    ("Agarro la antorcha.", "¿Qué tengo encima?", "inventory", "antorcha"),
    ("Charlo con Goruk el tabernero.", "¿Qué me contó Goruk?", "relation_memory", "goruk"),
    ("Me dirijo al bosque.", "¿Dónde estoy?", "location", "bosque"),
    ("Intento ir a la herrería desde aquí.", "¿Llegué a la herrería?", "illegal_move", "herreria"),
    ("Cojo las hierbas medicinales.", "¿Qué llevo ahora?", "inventory", "hierbas"),
    # SRD 1: criatura (SRD: Wolf CA 13, PG 11, mordida 2d4+2 + salvación FUE CD 11 o derribado)
    ("Según las reglas, ¿qué estadísticas tiene un lobo?", "¿Qué pasa si me muerde?", "srd", "13,11"),
    ("Me adentro en la cueva.", "¿Dónde me encuentro?", "location", "cueva"),
    ("Lucho contra el goblin.", "¿Cuántos puntos de vida tengo?", "hp_mentioned", "hp"),
    ("Recojo el oro del cofre.", "¿Cuánto oro tengo ahora?", "gold_change", "gold"),
    ("Exploro un rincón oscuro de la cueva por si hay un túnel.", "¿Descubrí algo?", "expansion", "nuevo"),
    ("Atravieso lo que acabo de descubrir.", "¿En qué lugar estoy?", "new_location", "dynamic"),
    # SRD 2: criatura (SRD: Skeleton CA 13, PG 13, inmune al veneno)
    ("Según las reglas, ¿cuántos puntos de golpe tiene un esqueleto?", "¿Es inmune a algo?", "srd", "13,venen"),
    ("Regreso a la cueva.", "¿Puedo volver a ese lugar nuevo?", "connection_check", "si"),
    ("Camino hasta el bosque.", "¿Dónde estoy?", "location", "bosque"),
    ("Sigo hacia la plaza.", "¿Dónde estoy ahora?", "location", "plaza"),
    ("Busco algún forastero en la plaza al que no conozca.", "¿Hay alguien nuevo?", "npc_creation", "nuevo"),
    # SRD 3: objeto (SRD: Bag of Holding, hasta 500 libras, 64 pies cúbicos, pesa 15 libras)
    ("¿Qué hace exactamente una bolsa de contención según las reglas?", "¿Cuánto peso aguanta?", "srd", "500"),
    ("Me curo con la poción.", "¿Todavía tengo la poción?", "potion_used", "pocion"),
    ("Paso a la herrería.", "¿Dónde estoy?", "location", "herreria"),
    # SRD 4: regla (SRD: Cover — media cobertura +2 a CA, tres cuartos +5)
    ("Según las reglas, ¿cuánto ayuda estar detrás de media cobertura?", "¿Y si la cobertura es de tres cuartos?",
     "srd", "+2,+5"),
    ("¿Cómo vuelvo a la cueva desde aquí?", "¿Qué lugares atravieso?", "route", "plaza→bosque→cueva"),
    ("¿Quién me advirtió del bosque?", "¿Con qué palabras?", "relation_memory", "goruk"),
    ("Reviso todo lo que llevo.", "¿Qué tengo exactamente?", "inventory_full",
     "espada oxidada,antorcha,hierbas medicinales"),
    ("¿Cuántos sitios nuevos encontré?", "¿Cuáles son?", "expansion_memory", "dynamic"),
    ("Hazme un resumen de la aventura.", "¿Qué fue lo más importante?", "event_memory", "goblin"),
]

EVENTOS_CANONICOS_HOLDOUT2 = (
    "miró la taberna, tomó la antorcha, habló con Goruk (advirtió sobre el bosque), fue al bosque, intentó ir "
    "a la herrería sin conexión, recogió hierbas medicinales, fue a la cueva, luchó contra el goblin, tomó 50 "
    "monedas del cofre, exploró buscando un túnel, volvió al bosque y a la plaza, buscó a un forastero, usó la "
    "poción de curación, fue a la herrería.")

SRD_GOLD_HOLDOUT2 = {
    TEST_SEQUENCE_HOLDOUT2[6][0]: ["monster:wolf"],
    TEST_SEQUENCE_HOLDOUT2[12][0]: ["monster:skeleton"],
    TEST_SEQUENCE_HOLDOUT2[17][0]: ["magic_item:bag-of-holding"],
    TEST_SEQUENCE_HOLDOUT2[20][0]: ["rule:cover"],
}

SRD_GOLD.update(SRD_GOLD_HOLDOUT2)

# ────────────────────────────────────────────────────────────────────────────
# Secuencia HOLD-OUT 3 — escrita ANTES de evaluar la v3 (14-oct, noche)
# ────────────────────────────────────────────────────────────────────────────
# El hold-out 2 ya se usó para evaluar v2 y mirar sus fallos. Esta secuencia da el número limpio de la v3.
# Cubre lo nuevo de la v3: comprar (Elena) y pagar un servicio (Bron), combate de varias rondas (CA y PG del
# SRD), diálogo con NPCs y reglas del SRD que no aparecieron antes (oso pardo, caídas).
# Recorrido: taberna → plaza → herrería → plaza → bosque → cueva → (descubrimiento) → cueva → bosque.

TEST_SEQUENCE_HOLDOUT3 = [
    ("Echo una mirada por la taberna.", "¿Qué cosas hay que pueda tomar?", "item_mention", "antorcha"),
    ("Le pregunto a Goruk qué se cuenta por el pueblo.", "¿Qué me dijo Goruk?", "relation_memory", "goruk"),
    ("Me dirijo a la plaza.", "¿En dónde estoy parado?", "location", "plaza"),
    ("Bebo mi poción de curación.", "¿Me queda alguna poción?", "potion_used", "pocion"),
    ("Le compro una poción a Elena.", "¿Cuánto oro me queda?", "gold_change", "gold"),
    ("Reviso lo que llevo.", "¿Llevo alguna poción ahora?", "inventory", "pocion"),
    ("Voy a la herrería.", "¿Dónde estoy ahora?", "location", "herreria"),
    ("Le pago a Bron para que afile mi espada.", "¿Cómo está mi espada ahora?", "inventory", "afilada"),
    ("Intento cruzar directo al bosque desde aquí.", "¿Llegué al bosque?", "illegal_move", "bosque"),
    ("Vuelvo a la plaza.", "¿Dónde estoy?", "location", "plaza"),
    ("Me adentro en el bosque.", "¿Dónde me encuentro?", "location", "bosque"),
    # SRD 1: criatura (SRD: Brown Bear CA 11, PG 34, mordisco 1d8+4, garras 2d6+4)
    ("Según las reglas, ¿qué tan resistente es un oso pardo?", "¿Cuánto daño hacen sus garras?", "srd", "11,34"),
    ("Sigo hasta la cueva.", "¿Dónde estoy?", "location", "cueva"),
    ("Ataco al goblin.", "¿Cómo va la pelea?", "hp_mentioned", "hp"),
    ("Lo ataco de nuevo.", "¿Cuánta vida me queda?", "hp_mentioned", "hp"),
    ("Sigo atacando al goblin.", "¿Cuánta vida tengo?", "hp_mentioned", "hp"),
    ("Examino los rincones de la cueva por si hay otra salida.", "¿Encontré algo?", "expansion", "nuevo"),
    # SRD 2: regla (SRD: The Environment — caída: 1d6 contundente por cada 10 pies, máximo 20d6)
    ("Según las reglas, ¿cuánto daño hace una caída?", "¿Hay un máximo?", "srd", "1d6,10"),
    ("Entro en lo que haya encontrado.", "¿Dónde estoy ahora?", "new_location", "dynamic"),
    ("Regreso a la cueva.", "¿Puedo volver a ese sitio?", "connection_check", "si"),
    ("Vuelvo al bosque.", "¿Dónde estoy?", "location", "bosque"),
    ("¿Cómo llego a la herrería desde aquí?", "¿Qué lugares cruzo?", "route", "plaza→herreria"),
    ("¿Quién me afiló la espada?", "¿Cuánto le pagué?", "relation_memory", "bron"),
    ("¿Qué sitios nuevos conocí?", "¿Cuántos son?", "expansion_memory", "dynamic"),
    ("Cuéntame la aventura completa.", "¿Qué fue lo más importante?", "event_memory", "goblin"),
    ("Reviso todo mi equipo.", "¿Qué llevo exactamente?", "inventory_full", "espada afilada,pocion de curacion"),
]

EVENTOS_CANONICOS_HOLDOUT3 = (
    "miró la taberna, habló con Goruk, fue a la plaza, bebió su poción, compró una poción a Elena por 10 de oro, "
    "fue a la herrería, pagó 5 de oro a Bron para afilar la espada, intentó ir al bosque sin conexión, volvió a la "
    "plaza, fue al bosque y a la cueva, combatió al goblin varias rondas, exploró la cueva buscando otra salida.")
ORO_CANONICO_HOLDOUT3 = "5 (15 iniciales − 10 de la poción)"

SRD_GOLD_HOLDOUT3 = {
    TEST_SEQUENCE_HOLDOUT3[11][0]: ["monster:brown-bear"],
    TEST_SEQUENCE_HOLDOUT3[17][0]: ["rule:the-environment"],
}

SRD_GOLD.update(SRD_GOLD_HOLDOUT3)

# "holdout3" = evaluación de la v3 (escrita antes de evaluarla) · "holdout2" = evaluación final de v2 ·
# "holdout" = hold-out 1 (usado para corregir) · "dev" = la usada para ajustar
SECUENCIA = os.environ.get("SECUENCIA", "holdout3")
TEST_SEQUENCE_V2 = {"holdout3": TEST_SEQUENCE_HOLDOUT3, "holdout2": TEST_SEQUENCE_HOLDOUT2,
                    "holdout": TEST_SEQUENCE_HOLDOUT, "dev": TEST_SEQUENCE_DEV}[SECUENCIA]
EVENTOS_CANONICOS = {"holdout3": EVENTOS_CANONICOS_HOLDOUT3, "holdout2": EVENTOS_CANONICOS_HOLDOUT2,
                     "holdout": EVENTOS_CANONICOS_HOLDOUT, "dev": EVENTOS_CANONICOS_DEV}[SECUENCIA]
ORO_CANONICO = ORO_CANONICO_HOLDOUT3 if SECUENCIA == "holdout3" else "65 (15 iniciales + 50 del cofre)"
print(f"Secuencia de evaluación: {SECUENCIA} ({len(TEST_SEQUENCE_V2)} turnos)")


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# 5.1 Ejecutar BASELINE
# ────────────────────────────────────────────────────────────────────────────

print("=" * 78)
print("EJECUTANDO BASELINE (LLM directo)")
print("=" * 78)

baseline_dm = BaselineDM(llm)
baseline_responses = []

for i, (action, question, field, expected) in enumerate(TEST_SEQUENCE_V2):
    print(f"\n--- Turno {i+1} ---")
    resp_action = baseline_dm.play(action)
    print(f"Jugador: {action}")
    print(f"DM: {resp_action[:200]}...")
    resp_check = baseline_dm.play(question)
    print(f"Verificacion: {question}")
    print(f"DM: {resp_check[:200]}...")
    baseline_responses.append({
        "turn": i + 1, "action": action, "question": question,
        "response_action": resp_action, "response_check": resp_check,
        "field": field, "expected": expected,
    })

print(f"\nBaseline completado. Turnos: {len(baseline_responses)}")
print(f"Tiempo total: {sum(baseline_dm.turn_times):.1f}s")

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# 5.2 Ejecutar AGENTE v1 (single agent)
# ────────────────────────────────────────────────────────────────────────────

print("=" * 78)
print("EJECUTANDO AGENTE v1 (single agent con herramientas)")
print("=" * 78)

agent_v1 = AgentDMv1(app_v1)
v1_responses = []

for i, (action, question, field, expected) in enumerate(TEST_SEQUENCE_V2):
    print(f"\n--- Turno {i+1} ---")
    resp_action = agent_v1.play(action)
    print(f"Jugador: {action}")
    print(f"DM: {resp_action[:200]}...")
    resp_check = agent_v1.play(question)
    print(f"Verificacion: {question}")
    print(f"DM: {resp_check[:200]}...")
    v1_responses.append({
        "turn": i + 1, "action": action, "question": question,
        "response_action": resp_action, "response_check": resp_check,
        "field": field, "expected": expected,
        "real_state": copy.deepcopy(game_state),  # deepcopy: el inventario sigue cambiando después
    })

print(f"\nAgente v1 completado. Turnos: {len(v1_responses)}")
print(f"Tiempo total: {sum(agent_v1.turn_times):.1f}s")

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# 5.3 Ejecutar MULTI-AGENTE v2
# ────────────────────────────────────────────────────────────────────────────

print("=" * 78)
print("EJECUTANDO MULTI-AGENTE v2 (narrator + world_builder + coherence)")
print("=" * 78)

dm_v2 = MultiAgentDM(app_v2)
v2_responses = []

for i, (action, question, field, expected) in enumerate(TEST_SEQUENCE_V2):
    print(f"\n--- Turno {i+1} ---")
    resp_action = dm_v2.play(action)
    print(f"Jugador: {action}")
    print(f"DM: {resp_action[:200]}...")
    resp_check = dm_v2.play(question)
    print(f"Verificacion: {question}")
    print(f"DM: {resp_check[:200]}...")
    v2_responses.append({
        "turn": i + 1, "action": action, "question": question,
        "response_action": resp_action, "response_check": resp_check,
        "field": field, "expected": expected,
        "real_state": copy.deepcopy(game_state),  # deepcopy: el inventario sigue cambiando después
        "memoria": _world_memory(),
        "mapa": {k: list(v["connections"]) for k, v in world["locations"].items()},
        "world_snapshot": {k: {"dynamic": v.get("dynamic", False)} for k, v in world["locations"].items()},
    })

print(f"\nMulti-agente v2 completado. Turnos: {len(v2_responses)}")
print(f"Tiempo total: {sum(dm_v2.turn_times):.1f}s")
print(f"Expansiones del mundo: {len(dm_v2.expansion_log)}")
for exp in dm_v2.expansion_log:
    print(f"  Turno {exp['turn']}: {exp['location']} — {exp['description'][:80]}")
print(f"\nMundo final: {_world_summary()}")
print(f"\nTokens de prompt por jugada: máx={max(dm_v2.context_tokens)}, "
      f"promedio={sum(dm_v2.context_tokens)/len(dm_v2.context_tokens):.0f}")
print(f"\n── MEMORIA DEL MUNDO (registro persistente) ──\n{_world_memory()}")

# Anclaje al SRD de lo creado por el WorldBuilder: ¿criaturas y objetos basados en datos oficiales?
creados = ([("npc", n, d) for n, d in world["npcs"].items() if d.get("dynamic")] +
           [("objeto", n, d) for n, d in world["items"].items() if d.get("dynamic")])
anclados = [(t, n, d["srd_ref"]) for t, n, d in creados if d.get("srd_ref")]
print(f"\nANCLAJE AL SRD: {len(anclados)}/{len(creados)} criaturas/objetos creados basados en el SRD")
for t, n, ref in anclados:
    extra = (f" → CA {world['npcs'][n]['defense']}, PG {world['npcs'][n]['hp']}, daño {world['npcs'][n].get('attack')}"
             if t == "npc" else "")
    print(f"  {t}: {n} ← {srd_docs[ref]['name']}{extra}")
anclaje_srd = {"creados": len(creados), "anclados": len(anclados), "detalle": anclados}

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Grafo del mundo al final de la partida de v2
# ────────────────────────────────────────────────────────────────────────────

draw_graph(titulo=f"Grafo del mundo tras {game_state['turn']} jugadas "
                  f"({G.number_of_nodes()} entidades, {G.number_of_edges()} relaciones)")
plt.tight_layout()
plt.savefig("grafo_mundo_v2.png", dpi=120)
plt.show()

print("Relaciones narrativas (cronista):")
for s_, r_, o_, t_ in narrative_triples:
    print(f"  jugada {t_}: {s_} —{r_}→ {o_}")
print("\nEjemplo de recuperación para '¿Cómo vuelvo a la cueva desde aquí?':")
print(graph_context("¿Cómo vuelvo a la cueva desde aquí?"))

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# 5.4 Evaluación por keywords (extendida para v2)
# ────────────────────────────────────────────────────────────────────────────

def check_consistency_v2(response: str, field: str, expected: str,
                          real_state: dict = None, world_snapshot: dict = None) -> bool:
    resp = unicodedata.normalize("NFKD", response.lower()).encode("ascii", "ignore").decode()

    if field == "item_mention":
        return expected.lower() in resp
    elif field == "inventory":
        return expected.lower() in resp
    elif field == "location":
        return expected.lower() in resp
    elif field == "illegal_move":
        illegal_indicators = ["no puedes", "no puede", "no es posible", "no hay camino",
                              "imposible", "no conecta", "solo puedes ir", "debes ir",
                              "no esta conectad", "no hay ruta", "no directa"]
        destino = "herreria" if expected == "no" else expected  # dev: herrería; hold-out: el destino ilegal
        went_there = destino in resp and any(w in resp for w in ["llegas", "entras", f"estas en la {destino}"])
        if went_there:
            return False
        return any(ind in resp for ind in illegal_indicators) or (expected == "no" and "bosque" in resp)
    elif field == "inventory_full":
        items = expected.split(",")
        return sum(1 for item in items if item.strip().lower() in resp) >= 2
    elif field == "hp_mentioned":
        return bool(re.search(r"\d+\s*/\s*\d+", resp)) or "hp" in resp or "vida" in resp
    elif field == "gold_change":
        return bool(re.search(r"\d+", resp))
    elif field == "potion_used":
        return any(ind in resp for ind in ["usaste", "bebiste", "consumiste", "ya no tienes",
                                            "se elimino", "curacion", "recuper"])
    elif field == "route":
        lugares = expected.split("→") if "→" in expected else ["bosque", "cueva"]
        return all(l in resp for l in lugares)
    elif field == "srd":
        return all(k in resp for k in expected.split(","))
    elif field == "relation_memory":
        return "goruk" in resp
    elif field == "event_memory":
        return expected.lower() in resp or "venciste" in resp or "mataste" in resp or "derrota" in resp
    elif field == "expansion":
        expansion_indicators = ["descubr", "encontr", "pasadizo", "puerta", "secreto",
                                "nuevo", "oculto", "escondid", "camino", "sendero",
                                "abertura", "grieta", "tunel"]
        return any(ind in resp for ind in expansion_indicators)
    elif field == "new_location":
        # Verificar si el jugador está en una ubicación dinámica
        if world_snapshot:
            current_loc = real_state.get("location", "") if real_state else ""
            loc_info = world_snapshot.get(current_loc, {})
            return loc_info.get("dynamic", False)
        return any(ind in resp for ind in ["nuevo", "descubr", "secreto"])
    elif field == "exploration":
        return len(resp) > 50  # al menos una descripción sustancial
    elif field == "connection_check":
        return any(ind in resp for ind in ["si", "sí", "puedes", "volver", "conecta"])
    elif field == "npc_creation":
        return any(ind in resp for ind in ["nuevo", "personaje", "desconocid", "alguien",
                                            "nunca hab", "extran"])
    elif field == "expansion_memory":
        return any(ind in resp for ind in ["nuevo", "descubr", "creado", "encontr"])
    elif field == "potion_gone":
        return any(ind in resp for ind in ["ya usaste", "ya no tienes", "consumida",
                                            "no tiene", "ya la bebiste"]) or "pocion" not in resp
    return False


# Evaluar los tres sistemas
results_kw = []
for br, v1r, v2r in zip(baseline_responses, v1_responses, v2_responses):
    b_text = br["response_action"] + " " + br["response_check"]
    v1_text = v1r["response_action"] + " " + v1r["response_check"]
    v2_text = v2r["response_action"] + " " + v2r["response_check"]

    b_ok = check_consistency_v2(b_text, br["field"], br["expected"])
    v1_ok = check_consistency_v2(v1_text, v1r["field"], v1r["expected"],
                                  v1r.get("real_state"), None)
    v2_ok = check_consistency_v2(v2_text, v2r["field"], v2r["expected"],
                                  v2r.get("real_state"), v2r.get("world_snapshot"))
    results_kw.append({
        "Turno": br["turn"],
        "Accion": br["action"][:40],
        "Verifica": br["field"],
        "Baseline": "OK" if b_ok else "FALLO",
        "Agente v1": "OK" if v1_ok else "FALLO",
        "Multi v2": "OK" if v2_ok else "FALLO",
    })

df_kw = pd.DataFrame(results_kw)
print("EVALUACIÓN POR KEYWORDS (3 sistemas)")
print(df_kw.to_string(index=False))

n = len(results_kw)
b_ok_kw = sum(1 for r in results_kw if r["Baseline"] == "OK")
v1_ok_kw = sum(1 for r in results_kw if r["Agente v1"] == "OK")
v2_ok_kw = sum(1 for r in results_kw if r["Multi v2"] == "OK")
print(f"\nCONSISTENCIA (keywords):")
print(f"  Baseline:    {b_ok_kw}/{n} ({b_ok_kw/n:.0%})")
print(f"  Agente v1:   {v1_ok_kw}/{n} ({v1_ok_kw/n:.0%})")
print(f"  Multi v2:    {v2_ok_kw}/{n} ({v2_ok_kw/n:.0%})")

### 5.5 LLM-as-judge con logprobs

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# 5.5 LLM-as-judge con logprobs (patrón Semana 2 Clase 3) — versión mejorada
# ────────────────────────────────────────────────────────────────────────────
# Mejoras respecto al juez simple (una nota de "consistencia", 0 ejemplos, mismo modelo):
#   1. Tres rúbricas separadas (como formato vs contenido en la clase), cada una 0-4 con logprobs:
#        consistencia (50%) · reglas (30%) · narrativa (20%)
#   2. Few-shot: 3 ejemplos calibrados por rúbrica (4, 2 y 0).
#   3. Juez más grande que el modelo evaluado: qwen2.5:72b juzga a qwen2.5:7b.
#   4. Acuerdo entre jueces: el 7B también califica; se mide cuánto difiere del 72B.
#      (Repetir el mismo prompt no sirve: con logprobs el valor esperado es determinista.)
#   5. Ground truth justo: el baseline no tiene estado interno, así que se le compara con el
#      estado canónico del guion; en los turnos de memoria, cada sistema se compara con lo que
#      él mismo narró antes (autoconsistencia).

client_judge = OpenAI(base_url=f"{OLLAMA_BASE_URL}/v1", api_key="ollama")

SISTEMA_JUEZ = "Eres un evaluador estricto de partidas de D&D. Respondes solo con un digito del 0 al 4."
DIGITOS_STR = ["0", "1", "2", "3", "4"]

REGLAS_JUEGO = (
    "Mapa: taberna–plaza, taberna–bosque, plaza–herreria, plaza–bosque, bosque–cueva "
    "(más los lugares que el DM haya descubierto). Solo se puede ir a lugares conectados. "
    "Combate (D&D 5e simplificado): el jugador ataca con d20+5 contra la CA del enemigo (goblin: CA 15, 7 PG; "
    "suele requerir varios golpes); el enemigo ataca con d20+su bonificador contra la CA del jugador (14). "
    "La poción cura 2d4+2 (máximo 20 HP) y se consume. Elena vende pociones a 10 de oro; Bron afila la espada por 5. "
    "No se pueden recoger objetos que no están en el lugar.")

RUBRICAS = {
    "consistencia": {
        "peso": 0.5,
        "definicion": ("¿El DM RESPONDE la pregunta con datos que coinciden con el estado real (ubicación, HP, oro, "
                       "inventario, lugares, personajes)? Si no responde (vacío, evasivo o texto técnico como "
                       "'CallCheck_state()'), la nota es 0: no responder no es ser consistente."),
        "escala": "4 = responde y coincide en todo · 3 = un detalle menor mal · 2 = mitad correcta · 1 = casi todo mal · 0 = contradice el estado real o no responde",
        "ejemplos": [
            ("¿Dónde estoy?", "El jugador está en: bosque",
             "Estás en el bosque, rodeado de árboles altos.", 4),
            ("¿Qué tengo en mi inventario?", "Inventario real: espada oxidada, pocion de curacion, antorcha",
             "Llevas una espada oxidada, una poción de curación y la antorcha que acabas de recoger.", 4),
            ("¿Qué objetos hay aquí?", "En la taberna hay: antorcha", "CallCheck_state()", 0),
            ("¿Qué tengo?", "Inventario: espada oxidada, antorcha, pocion de curacion",
             "Tienes una espada oxidada y una antorcha.", 2),
            ("¿Dónde estoy?", "El jugador está en: bosque",
             "Te encuentras en la herrería; el calor del horno te sofoca.", 0),
        ],
    },
    "reglas": {
        "peso": 0.3,
        "definicion": ("¿El DM resuelve la acción del jugador respetando las mecánicas del juego (movimiento solo a "
                       "lugares conectados, combate con dados, poción que se consume) sin inventar capacidades ni "
                       "resultados? Si no resuelve la acción (vacío o texto técnico), la nota es 0."),
        "escala": "4 = resuelve la acción respetando las reglas · 2 = la resuelve a medias o sin mecánica · 0 = viola una regla o no resuelve la acción",
        "ejemplos": [
            ("Voy a la herrería. (desde el bosque)", "Movimiento ILEGAL: el bosque no conecta con la herrería",
             "No hay camino directo del bosque a la herrería. Puedes ir a la plaza, la taberna o la cueva.", 4),
            ("Ataco al goblin.", "Combate: d20 >= 10 acierta, daño 1d6",
             "Derrotas al goblin de un solo golpe.", 2),
            ("Voy a la herrería. (desde el bosque)", "Movimiento ILEGAL: el bosque no conecta con la herrería",
             "Llegas a la herrería y Bron el herrero te saluda.", 0),
            ("Recojo la antorcha.", "En la taberna hay: antorcha", "CallCheck_state()", 0),
        ],
    },
    "narrativa": {
        "peso": 0.2,
        "definicion": "¿La narración es inmersiva, clara y coherente con la escena (sin texto técnico ni errores)?",
        "escala": "4 = inmersiva y clara · 2 = correcta pero plana · 0 = rota, técnica o incoherente",
        "ejemplos": [
            ("Voy al bosque.", "-",
             "Los árboles se cierran sobre ti y la luz se vuelve verde y tenue. Entre la maleza, algo cruje.", 4),
            ("Voy al bosque.", "-", "Estás en el bosque. Hay árboles.", 2),
            ("Voy al bosque.", "-", 'CallCheck_state: {"name": "check_state", "arguments": {}}', 0),
        ],
    },
}

PROMPT_JUEZ = """\
Evalúa la respuesta de un Dungeon Master de D&D según esta rúbrica.

RÚBRICA — {nombre}: {definicion}
{escala}

EJEMPLOS CALIFICADOS:
{ejemplos}

AHORA EVALÚA ESTE CASO:
Acción del jugador: {accion}
Pregunta de verificación: {pregunta}
Estado real del juego (ground truth): {estado}
Reglas del juego: {reglas}

Respuesta del DM a la acción:
<accion_dm>
{r_accion}
</accion_dm>

Respuesta del DM a la pregunta:
<pregunta_dm>
{r_pregunta}
</pregunta_dm>

Responde solo con un digito del 0 al 4."""


def _ejemplos_txt(rubrica):
    return "\n".join(f"- Jugador: {a} | Estado real: {e} | DM: \"{r}\" → {n}"
                     for a, e, r, n in RUBRICAS[rubrica]["ejemplos"])


# Turnos donde se evalúa la memoria de expansiones (autoconsistencia)
CAMPOS_EXPANSION = {"expansion", "npc_creation", "exploration", "new_location"}


def build_ground_truth_v2(r, historial):
    """Ground truth del turno r. Usa el estado real si el sistema lo tiene (agentes);
    si no (baseline), el estado canónico del guion. Para memoria: lo que el sistema narró antes."""
    field, expected, real = r["field"], r["expected"], r.get("real_state")
    memoria = r.get("memoria")  # registro del mundo (solo v2)
    if field == "item_mention":
        gt = f"En la ubicación actual hay: {expected}"
    elif field in ("inventory", "inventory_full"):
        gt = f"El inventario debe incluir: {expected}"
        if real:
            gt += f". Inventario real: {', '.join(real['inventory'])}"
    elif field == "location":
        gt = f"El jugador está en: {expected}"
    elif field == "illegal_move":
        gt = ("Movimiento ILEGAL: el bosque no conecta con la herrería. El jugador sigue en el bosque."
              if expected == "no" else
              f"Movimiento ILEGAL hacia {expected}: no hay conexión directa desde donde está. El jugador NO se mueve.")
    elif field == "hp_mentioned":
        gt = (f"HP real: {real['hp']}/{real['max_hp']}" if real
              else "HP entre 0 y 20, consistente con el daño narrado en el combate")
    elif field == "gold_change":
        gt = f"Oro real: {real['gold']}" if real else f"Oro: {ORO_CANONICO}"
    elif field == "potion_used":
        gt = "La poción se consumió: ya no está en el inventario y el HP subió (máximo 20)."
    elif field == "event_memory":
        gt = f"Eventos clave del guion: {EVENTOS_CANONICOS}"
        if real:
            gt += f" Eventos registrados: {real['events'][-12:]}"
    elif field in CAMPOS_EXPANSION or field == "connection_check":
        gt = ("El jugador busca o explora algo: el DM debe describir un hallazgo concreto "
              "(o decir claramente que no hay nada) y ser consistente con lo que narró antes.")
        if memoria:
            gt += f" Registro del mundo: {memoria[:400]}"
    elif field == "route":
        ruta = expected.replace("→", " → ") if "→" in expected else "taberna → bosque → cueva"
        if real:  # oráculo programático: ruta más corta desde donde REALMENTE está el jugador
            mapa = nx.Graph([(a, b) for a, cs in (r.get("mapa") or {k: v["connections"] for k, v in
                             WORLD_INITIAL["locations"].items()}).items() for b in cs])
            destino = expected.split("→")[-1] if "→" in expected else "cueva"
            try:
                ruta = " → ".join(nx.shortest_path(mapa, real["location"], destino))
            except (nx.NetworkXNoPath, nx.NodeNotFound):
                pass
        gt = (f"Ruta más corta: {ruta}. Una ruta válida pero más larga es un detalle menor. "
              "Es un error grave usar una conexión inexistente.")
    elif field == "relation_memory":
        npc = next(n for n in WORLD_INITIAL["npcs"] if expected in n.lower())
        gt = f'El jugador habló con {npc}, que le dijo: "{WORLD_INITIAL["npcs"][npc]["dialogue"]}"'
    elif field == "srd":
        gt = "Según las reglas oficiales (SRD 5.1): " + " || ".join(srd_docs[d]["text"][:600] for d in SRD_GOLD[r["action"]])
    elif field == "expansion_memory":
        narrado = [h["response_check"][:200].replace("\n", " ") for h in historial
                   if h["field"] in CAMPOS_EXPANSION]
        gt = "Debe coincidir con los descubrimientos que el propio DM narró antes: " + " || ".join(narrado)
        if memoria:
            gt += f" Registro del mundo: {memoria[:400]}"
    else:
        gt = f"Esperado: {expected}"
    return gt


def _nota_logprobs(modelo, prompt):
    """Valor esperado sobre los dígitos 0-4 (logprobs) × 25 → nota 0-100."""
    response = client_judge.chat.completions.create(
        model=modelo,
        messages=[{"role": "system", "content": SISTEMA_JUEZ}, {"role": "user", "content": prompt}],
        temperature=0, max_tokens=1, logprobs=True, top_logprobs=10,
    )
    token_logprobs = response.choices[0].logprobs.content[0].top_logprobs
    logprob_dict = {lp.token.strip(): lp.logprob for lp in token_logprobs}
    raw_probs = [math.exp(logprob_dict.get(d, -23)) for d in DIGITOS_STR]
    total = sum(raw_probs)
    return sum(i * p / total for i, p in enumerate(raw_probs)) * 25


def calificar_turno(r, historial, modelo):
    """Notas 0-100 por rúbrica y nota final ponderada."""
    estado = build_ground_truth_v2(r, historial)
    notas = {}
    for nombre, rub in RUBRICAS.items():
        # Crédito parcial (como ReferenceCoverage): la respuesta a la acción y la respuesta a la pregunta
        # se califican por separado y se promedian → un seguimiento malo no anula una respuesta buena.
        parciales = []
        for r_acc, r_preg in ((r["response_action"][:600], "(no se evalúa en esta parte)"),
                              ("(no se evalúa en esta parte)", r["response_check"][:600])):
            prompt = PROMPT_JUEZ.format(
                nombre=nombre, definicion=rub["definicion"], escala=rub["escala"], ejemplos=_ejemplos_txt(nombre),
                accion=r["action"], pregunta=r["question"], estado=estado, reglas=REGLAS_JUEGO,
                r_accion=r_acc, r_pregunta=r_preg)
            parciales.append(_nota_logprobs(modelo, prompt))
        notas[nombre] = sum(parciales) / 2
    notas["final"] = sum(RUBRICAS[k]["peso"] * notas[k] for k in RUBRICAS)
    return notas


SISTEMAS = {"Baseline": baseline_responses, "Agente v1": v1_responses, "Multi v2": v2_responses}
JUECES = {"72B": JUDGE_MODEL, "7B": OLLAMA_MODEL}

print("=" * 78)
print(f"EVALUACIÓN LLM-AS-JUDGE (logprobs, 3 rúbricas, few-shot) — juez principal: {JUDGE_MODEL}")
print("=" * 78)

filas = []
for sistema, respuestas in SISTEMAS.items():
    for i, r in enumerate(respuestas):
        for juez, modelo in JUECES.items():
            notas = calificar_turno(r, respuestas[:i], modelo)
            filas.append({"sistema": sistema, "turno": i + 1, "juez": juez, "field": r["field"], **notas})
    print(f"  {sistema}: listo")

df_juez = pd.DataFrame(filas)
import os
df_juez.to_csv(f"juez_por_turno{os.environ.get('RUN_TAG', '')}.csv", index=False)  # notas por turno, rúbrica y juez (para análisis)
principal = df_juez[df_juez.juez == "72B"]

# Variables que usan las celdas siguientes (tabla y gráficos): nota final del juez 72B
scores_b = principal[principal.sistema == "Baseline"].sort_values("turno")["final"].tolist()
scores_v1 = principal[principal.sistema == "Agente v1"].sort_values("turno")["final"].tolist()
scores_v2 = principal[principal.sistema == "Multi v2"].sort_values("turno")["final"].tolist()
avg_b, avg_v1, avg_v2 = (sum(s) / n for s in (scores_b, scores_v1, scores_v2))
ok_b, ok_v1, ok_v2 = (sum(1 for x in s if x >= 75) for s in (scores_b, scores_v1, scores_v2))

print("\nNOTA POR RÚBRICA (juez 72B, promedio 0-100):")
por_rubrica = principal.groupby("sistema")[list(RUBRICAS) + ["final"]].mean().round(1)
print(por_rubrica.loc[list(SISTEMAS)].to_string())

print(f"\nNOTA FINAL PONDERADA: Baseline={avg_b:.1f} | v1={avg_v1:.1f} | v2={avg_v2:.1f}")
print(f"TURNOS ≥ 75: Baseline={ok_b}/{n} | v1={ok_v1}/{n} | v2={ok_v2}/{n}")

# Acuerdo entre jueces (7B vs 72B)
pares = df_juez.pivot_table(index=["sistema", "turno"], columns="juez", values="final")
dif = (pares["7B"] - pares["72B"]).abs()
print(f"\nACUERDO ENTRE JUECES (nota final): diferencia media |7B − 72B| = {dif.mean():.1f} puntos, "
      f"correlación = {pares['7B'].corr(pares['72B']):.2f}")
print(f"Turnos con evaluación inestable (|7B − 72B| > 25): {(dif > 25).sum()}/{len(dif)}")
print("\nNota promedio según cada juez:")
print(df_juez.groupby(["juez", "sistema"])["final"].mean().unstack().round(1)[list(SISTEMAS)].to_string())



# Respuestas completas de los 3 sistemas (para reevaluar sin volver a jugar)
with open(f"respuestas{os.environ.get('RUN_TAG', '')}.json", "w") as f:
    json.dump({**SISTEMAS, "anclaje_srd_v2": anclaje_srd}, f, ensure_ascii=False, default=str)


# ────────────────────────────────────────────────────────────────────────────
# 5.6 Faithfulness con RAGAS (Semana 3, Clase 4)
# ────────────────────────────────────────────────────────────────────────────
# ¿Las afirmaciones de la respuesta están respaldadas por el contexto? RAGAS descompone la
# respuesta en afirmaciones y verifica cada una contra el contexto: faithfulness = respaldadas / total.
# Evaluador: el juez 72B vía la API compatible con OpenAI de Ollama.
# Contexto: en los turnos de REGLAS, el documento oficial del SRD; en el resumen de la aventura,
# los eventos reales (agentes) o los del guion (baseline). Así los 3 sistemas se miden igual.
# (Requiere ragas 0.4 y langchain-community==0.3.31; ver dependencias en CLAUDE.md.)

from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import Faithfulness

ragas_llm = llm_factory(JUDGE_MODEL, provider="openai",
                        client=AsyncOpenAI(base_url=f"{OLLAMA_BASE_URL}/v1", api_key="ollama"))
faithfulness_ragas = Faithfulness(llm=ragas_llm)


def contexto_faithfulness(r):
    if r["field"] == "srd":
        return "\n\n".join(srd_docs[d]["text"] for d in SRD_GOLD[r["action"]])
    real = r.get("real_state")
    return "Eventos: " + ("; ".join(real["events"]) if real else EVENTOS_CANONICOS)


filas_f = []
for sistema, respuestas in SISTEMAS.items():
    for r in respuestas:
        if r["field"] in ("srd", "event_memory"):
            respuesta = re.sub(r"\[HP:[^\]]*\]", "", r["response_action"] + "\n" + r["response_check"])
            try:
                res = await faithfulness_ragas.ascore(user_input=f"{r['action']} {r['question']}", response=respuesta,
                                                      retrieved_contexts=[contexto_faithfulness(r)])
                score = res.value
            except Exception as e:
                print(f"  RAGAS falló en {sistema} turno {r['turn']}: {type(e).__name__}")
                score = None
            filas_f.append({"sistema": sistema, "turno": r["turn"], "field": r["field"], "faithfulness": score})

df_faith = pd.DataFrame(filas_f)
if len(df_faith):
    df_faith.to_csv(f"faithfulness{os.environ.get('RUN_TAG', '')}.csv", index=False)
    print("FAITHFULNESS (RAGAS, evaluador qwen2.5:72b) — 1.0 = todo lo afirmado está respaldado")
    print(df_faith.pivot_table(index="turno", columns="sistema", values="faithfulness").round(2)
          [list(SISTEMAS)].to_string())
    print("\nPromedio por tipo de turno:")
    print(df_faith.groupby(["field", "sistema"]).faithfulness.mean().unstack().round(2)[list(SISTEMAS)].to_string())


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# 5.7 Suite de DIÁLOGO (v3) — desarrollo, no hold-out
# ────────────────────────────────────────────────────────────────────────────
# Mide lo que agregó el DialogueAgent + buy_item. Es una prueba de DESARROLLO (se diseñó mirando el
# sistema), no un hold-out. Los 3 sistemas juegan la misma secuencia:
#   - informativa: Goruk sobre la cueva → debe mencionar al goblin (está en su perfil)
#   - límite de conocimiento: Goruk sobre el dragón → debe ABSTENERSE (patrón "missing" de CRAG)
#   - negociación: regatear con Elena → mantiene el precio fijo de 10 (y no vende)
#   - compra: el oro y el inventario deben cambiar de verdad
#   - consistencia: Goruk de nuevo sobre la cueva → no debe contradecirse
#   - pistas: Goruk menciona a la bruja (nodo especulativo) → "voy al pantano" crea el lugar → buscarla la confirma
# Métricas: faithfulness (RAGAS) de lo que dicen los NPCs contra su perfil; abstención y consistencia
# con el juez 72B (salida estructurada); compra verificada contra el estado real.

SUITE_DIALOGO = [
    ("info", "Le pregunto a Goruk qué sabe sobre la cueva.", "Goruk el tabernero"),
    ("limite", "Le pregunto a Goruk cómo derrotar al dragón.", "Goruk el tabernero"),
    ("mover", "Voy a la plaza.", None),
    ("negociacion", "Le pregunto a Elena si me deja la poción en 5 monedas.", "Elena la comerciante"),
    ("compra", "Le compro una poción a Elena.", "Elena la comerciante"),
    ("mover", "Vuelvo a la taberna.", None),
    ("consistencia", "Le vuelvo a preguntar a Goruk qué hay en la cueva.", "Goruk el tabernero"),
    # Pistas narrativas (error 10): una mención crea un nodo especulativo; seguirla lo vuelve real
    ("pista_mencion", "Le pregunto a Goruk si sabe algo de una bruja.", "Goruk el tabernero"),
    ("pista_seguir", "Voy al pantano a buscar a la bruja.", None),
    ("pista_confirmar", "Busco a la bruja entre la niebla.", None),
]


class Abstencion(BaseModel):
    se_abstiene: bool = Field(description="True si el personaje admite que no sabe / no tiene esa información.")
    inventa: bool = Field(description="True si da información concreta que no está en su perfil (consejos, datos).")


class Consistencia(BaseModel):
    se_contradice: bool = Field(description="True si la respuesta nueva contradice a la anterior.")


_abst = ChatOllama(model=JUDGE_MODEL, base_url=OLLAMA_BASE_URL, temperature=0).with_structured_output(
    Abstencion, method="json_schema")
_cons = ChatOllama(model=JUDGE_MODEL, base_url=OLLAMA_BASE_URL, temperature=0).with_structured_output(
    Consistencia, method="json_schema")


def perfil_npc(npc):
    d = WORLD_INITIAL["npcs"][npc]
    return f"{npc}. {d.get('personalidad', '')} Sabe: " + " ".join(conocimientos_npc(d))


filas_d = []
for sistema, crear in {"Baseline": lambda: BaselineDM(llm), "Agente v1": lambda: AgentDMv1(app_v1),
                       "Multi v3": lambda: MultiAgentDM(app_v2)}.items():
    reset_game()  # el baseline no tiene estado propio: parte del estado inicial
    dm, respuestas = crear(), {}
    for tipo, accion, npc in SUITE_DIALOGO:
        oro_antes, inv_antes = game_state["gold"], len(game_state["inventory"])
        resp = re.sub(r"\[HP:[^\]]*\]", "", dm.play(accion)).strip()
        fila = {"sistema": sistema, "tipo": tipo, "accion": accion, "respuesta": resp}
        if npc and tipo in ("info", "negociacion", "consistencia", "limite"):
            res = await faithfulness_ragas.ascore(user_input=accion, response=resp, retrieved_contexts=[perfil_npc(npc)])
            fila["faithfulness"] = res.value
        if tipo == "info":
            fila["menciona_goblin"] = "goblin" in _norm(resp)
        if tipo == "limite":
            a = _abst.invoke(f"Perfil del personaje: {perfil_npc(npc)}\nPregunta: {accion}\nRespuesta: {resp}")
            fila["se_abstiene"], fila["inventa"] = a.se_abstiene, a.inventa
        if tipo == "negociacion":
            fila["precio_10"] = "10" in resp
            fila["no_vendio"] = game_state["gold"] == oro_antes  # regatear no es comprar
        if tipo == "compra":  # solo los agentes tienen estado real; el baseline no puede comprar de verdad
            fila["compra_real"] = sistema != "Baseline" and game_state["gold"] == oro_antes - 10 \
                and len(game_state["inventory"]) == inv_antes + 1
        pistas = game_state.get("especulativos", {})  # solo el multi-agente tiene grafo con pistas
        if tipo == "pista_mencion":
            fila["pista_creada"] = sistema == "Multi v3" and any("bruja" in k for k in pistas)
        if tipo == "pista_seguir":
            fila["lugar_creado"] = sistema == "Multi v3" and any(v.get("dynamic") for v in world["locations"].values())
        if tipo == "pista_confirmar":
            fila["pista_confirmada"] = sistema == "Multi v3" and any(p.get("confirmada") for k, p in pistas.items() if "bruja" in k)
        if tipo == "consistencia":
            fila["se_contradice"] = _cons.invoke(f"Respuesta anterior: {respuestas['info']}\nRespuesta nueva: {resp}").se_contradice
        respuestas[tipo] = resp
        filas_d.append(fila)

df_dialogo = pd.DataFrame(filas_d)
df_dialogo.to_csv(f"dialogo{os.environ.get('RUN_TAG', '')}.csv", index=False)
resumen = df_dialogo.groupby("sistema").agg(
    faithfulness=("faithfulness", "mean"),
    menciona_goblin=("menciona_goblin", "max"),
    se_abstiene=("se_abstiene", "max"),
    inventa=("inventa", "max"),
    precio_10=("precio_10", "max"),
    no_vendio=("no_vendio", "max"),
    compra_real=("compra_real", "max"),
    se_contradice=("se_contradice", "max"),
    pista_creada=("pista_creada", "max"),
    lugar_creado=("lugar_creado", "max"),
    pista_confirmada=("pista_confirmada", "max"),
).round(2)
print("SUITE DE DIÁLOGO (desarrollo)")
print(resumen.loc[["Baseline", "Agente v1", "Multi v3"]].to_string())


## 6. Comparación cuantitativa

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# 6.1 Tabla comparativa
# ────────────────────────────────────────────────────────────────────────────

t_b = sum(baseline_dm.turn_times)
t_v1 = sum(agent_v1.turn_times)
t_v2 = sum(dm_v2.turn_times)

comparacion = pd.DataFrame({
    "Métrica": [
        "Keywords (binaria)",
        "Juez ≥ 75 (binaria)",
        "Nota promedio juez (0-100)",
        "Tiempo total (s)",
        "Tiempo por turno (s)",
        "Ubicaciones creadas",
        "NPCs creados",
        "Herramientas",
        "Nodos en el grafo",
    ],
    "Baseline": [
        f"{b_ok_kw}/{n} ({b_ok_kw/n:.0%})",
        f"{ok_b}/{n} ({ok_b/n:.0%})",
        f"{avg_b:.1f}",
        f"{t_b:.1f}",
        f"{t_b/len(baseline_dm.turn_times):.1f}",
        "0", "0", "0", "0",
    ],
    "Agente v1": [
        f"{v1_ok_kw}/{n} ({v1_ok_kw/n:.0%})",
        f"{ok_v1}/{n} ({ok_v1/n:.0%})",
        f"{avg_v1:.1f}",
        f"{t_v1:.1f}",
        f"{t_v1/len(agent_v1.turn_times):.1f}",
        "0", "0", "6", "2 (agent, tools)",
    ],
    "Multi v2": [
        f"{v2_ok_kw}/{n} ({v2_ok_kw/n:.0%})",
        f"{ok_v2}/{n} ({ok_v2/n:.0%})",
        f"{avg_v2:.1f}",
        f"{t_v2:.1f}",
        f"{t_v2/len(dm_v2.turn_times):.1f}",
        str(len(dm_v2.expansion_log)),
        str(len(game_state.get('npcs_created', []))),
        "10",
        "5 (pensar, actuar, narrar, world_builder, coherence)",
    ],
})

print("=" * 78)
print("COMPARACIÓN DE RESULTADOS (3 sistemas)")
print("=" * 78)
print(comparacion.to_string(index=False))

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# 6.2 Gráficos comparativos
# ────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
colors = {"Baseline": "#ef5350", "Agente v1": "#FFA726", "Multi v2": "#66BB6A"}

# --- Panel 1: Nota del juez por turno ---
ax = axes[0]
x = range(1, n + 1)
w = 0.25
ax.bar([i - w for i in x], scores_b, w, label="Baseline", color=colors["Baseline"], alpha=0.8)
ax.bar(list(x), scores_v1, w, label="Agente v1", color=colors["Agente v1"], alpha=0.8)
ax.bar([i + w for i in x], scores_v2, w, label="Multi v2", color=colors["Multi v2"], alpha=0.8)
ax.axhline(75, color="gray", linestyle="--", alpha=0.5, label="Umbral (75)")
ax.set_xlabel("Turno")
ax.set_ylabel("Nota del juez (0-100)")
ax.set_title("Nota del juez por turno")
ax.legend(fontsize=8)
ax.set_xticks(list(x))

# --- Panel 2: Resumen de métricas ---
ax = axes[1]
metrics = ["Nota promedio\n(juez 0-100)", "Consistencia\n(keywords %)", "Consistencia\n(juez ≥75 %)"]
vals_b = [avg_b, b_ok_kw/n*100, ok_b/n*100]
vals_v1 = [avg_v1, v1_ok_kw/n*100, ok_v1/n*100]
vals_v2 = [avg_v2, v2_ok_kw/n*100, ok_v2/n*100]
y_pos = range(len(metrics))
h = 0.25
ax.barh([y - h for y in y_pos], vals_b, h, label="Baseline", color=colors["Baseline"], alpha=0.8)
ax.barh(list(y_pos), vals_v1, h, label="Agente v1", color=colors["Agente v1"], alpha=0.8)
ax.barh([y + h for y in y_pos], vals_v2, h, label="Multi v2", color=colors["Multi v2"], alpha=0.8)
for i, (vb, v1, v2) in enumerate(zip(vals_b, vals_v1, vals_v2)):
    ax.text(vb + 1, i - h, f"{vb:.0f}", va="center", fontsize=8, color=colors["Baseline"])
    ax.text(v1 + 1, i, f"{v1:.0f}", va="center", fontsize=8, color=colors["Agente v1"])
    ax.text(v2 + 1, i + h, f"{v2:.0f}", va="center", fontsize=8, color=colors["Multi v2"])
ax.set_yticks(list(y_pos))
ax.set_yticklabels(metrics)
ax.set_xlabel("Puntuación")
ax.set_title("Resumen de métricas")
ax.legend(fontsize=8)

# --- Panel 3: Expansión del mundo ---
ax = axes[2]
expansion_metrics = ["Ubicaciones\ncreadas", "NPCs\ncreados", "Tiempo/turno\n(segundos)"]
n_locs_created = len(dm_v2.expansion_log)
n_npcs_created = len(game_state.get("npcs_created", []))
exp_b = [0, 0, t_b/len(baseline_dm.turn_times)]
exp_v1 = [0, 0, t_v1/len(agent_v1.turn_times)]
exp_v2 = [n_locs_created, n_npcs_created, t_v2/len(dm_v2.turn_times)]
y_pos2 = range(len(expansion_metrics))
ax.barh([y - h for y in y_pos2], exp_b, h, label="Baseline", color=colors["Baseline"], alpha=0.8)
ax.barh(list(y_pos2), exp_v1, h, label="Agente v1", color=colors["Agente v1"], alpha=0.8)
ax.barh([y + h for y in y_pos2], exp_v2, h, label="Multi v2", color=colors["Multi v2"], alpha=0.8)
for i, (vb, v1, v2) in enumerate(zip(exp_b, exp_v1, exp_v2)):
    ax.text(max(vb, 0.3), i - h, f"{vb:.1f}", va="center", fontsize=8, color=colors["Baseline"])
    ax.text(max(v1, 0.3), i, f"{v1:.1f}", va="center", fontsize=8, color=colors["Agente v1"])
    ax.text(max(v2, 0.3) + 0.1, i + h, f"{v2:.1f}", va="center", fontsize=8, color=colors["Multi v2"])
ax.set_yticks(list(y_pos2))
ax.set_yticklabels(expansion_metrics)
ax.set_xlabel("Valor")
ax.set_title("Expansión y costo")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 7. Demo interactiva (con traza de cada agente)

`demo_interactiva_v2()` en el notebook, o `python jugar.py` en la terminal. Con la traza activa se ve, jugada a jugada, qué decide cada agente, qué recupera del grafo y del SRD, y qué reglas deterministas corrigieron al planner.


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# 7. Demo interactiva con traza (no se ejecuta en batch)
# ────────────────────────────────────────────────────────────────────────────
# Muestra en vivo qué hace cada agente: WorldBuilder (qué crea y a qué lo ancla en el SRD),
# CoherenceChecker (veredicto), planner ReAct (pensamiento → acción, y si una regla la cambió),
# herramientas (observación), contexto recuperado del grafo y cronista (tripletas al grafo).
# También desde la terminal: python jugar.py

COMANDOS = """Comandos: /grafo (dibuja el grafo)  /memoria (resumen del grafo)  /contexto <texto> (qué recupera el grafo)
          /srd <consulta> (qué recupera el SRD)  /estado  /traza on|off  /salir"""


def demo_interactiva_v2(traza=True):
    # Para imágenes: os.environ["IMAGENES"] = "1" ANTES de ejecutar la celda del grafo
    """Demo interactiva con el multi-agente v2. Ejecutar manualmente."""
    dm = MultiAgentDM(app_v2)
    print("=" * 70)
    print("  DUNGEON MASTER v2 — Multi-Agente con Mundo Dinámico (modo traza)")
    print("  " + COMANDOS)
    print("=" * 70)
    while True:
        try:
            action = input("\n> ").strip()
        except (EOFError, KeyboardInterrupt):
            break
        if not action or action == "/salir":
            break
        if action == "/grafo":
            draw_graph(titulo=f"Grafo tras {game_state['turn']} jugadas")
            plt.show()
        elif action == "/memoria":
            print(graph_summary())
        elif action.startswith("/contexto"):
            print(graph_context(action[len("/contexto"):].strip() or game_state["location"]))
        elif action.startswith("/srd"):
            for d in buscar_srd(reescribir_consulta(action[4:].strip())):
                print(f"  [{d['id']}] {d['text'][:200]}")
        elif action == "/estado":
            print(check_state.invoke({}))
        elif action.startswith("/traza"):
            traza = action.endswith("on")
        else:
            print(f"\nDM: {dm.play(action, traza=traza)}")
            if dm.ultima_imagen:  # v3: la imagen de la escena, debajo del texto
                from IPython.display import Image, display
                display(Image(filename=dm.ultima_imagen, width=384))
    print("Fin de la partida.")

# demo_interactiva_v2()  # descomentar para jugar


## 8. Conclusiones

*Se actualizarán tras la ejecución completa.*

## 9. Declaración de uso de IAG

Este proyecto fue desarrollado con asistencia de inteligencia artificial generativa:

- **Claude Code (Anthropic)**: planificación de la arquitectura multi-agente, implementación del grafo LangGraph con múltiples nodos especializados, diseño de las herramientas de expansión dinámica del mundo, y configuración de la evaluación.
- **Ollama + qwen2.5:7B**: modelo local usado tanto para los agentes del juego como para el LLM-as-judge.

Todo el código fue revisado, ajustado y verificado por el estudiante.